# DATA-DRIVEN 1D MECHANICAL EARTH MODEL
## INCREMENT 4 — CHECKSHOT QC & TIME-DEPTH FRAMEWORK  (corrected by INCREMENT 4.1, then 4.1.1, then 4.1.2)

**Author:** Mikael Elgo
**Project classification:** Tier C — Screening-Level / Uncalibrated Educational 1D MEM. This notebook is a portfolio and educational workflow. It is **not** calibrated for operational drilling, well design, casing design, or real-world mud-weight decisions.

> **INCREMENT 4.1 — CORRECTIVE PATCH (p2mem 0.4.1).** This notebook builds the ORIGINAL Increment 4 checkshot/time-depth layer AND the Increment 4.1 corrective patch together, in one run. Increment 4.1 fixes a genuine, independently reproduced defect in Increment 4's TVDSS↔OWT/TWT inversion: a repeated (tied) value on the axis being inverted was resolved by silently keeping whichever tied row happened to be encountered first and discarding the other — an ORDER-DEPENDENT tie-break with no audit trail beyond a bare count. Increment 4.1 replaces this with an explicit, ORDER-INVARIANT axis-tie-conditioning policy (group by exact value, register every tied row, use the group's MEDIAN dependent value as the representative — see Step 11b below and `INCREMENT_04_1_MANIFEST.md`). Increment 4.1 is a corrective patch only: it does **not** begin Increment 5, and implements no formation-top, petrophysics, pore-pressure, mechanical-properties, stress, or wellbore-stability work.

> **INCREMENT 4.1.1 — NUMERICAL-VALIDATION CORRECTIVE PATCH (p2mem 0.4.1.1).** This notebook additionally builds the Increment 4.1.1 patch on top of 4.1, in the same run. An independent numerical-validation audit found that Increment 4.1's axis-tie reversal check could be silently defeated when a reversal returned to an already-seen value (global exact-value grouping absorbed it before the reversal was checked), that full-canonical-MD monotonicity was validated only inside the selected sonic run (not across the complete log), that zero in-coverage checkshot rows crashed `compare_checkshot_to_survey` with an untyped NumPy error, that a numerical-conditioning failure in one well could stop the entire checkshot batch, and that the unit-conversion helpers did not reject boolean/string/complex input. All five are corrected (see Step 11c below and `INCREMENT_04_1_1_MANIFEST.md`); **no real Poseidon 2/Boreas 1/Proteus 1ST2 result changes** as a result. Increment 4.1.1 is a narrowly scoped numerical-validation corrective patch only: it does **not** begin Increment 5.

> **INCREMENT 4.1.2 — COLAB LINE-ENDING REPRODUCIBILITY AND TRUTHFUL COMPLETION-GATE PATCH (packaging/notebook only — `p2mem` remains 0.4.1.1).** A real Google Colab fresh-runtime execution of the Increment 4.1.1 notebook reported `1 failed, 384 passed`: `tests/fixtures/checkshot_valid.txt` — the project's one intentionally CRLF-encoded fixture — reconstructed with LF line endings when written through its `%%writefile` cell, because Colab's own cell-source handling normalizes embedded line-ending bytes in ways this project's Linux-based build/verification tooling could not reproduce or detect. Separately, Step 9's `!pytest -v` line never checked its own exit code, so the notebook continued past that failing test and its completion gate printed every condition `[PASS]` regardless — a genuine, confirmed false-positive. Both are corrected here: `tests/fixtures/checkshot_valid.txt` is now written by an ordinary Python cell (Step 7a) that reconstructs its exact bytes from an escaped bytes-literal via `Path.write_bytes()`, immune to any text-mode or Colab-side line-ending normalization; Step 9 now runs pytest via `subprocess.run` with an explicit return-code check, setting `FULL_TEST_SUITE_PASSED` and raising `RuntimeError` on any failure; and the Completion Gate below now has an explicit, first-checked condition requiring that flag to be `True`. This is a packaging/notebook correction only — no numerical method, contract, raw-data policy, or scientific output changes; the Python implementation stays at `p2mem` 0.4.1.1. See `INCREMENT_04_1_2_MANIFEST.md`. Increment 4.1.2 does **not** begin Increment 5.

**Locked foundation:** Increment 3.1.1 (`p2mem` 0.3.1) — LAS ingestion, deviation-survey ingestion, minimum-curvature trajectory validation, and MD-to-TVD/TVDSS depth mapping for the four approved wells — has passed independent technical review (261 tests, clean-room re-extraction verified) and is treated as **LOCKED**. `p2mem/units.py`, `p2mem/models.py`, `p2mem/deviation_models.py`, `p2mem/trajectory.py`, `p2mem/depth_mapping.py`, `p2mem/io/las.py`, `p2mem/io/inventory.py`, `p2mem/io/deviation.py`, `p2mem/io/deviation_inventory.py`, `config/las_curve_contracts.yml`, and `config/deviation_survey_contracts.yml` are **not modified** in this notebook. This increment builds strictly on that foundation: it never re-parses, re-derives, or replaces the locked `petrel_source_trace` survey trajectory or the locked canonical LAS `VP_m_s`/`MD_m` arrays.

### Technical Objective

Add a validated checkshot (velocity survey) ingestion and time-depth layer on top of the locked foundation, for exactly three APPROVED checkshot files:

1. Strict, auditable ingestion of each admitted checkshot file, with an explicit per-file contract (`config/checkshot_contracts.yml`) keyed by exact filename and verified SHA-256 — raw files are never rewritten, renamed, or "cleaned".
2. Raw QC and provenance: row count, duplicate/repeated-tie detection, non-increasing steps, acquisition gaps, and an explicit, auditable duplicate-tie conditioning policy that never silently averages or forces monotonicity with artificial values.
3. A checkshot-Depth-vs-locked-survey-TVDSS comparison for every admitted well, evaluating (not assuming) whether the checkshot file's "Depth" column behaves as measured depth.
4. OWT/TWT handling via the LOCKED Increment 1 `owt_to_twt`/`twt_to_owt` unit functions, and average/interval velocity diagnostics with explicit NaN-on-invalid-interval semantics (never infinite or negative velocity).
5. Forward/inverse piecewise-linear time-depth interpolation, strictly within validated checkshot coverage — no extrapolation, ever — including a partial-coverage LAS MD-to-checkshot-time mapping for Poseidon 2.
6. A Poseidon-2-only sonic-checkshot drift diagnostic (trapezoidal integration of sonic slowness vs. the checkshot-interpolated OWT increment over an identical MD/Depth interval) — diagnostic only, no correction applied.

**Explicitly NOT implemented in this increment:** formation-top correction, lithology interpretation, shale-volume calculation, density modelling/shallow density reconstruction, vertical-stress integration, pore-pressure prediction (Eaton/Bowers/Gardner/resistivity-pressure methods), elastic properties, rock strength, stress modelling, wellbore-stability analysis, drift correction of sonic or checkshot data, or synthetic extension beyond measured checkshot coverage. Those belong to later, explicitly gated increments. **Increment 5 has not been started.**

### Approved Inputs and Well Roles

Only three checkshot files are admitted, by exact filename and independently verified SHA-256 (never a same-well file under a different name, e.g. `Poseidon1-Checkshot.txt`, `Kronos1-Checkshot.txt`, `Pharos1-Checkshot.txt` — none of those are approved substitutes):

| File | Project well | Identity evidence | Model use |
|---|---|---|---|
| `Poseidon2-Checkshot.txt` | Poseidon 2 | verified (unambiguous filename match) | **primary_model** — the only checkshot approved to define Poseidon 2's primary time-depth relationship |
| `Boreas1-Checkshot.txt` | Boreas 1 | verified (unambiguous filename match) | qc_only — newly admitted in this increment as supporting QC data; never transferred into Poseidon 2 |
| `Proteus1-Checkshot.txt` | Proteus 1ST2 | **inferred_unverified** — no embedded well identifier; association rests on filename/context/depth-tie plausibility only | qc_only — never described as verified |

**Poseidon North 1 has no approved checkshot file.** This is recorded as `checkshot_availability: NOT_AVAILABLE` — a factual data gap, never an ingestion failure, and never filled by substituting another well's file.

### Theory and Physical Basis

**One-way vs. two-way time.** Every admitted file's header states OWT (one-way time, seconds), "vertically corrected, relative to SRD (MSL)". Two-way time is obtained via the LOCKED, already-tested Increment 1 relationship $TWT = 2 \times OWT$ (`p2mem.units.owt_to_twt`) — never reimplemented here.

**Average and interval velocity.** $V_{avg} = TVDSS / OWT$ at each conditioned checkshot row; $V_{int,k} = \Delta TVDSS_k / \Delta OWT_k$ between consecutive conditioned rows. Both are transparent, definitional ratios — not empirical correlations. A non-positive $\Delta TVDSS$ or $\Delta OWT$ yields `NaN` with an explicit QC flag, never an infinite or negative velocity.

**Depth-basis evaluation.** Every admitted file's first column is labelled only "Depth" — never explicitly "MD". This notebook never silently renames it; it is preserved as `Depth_source_m` and evaluated empirically against the LOCKED `petrel_source_trace` survey MD→TVDSS relationship for the same well (piecewise-linear interpolation, `numpy.interp`), with the residual statistics (min/max/max-abs/mean/median/RMSE/endpoints) reported below — never hardcoded ahead of the actual recomputation.

**Duplicate-tie conditioning (Depth axis).** A small number of Depth values repeat exactly in the raw Poseidon 2 and Boreas 1 files (Proteus 1ST2 has none). Every raw row is preserved exactly; a separate Depth-CONDITIONED representation collapses each repeated-Depth group to one deterministic representative row using that group's per-column MEDIAN (documented as equal to the arithmetic mean for the size-2 groups observed here — a disclosed, not silent, choice). Every tie is individually registered (`checkshot_duplicate_tie_register.csv`).

**Order-invariant axis-tie conditioning for TVDSS↔OWT/TWT inversion (Increment 4.1).** Depth-tie conditioning alone does **not** guarantee that `TVDSS_conditioned_m` or `OWT_conditioned_s` is itself strictly increasing — `numpy.interp`'s requirement for TIME-domain (inverse) interpolation. Real, distinct Depth values in this project's actual files carry an equal TVDSS or OWT at adjacent depths: Poseidon 2 has two such TVDSS-axis ties and two such OWT-axis ties after Depth-tie conditioning; Boreas 1 has one TVDSS-axis tie and zero OWT-axis ties; Proteus 1ST2 has none of either (independently reproduced counts — see `INCREMENT_04_1_MANIFEST.md`; an earlier statement in `INCREMENT_04_MANIFEST.md` that TVDSS was strictly increasing after conditioning for all three wells was INCORRECT and has been corrected). Every tied value on the axis being inverted is grouped by exact equality regardless of parse order, its complete original rows are registered in a separate, typed audit register (`checkshot_time_axis_tie_register.csv` — distinct from, and never confused with, the Depth-axis register above), and the group's dependent-value MEDIAN becomes the axis-conditioned representative — order-invariant, and disclosed as reducing to the arithmetic mean for the size-2 groups observed here. A group whose tied rows also share an identical dependent value (`tie_kind="identical_pair"`) still collapses without changing the value, but is still counted and registered. A genuine reversal (not a tie) in the axis being inverted raises a typed error rather than being sorted, discarded, or forced monotonic — checked (Increment 4.1.1 fix) on the ORIGINAL, ungrouped axis sequence before any grouping is attempted, so a reversal that returns to an already-seen value can no longer be silently absorbed by grouping (see Step 11c). See Step 11b below for the real, recomputed counts and forward/inverse demonstration.

**Interpolation and integration methods.** All interpolation is piecewise-linear (`numpy.interp`) — never a spline — and never extrapolates: samples outside a validated coverage range are `NaN`, with explicit coverage counts. The sonic-checkshot drift diagnostic integrates slowness ($1/V_P$) against MD via the standard trapezoidal rule, implemented locally (`p2mem.time_depth.trapezoidal_integrate`) rather than via `numpy.trapz`/`trapezoid` (mutually exclusive across NumPy versions), over the longest contiguous run of finite, positive canonical `VP_m_s` samples — an algorithmic, reproducible selection criterion, not a hand-picked interval.

> **QUALITY-CONTROL NOTE:** this notebook reports checkshot-vs-survey residuals and the sonic-checkshot drift comparison as OBSERVED, diagnostic findings — a near-constant residual offset is described as "an observed datum-like offset pattern", never as "a proven datum error", and the sonic-drift comparison never modifies `VP_m_s`, `DTCO`, checkshot OWT, or the conditioned time-depth curve.

### Input Data and Contracts

#### Step 1 — Mount Google Drive

**Technical objective:** re-attach the persistent project folder from prior increments.

**Inputs/outputs:** no project inputs are read here; this only establishes the `/content/drive` mount point.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


#### Step 2 — Verify the Increment 3.1.1 foundation is present and locked

**Technical objective:** confirm that the locked LAS-ingestion, deviation-survey, and depth-mapping layers already exist in this Drive project folder, BEFORE this notebook adds anything on top. Increment 4 is explicitly instructed not to modify these files unless an actual blocking defect is demonstrated — none is claimed here.

**Failure behavior:** raises `RuntimeError` naming exactly which expected file is missing. This is a hard gate, not a warning.

In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = "/content/drive/MyDrive/Poseidon_1D_MEM"

_required_locked_files = [
    "pyproject.toml",
    os.path.join("p2mem", "__init__.py"),
    os.path.join("p2mem", "units.py"),
    os.path.join("p2mem", "models.py"),
    os.path.join("p2mem", "deviation_models.py"),
    os.path.join("p2mem", "trajectory.py"),
    os.path.join("p2mem", "depth_mapping.py"),
    os.path.join("p2mem", "io", "las.py"),
    os.path.join("p2mem", "io", "inventory.py"),
    os.path.join("p2mem", "io", "deviation.py"),
    os.path.join("p2mem", "io", "deviation_inventory.py"),
    os.path.join("config", "las_curve_contracts.yml"),
    os.path.join("config", "deviation_survey_contracts.yml"),
]
_missing = [f for f in _required_locked_files if not os.path.exists(os.path.join(PROJECT_ROOT, f))]
if _missing:
    raise RuntimeError(
        "Locked Increment 3.1.1 foundation is missing file(s): "
        + ", ".join(_missing)
        + ". Run the Increment 3.1.1 notebook first; Increment 4 does not "
        "reconstruct the locked foundation from memory."
    )
print("Locked Increment 3.1.1 foundation confirmed present:")
for f in _required_locked_files:
    print("  -", f)


#### Step 2b — Enter the project root before any relative file write

**Increment 3.1.1 lesson applied from the start:** a real Google Colab `Run all` from a fresh runtime previously exposed a working-directory execution-order defect in the Increment 3 notebook — Colab starts in `/content`, not `PROJECT_ROOT`, and every `%%writefile` cell below writes a RELATIVE path. This cell performs the `chdir` immediately after `PROJECT_ROOT` is defined and the locked foundation is confirmed present (Step 2), and strictly before the first relative `%%writefile` cell (Step 6) — never deferred to a later `%cd` cell.

**Failure behavior:** raises `RuntimeError` if the working directory does not actually resolve to `PROJECT_ROOT`, or if `p2mem` is not a directory there. Never silently creates a substitute directory.

In [ ]:
os.chdir(PROJECT_ROOT)

if Path.cwd().resolve() != Path(PROJECT_ROOT).resolve():
    raise RuntimeError(
        f"Failed to enter the project root. "
        f"Expected {PROJECT_ROOT}, actual working directory: {Path.cwd()}"
    )

if not Path("p2mem").is_dir():
    raise RuntimeError(
        f"Required p2mem directory is missing under {PROJECT_ROOT}. "
        "Extract the approved Increment package before running this notebook."
    )

print("Working directory confirmed:", Path.cwd())


#### Step 3 — Create the Increment 4 directory additions

**Technical objective:** create the new directories this increment needs, without touching any existing directory.

**Expected result:** `data/raw/checkshot/`, `outputs/04_checkshot_time_depth/` and `outputs/04_checkshot_time_depth/figures/` exist.

In [ ]:
for d in ["data/raw/checkshot", "outputs/04_checkshot_time_depth", "outputs/04_checkshot_time_depth/figures"]:
    os.makedirs(os.path.join(PROJECT_ROOT, d), exist_ok=True)
print("Increment 4 directories ready.")


#### Step 4 — Verify the three approved checkshot filenames are present (exact names only), and their SHA-256

**Technical objective:** require the exact approved filenames — `Poseidon2-Checkshot.txt`, `Boreas1-Checkshot.txt`, `Proteus1-Checkshot.txt` (WITH hyphens) — under `data/raw/checkshot/`, and stop cleanly, naming every missing file, if any are absent. No substitute file (`Poseidon1-Checkshot.txt`, `Kronos1-Checkshot.txt`, `Pharos1-Checkshot.txt`, or any other) is ever accepted in place of these three, and Poseidon North 1 is never given a substitute checkshot file.

**Assumptions:** the user has uploaded these three files to `/content/drive/MyDrive/Poseidon_1D_MEM/data/raw/checkshot/` before running this cell — this notebook cannot fetch them itself, they are private project inputs.

**Failure behavior:** raises `RuntimeError` listing every missing filename by its exact expected name.

In [ ]:
import hashlib

CK_DIR = os.path.join(PROJECT_ROOT, "data", "raw", "checkshot")
CK_FILES = {
    "Poseidon_2": "Poseidon2-Checkshot.txt",
    "Boreas_1": "Boreas1-Checkshot.txt",
    "Proteus_1ST2": "Proteus1-Checkshot.txt",
}

_missing_ck = [fn for fn in CK_FILES.values() if not os.path.exists(os.path.join(CK_DIR, fn))]
if _missing_ck:
    raise RuntimeError(
        "Missing required checkshot file(s) under "
        f"{CK_DIR}: {_missing_ck}. Increment 4 requires exactly these three "
        "approved checkshot files, under these exact (hyphenated) names - no "
        "other file is accepted as a substitute, and Poseidon North 1 has no "
        "approved checkshot file at all (recorded as NOT_AVAILABLE below)."
    )

print("All three approved checkshot files found. SHA-256:")
for key, fn in CK_FILES.items():
    p = os.path.join(CK_DIR, fn)
    print(f"  {key} ({fn}): {hashlib.sha256(open(p, 'rb').read()).hexdigest()}")


#### Step 5 — Install dependencies

**Technical objective:** install exactly the packages this increment's code and notebook display/plotting cells need. No new runtime dependency is added — `p2mem` itself still depends only on NumPy and PyYAML (see `pyproject.toml`); `pandas`/`matplotlib` remain notebook-only, unchanged from Increment 3.

In [ ]:
!pip install -q numpy pyyaml pytest pandas matplotlib


### Input Data and Contracts

#### Step 6 — Write the Increment 4 package files

**Technical objective:** write every new/updated source file for this increment, verbatim from the tested files on disk (this build script never hand-retypes code into notebook cells).

**New modules:** `p2mem/checkshot_models.py` (typed dataclasses), `p2mem/time_depth.py` (duplicate-tie conditioning, velocity diagnostics, forward/inverse time-depth interpolation, checkshot-vs-survey comparison, sonic-checkshot drift), `p2mem/io/checkshot.py` (checkshot parser and per-file contract resolver), `p2mem/io/checkshot_inventory.py` (deterministic output-table builders).

**Updated (Increment 4.1 corrective patch — version/documentation and numerical-validation hardening):** `pyproject.toml`, `p2mem/__init__.py`, `README.md` — package version bumped to 0.4.1; `p2mem/time_depth.py` (order-invariant axis-tie conditioning replacing the removed, order-dependent `_build_strictly_increasing_table`; numerical-validation hardening of `trapezoidal_integrate` and `compute_sonic_checkshot_drift`); `p2mem/checkshot_models.py` (new `AxisTimeDepthTieRegisterEntry` / `AxisConditionedLookupTable` types); `p2mem/io/checkshot.py` and `p2mem/io/checkshot_inventory.py` (wiring + the new axis-tie register/manifest fields).

**Updated again (Increment 4.1.1 numerical-validation corrective patch):** `pyproject.toml`, `p2mem/__init__.py`, `README.md` — package version bumped to 0.4.1.1; `p2mem/time_depth.py` (reversal detection on the ORIGINAL, ungrouped axis sequence before any grouping; full-canonical-MD validation in `compute_sonic_checkshot_drift`/`find_longest_finite_positive_run`; a typed `TimeDepthError` for zero in-coverage checkshot rows in `compare_checkshot_to_survey`; a local `_reject_ambiguous_dtype` guard wired into `seconds_to_milliseconds`/`milliseconds_to_seconds`); `p2mem/io/checkshot.py` (`load_checkshot_surveys` now catches `TimeDepthError` per well, isolating a numerical-conditioning failure exactly like every other expected per-well failure). Every locked file from prior increments is intentionally NOT rewritten here.

**Increment 4.1.2 (this notebook rebuild):** packaging/notebook correction only — **no `p2mem` source file is changed and the version stays at 0.4.1.1.** Only this notebook itself (`tests/fixtures/checkshot_valid.txt` now written via Step 7a's explicit-bytes cell rather than `%%writefile`; Step 9's test-running cell; the Completion Gate cell) and `INCREMENT_04_1_1_MANIFEST.md` (a correction notice — see `INCREMENT_04_1_2_MANIFEST.md` Section 1) change.

In [ ]:
%%writefile pyproject.toml
[build-system]
requires = ["setuptools>=68.0"]
build-backend = "setuptools.build_meta"

[project]
name = "p2mem"
version = "0.4.1.1"
description = "Screening-level 1D Mechanical Earth Model workflow for Poseidon 2 (Tier C, uncalibrated / educational)."
readme = "README.md"
requires-python = ">=3.9"
license = { text = "All Rights Reserved. Copyright (c) 2026 Mikael Elgo. This is a personal portfolio project; no license is granted for reuse, redistribution, or commercial use without the author's explicit written permission." }
authors = [
    { name = "Mikael Elgo" }
]
keywords = ["geomechanics", "mechanical-earth-model", "pore-pressure", "wellbore-stability", "portfolio-project"]
classifiers = [
    "Development Status :: 3 - Alpha",
    "Programming Language :: Python :: 3",
    "Intended Audience :: Science/Research",
    "Topic :: Scientific/Engineering",
    "License :: Other/Proprietary License",
]

# Runtime dependencies are deliberately minimal. No unit-handling libraries
# (e.g. Pint) are used: unit conversions are implemented explicitly in
# p2mem.units so that every conversion factor is visible, documented, and
# testable rather than delegated to a third-party unit registry. PyYAML is
# added in Increment 2 for exactly one purpose: parsing the human-authored,
# human-reviewable per-file LAS curve contracts in
# config/las_curve_contracts.yml - a plain-text, diffable format was judged
# preferable to a hand-rolled config parser or a hard-coded Python dict.
dependencies = [
    "numpy>=1.24",
    "pyyaml>=6.0",
]

[project.optional-dependencies]
dev = [
    "pytest>=7.4",
]

[tool.setuptools.packages.find]
include = ["p2mem*"]

[tool.pytest.ini_options]
testpaths = ["tests"]
python_files = ["test_*.py"]


In [ ]:
%%writefile p2mem/__init__.py
"""
p2mem - Poseidon 2 1D Mechanical Earth Model workflow package.

Project classification: Tier C - Screening-Level / Uncalibrated Educational
1D Mechanical Earth Model (see project design review, Rev 1). Nothing in
this package should be presented as a calibrated, operational, or
field-validated result unless an explicit independent calibration record
is attached to that specific output.

This package is under incremental, gated construction.

* Increment 1 / 1.1 delivered the project skeleton and the unit-control
  system (``p2mem.units``).
* Increment 2 added an auditable LAS-ingestion layer with explicit
  per-file curve contracts (``p2mem.io.las``, ``p2mem.io.inventory``,
  ``p2mem.models``) for the four approved wells (Poseidon 2, Boreas 1,
  Poseidon North 1, Proteus 1ST2). It performs LAS parsing, curve-identity
  resolution, NULL-sentinel handling, and factual inventory generation
  ONLY - no deviation-survey processing, MD-to-TVD/TVDSS transformation,
  checkshot processing, formation-top correction, petrophysical
  interpretation, or any later-phase geomechanical calculation.
* Increment 2.1 / 2.1.1 are corrective patches to Increment 2, applied
  after independent technical audits, WITHOUT changing scope or the
  underlying LAS-parsing/curve-resolution architecture (which both audits
  found sound). 2.1 corrected: canonical array naming (every array is now
  explicitly unit-suffixed, e.g. ``VP_m_s`` rather than ``DTCO``, so a
  name can never be mistaken for the wrong physical quantity or unit);
  the measured-depth curve is now located via an explicit contract role
  rather than by matching a canonical name spelled "DEPT"; several
  file-identity checks (filename, SHA-256, WELL, VERS, WRAP, NULL) that
  were not previously blocking now are; curve-coverage statistics now
  report raw AND canonical values with explicit units; and per-well
  batch failures are now typed (``p2mem.models.IngestionFailure``)
  instead of bare caught exceptions. 2.1.1 corrected a packaging-only gap
  (three notebook ``%%writefile`` cells that had drifted from their
  packaged source files). See ``INCREMENT_02_v2.1_MANIFEST.md`` and
  ``INCREMENT_02_v2.1.1_MANIFEST.md`` for the full audits and
  corrected-file checksums.
* Increment 3 adds Petrel deviation-survey ingestion with explicit
  per-file contracts (``p2mem.io.deviation``), a standard minimum-
  curvature trajectory engine with a numerically stable ratio-factor
  limit (``p2mem.trajectory``), an explicit MD-referenced/TVD-referenced/
  TVDSS depth-reference framework and MD-to-TVD/TVDSS interpolation with
  no silent extrapolation (``p2mem.depth_mapping``), and typed dataclasses
  for all of the above (``p2mem.deviation_models``) - for the same four
  approved wells. It independently reproduces the Petrel-supplied
  trajectory to millimetre scale for three of the four wells and
  discloses (rather than resolves) a real, larger trajectory-
  reconstruction discrepancy found in Proteus 1ST2's deeper section - see
  ``INCREMENT_03_MANIFEST.md``. It performs deviation-survey ingestion,
  trajectory validation, and depth mapping ONLY - no checkshot
  processing, formation-top correction, petrophysical interpretation, or
  any later-phase geomechanical calculation.
* Increment 3.1 is a corrective patch to Increment 3, applied after an
  independent technical audit, WITHOUT changing scope, equations, real
  well data, or the locked LAS/Increment-2.1.1 foundation. It corrected
  four defects: (1) the four deviation-survey source filenames are the
  exact, literal names as they exist in Google Drive, which contain
  spaces (e.g. ``"Poseidon 2_dev.txt"``) - Increment 3 had incorrectly
  substituted underscores in the contract keys, notebook mapping, and
  tests, which would have failed to resolve against the real files;
  internal well keys (e.g. ``Poseidon_2``) remain underscored and are
  unaffected; (2) every exported CSV/JSON/manifest field is now
  guaranteed to carry a basename only, never a full environment-dependent
  build path (runtime-only diagnostic objects may still retain one);
  (3) the previously undisclosed inference that the supplied ``DLS``
  column is normalized as degrees per 30 metres is now explicitly flagged
  with a new, independently per-file-verified
  ``DLS_NORMALIZATION_INFERRED_AS_DEG_PER_30M`` WARNING (mirroring the
  pre-existing MD-unit-inference warning); (4) the dogleg angle between
  successive stations is now computed with a numerically stable
  ``arctan2(||cross||, dot)`` vector formulation (``p2mem.trajectory``)
  instead of ``arccos``, which was ill-conditioned near a zero dogleg and
  previously reported a spurious ~1e-6-degree value for two stations with
  identical inclination/azimuth. The real four-well data, station counts,
  tolerances, and the unresolved Proteus 1ST2 trajectory discrepancy are
  all unchanged by this patch. See ``INCREMENT_03_1_MANIFEST.md`` for the
  full audit and re-verification record.

* Increment 4 adds checkshot (velocity survey) ingestion with explicit
  per-file contracts (``p2mem.io.checkshot``, ``config/checkshot_
  contracts.yml``), typed checkshot dataclasses (``p2mem.checkshot_
  models``), deterministic checkshot inventory/QC-table builders
  (``p2mem.io.checkshot_inventory``), and a numerical time-depth layer
  (``p2mem.time_depth``): duplicate-tie detection/conditioning, average/
  interval velocity diagnostics, checkshot-vs-locked-survey depth-
  reference comparison, forward/inverse piecewise-linear time-depth
  interpolation with explicit coverage masking (no extrapolation), LAS
  MD-to-checkshot-time mapping within validated checkshot coverage only,
  and a Poseidon-2-only sonic-checkshot drift diagnostic (trapezoidal
  integration of sonic slowness vs. the checkshot-interpolated OWT
  increment over the same MD/Depth interval). Three checkshot files are
  admitted: ``Poseidon2-Checkshot.txt`` (Poseidon 2 - the ONLY checkshot
  approved to define a primary time-depth relationship),
  ``Boreas1-Checkshot.txt`` and ``Proteus1-Checkshot.txt`` (Boreas 1 and
  Proteus 1ST2 - supporting QC data only, newly admitted in this
  increment, never transferred into Poseidon 2 as a substitute time-depth
  model). Proteus 1ST2's association with ``Proteus1-Checkshot.txt`` is
  explicitly disclosed as inferred/unverified (no embedded well
  identifier). Poseidon North 1 has no approved checkshot file
  (``checkshot_availability: NOT_AVAILABLE`` - a factual data gap, not an
  ingestion failure). This increment reuses the LOCKED Increment 1
  ``owt_to_twt``/``twt_to_owt`` unit functions and the LOCKED Increment
  3/3.1.1 ``petrel_source_trace`` survey trajectory unchanged; it performs
  checkshot QC and time-depth framework work ONLY - no formation-top
  correction, lithology interpretation, density modelling, pore-pressure
  prediction, elastic properties, rock strength, stress modelling, or
  wellbore-stability analysis. See ``INCREMENT_04_MANIFEST.md`` for the
  full technical detail and independently recomputed statistics. NOTE:
  ``INCREMENT_04_MANIFEST.md`` contained one identified defect, corrected
  by Increment 4.1 below - do not rely on its original, uncorrected
  statement that TVDSS is strictly increasing after Depth-tie conditioning
  for all three wells.

* Increment 4.1 is a narrowly scoped corrective patch to Increment 4,
  applied after an independent technical/numerical-method audit, WITHOUT
  beginning Increment 5 or any formation-top/petrophysics/pore-pressure/
  mechanical-properties/stress/wellbore-stability work. It corrected two
  defects and hardened two numerical contracts: (1) ``tvdss_to_owt``/
  ``owt_to_tvdss`` (and their TWT equivalents) previously resolved a
  repeated (tied) value on the axis being inverted by silently keeping
  whichever tied row appeared first in the Depth-conditioned table and
  discarding the other (``p2mem.time_depth._build_strictly_increasing_
  table``, REMOVED) - an ORDER-DEPENDENT tie-break with no audit trail
  beyond a bare count. This is replaced by ``build_axis_conditioned_
  lookup_table``/``build_axis_conditioned_tables_for_well``: an explicit,
  ORDER-INVARIANT policy that groups every tied value by exact equality
  regardless of parse order, registers every tied row in a new typed
  audit register (``AxisTimeDepthTieRegisterEntry`` /
  ``checkshot_time_axis_tie_register.csv`` - separate from, and never
  confused with, the pre-existing Depth-axis ``DuplicateTieRegisterEntry``
  register), and uses the tied group's dependent-value MEDIAN as the
  conditioned representative (order-invariant; disclosed as reducing to
  the arithmetic mean for the size-2 groups observed in this project's
  real data). A genuine reversal (not a tie) in the axis being inverted
  raises ``TimeDepthError`` rather than being sorted, discarded, or forced
  monotonic. (2) ``INCREMENT_04_MANIFEST.md``'s statement that TVDSS is
  strictly increasing after Depth-tie conditioning for all three wells was
  INCORRECT - independently reproduced counts (Poseidon 2: two TVDSS-axis
  and two OWT-axis ties; Boreas 1: one TVDSS-axis tie, zero OWT-axis ties;
  Proteus 1ST2: none of either) are now documented in
  ``INCREMENT_04_1_MANIFEST.md`` and reflected in this module's own
  docstrings. (3) ``trapezoidal_integrate`` and (4) ``compute_sonic_
  checkshot_drift`` are hardened to validate their numerical
  preconditions (finite, one-dimensional, equal-length, strictly
  increasing MD/x where required) rather than silently integrating
  invalid input - most notably, a decreasing or duplicate MD run can no
  longer silently produce a physically invalid NEGATIVE transit time; it
  now raises ``TimeDepthError``. None of this hardening changes the
  already-verified real Poseidon 2 sonic-drift result, which is
  bit-for-bit unchanged. See ``INCREMENT_04_1_MANIFEST.md`` for the full
  audit, corrected statistics, and re-verification record.

* Increment 4.1.1 is a narrowly scoped numerical-validation corrective
  patch to Increment 4.1, applied after an independent numerical-method/
  software-QA audit, WITHOUT beginning Increment 5 or any formation-top/
  petrophysics/pore-pressure/mechanical-properties/stress/wellbore-
  stability work. It corrected four blocking defects and one input-safety
  gap, none of which altered any previously verified REAL Poseidon
  2/Boreas 1/Proteus 1ST2 result: (1) ``build_axis_conditioned_lookup_
  table`` grouped ALL occurrences of an identical axis value together
  GLOBALLY before checking for a reversal, so a reversal that returned to
  an already-seen value (e.g. ``[100.0, 200.0, 100.0]``) was silently
  hidden rather than raising ``TimeDepthError`` - it now evaluates the
  ORIGINAL, ungrouped sequence's successive differences for negativity
  BEFORE any grouping is attempted, which is provably equivalent to the
  4.1 behavior for every legitimate adjacent tie and strictly stronger
  against a non-adjacent reversal. (2) ``compute_sonic_checkshot_drift``/
  ``find_longest_finite_positive_run`` validated MD monotonicity only
  within the selected finite-positive-VP run, so a decreasing or
  duplicate MD value outside that run (e.g. at a NaN-VP station) could
  pass silently - the COMPLETE canonical ``md_m`` array is now required
  finite and strictly increasing before run-selection (the real Poseidon
  2 MD array, 31,897 samples, was independently re-verified to already
  satisfy this). (3) ``compare_checkshot_to_survey`` reached an untyped
  NumPy ``ValueError`` ("zero-size array to reduction operation") if
  every checkshot Depth row fell outside the locked survey's own MD
  coverage - it now raises a typed ``TimeDepthError`` naming the well,
  the checkshot Depth range, and the survey MD coverage. (4)
  ``p2mem.io.checkshot.load_checkshot_surveys`` did not catch
  ``TimeDepthError`` raised during numerical conditioning, so a defect in
  one well's data could stop the entire batch - it is now caught per well
  (never via a blanket ``except Exception``) and recorded as a typed
  ``CheckshotIngestionFailure(error_type="numerical_conditioning_
  failure")``, isolated exactly like every other expected per-well
  failure. (5) ``seconds_to_milliseconds``/``milliseconds_to_seconds``
  coerced their input directly, unlike ``p2mem.units``'s Increment-1
  input-safety policy, so a boolean, numeric-looking string, or complex
  value would be silently reinterpreted rather than rejected - both now
  reject such input with ``TypeError`` via a local, documented copy of
  ``p2mem.units``'s identical private dtype check (``p2mem/units.py``
  itself remains LOCKED and unmodified). See
  ``INCREMENT_04_1_1_MANIFEST.md`` for the full audit, the regression-test
  list, and the re-verification record.

Subsequent increments (formation-top correction, petrophysics, pore
pressure, elastic properties, strength, stress, and wellbore-stability
screening) are added one validated phase at a time and are intentionally
absent from this version - importing them will fail until they exist.
"""

__version__ = "0.4.1.1"

# Fixed project-wide assurance tier. Referenced by later modules (reporting,
# plotting) so that every generated output can stamp its own classification
# without each module re-declaring the string. This value must not be
# changed without a documented calibration event (e.g. a verified RFT/MDT,
# LOT/XLOT, or core-calibrated log tie) recorded in the method-and-citation
# register.
ASSURANCE_TIER = "Tier C - Screening-Level / Uncalibrated Educational"

__all__ = ["__version__", "ASSURANCE_TIER"]


In [ ]:
%%writefile README.md
# Poseidon 2 — 1D Mechanical Earth Model

**Author:** Mikael Elgo

**Project classification:** Tier C — Screening-Level / Uncalibrated Educational 1D Mechanical Earth Model (MEM)

> **This project is screening-level, uncalibrated, and educational in nature. It is NOT validated against independent field measurements (no confirmed RFT/MDT pressure points, LOT/XLOT tests, or core-calibrated log ties are currently incorporated), and it must NOT be used for operational drilling, well-design, or any real-world decision-making. It exists to demonstrate a technically defensible, transparent, modular geomechanics workflow — not to produce field-ready predictions.**

---

## Purpose and technical scope

This repository implements a modular, reproducible 1D Mechanical Earth Model workflow for the Poseidon 2 well, built from well logs, deviation surveys, checkshot data, formation tops, and Vp/Vs data supplied for the project. The intended end-to-end scope (delivered incrementally, one validated phase at a time) covers:

- data quality control and depth alignment across LAS logs, deviation surveys, and checkshot data
- pore-pressure prediction (Eaton-family methods, contingent on a defensible normal compaction trend)
- elastic properties (dynamic Vp/Vs-derived Poisson's ratio, and density-dependent moduli where density coverage permits)
- rock-strength estimation
- vertical-stress (overburden) modelling
- horizontal-stress and wellbore-stability screening (Kirsch elastic wall-stress equations with Mohr–Coulomb/Mogi–Coulomb failure criteria)
- uncertainty treatment via deterministic low/base/high scenarios and one-at-a-time sensitivity (tornado) analysis, rather than unsupported probabilistic distributions

Every empirical or correlation-based relationship used anywhere in this project (Eaton, Bowers, Gardner, Castagna, etc.) is required to have a recorded source, stated units, applicability range, and calibration status in the project's method-and-citation register *before* it is implemented in code. Nothing is fabricated or assumed silently: missing measurements, missing calibration points, and unavailable data are always reported as unavailable rather than filled in.

This is a personal portfolio project intended to demonstrate scientific rigor, reproducibility, and honest handling of data limitations — not a commercial or operational deliverable.

## Current implementation status

**Increment 4 / 4.1 / 4.1.1 (this release, v0.4.1.1): Checkshot (Velocity Survey) Ingestion, Duplicate-Tie Conditioning, and Time–Depth Framework.** Builds on the LOCKED Increment 3.1.1 deviation-survey/depth-mapping layer and the LOCKED Increment 2.1.1 LAS-ingestion layer (both unmodified - see below) by adding auditable checkshot parsing with explicit per-file contracts, raw-vs-conditioned duplicate-tie handling, average/interval velocity diagnostics, checkshot-vs-locked-survey depth-reference comparison, a coverage-masked forward/inverse piecewise-linear time-depth interpolation layer, LAS MD-to-checkshot-time mapping within validated coverage only, and a Poseidon-2-only sonic-checkshot drift diagnostic. See `INCREMENT_04_MANIFEST.md` for the full technical design and the independently recomputed real-data statistics, `INCREMENT_04_1_MANIFEST.md` for the Increment 4.1 corrective patch (order-invariant axis-tie conditioning for TVDSS↔OWT/TWT inversion, replacing an order-dependent defect; numerical-validation hardening — see "Increment 4.1 update" below), and `INCREMENT_04_1_1_MANIFEST.md` for the Increment 4.1.1 numerical-validation corrective patch (a hidden-reversal grouping defect, incomplete full-MD validation, an untyped zero-coverage crash, missing batch isolation for numerical failures, and a unit-helper input-safety gap — see "Increment 4.1.1 update" below). Checkshot data QC, duplicate-tie conditioning, and time-depth interpolation only — no formation-top correction, lithology interpretation, density modelling, pore-pressure prediction, elastic properties, rock strength, stress modelling, or wellbore-stability analysis is performed in this increment.

New in Increment 4:
- `p2mem/checkshot_models.py` — typed, frozen dataclasses for every checkshot header/contract/raw-row/duplicate-tie/velocity-diagnostic/depth-comparison/time-mapping/sonic-drift result object, mirroring the deviation-survey layer's design philosophy. Raw (`_source_`) values are always kept explicitly separate from conditioned (`_conditioned_`) values — never overwritten, never mixed.
- `p2mem/io/checkshot.py` — an auditable checkshot (velocity-survey) parser and per-file contract resolver for the three approved checkshot files (`Poseidon2-Checkshot.txt`, `Boreas1-Checkshot.txt`, `Proteus1-Checkshot.txt`), keyed by their exact literal source filenames. File-identity checks (filename, SHA-256, header/survey-statement text, column order, row width, numeric structure) are enforced as blocking `ERROR`s before any time-depth computation is attempted. Raises disclosure `WARNING`s including `DEPTH_BASIS_NOT_EXPLICITLY_DECLARED` (the source file's first column is labelled only `Depth`, never assumed to be MD without evidence) and, for Proteus 1ST2, `WELL_IDENTITY_INFERRED_UNVERIFIED` (the file carries no embedded well identifier tying it to Proteus 1ST2).
- `p2mem/time_depth.py` — the numerical time-depth layer: duplicate/repeated-tie detection and deterministic, disclosed median-based conditioning (raw rows always preserved and separately registered; ties never silently averaged, never force-monotonized with artificial epsilon increments); average velocity (`Vavg = TVDSS / OWT`) and interval velocity (`Vint = ΔTVDSS / ΔOWT`, NaN — never infinite or negative — for any invalid, zero, or non-increasing interval); checkshot-vs-locked-survey depth-reference comparison (residual = survey-interpolated TVDSS − checkshot-supplied TVDSS); forward/inverse piecewise-linear time-depth interpolation with explicit coverage masking (points outside validated checkshot coverage are reported as not-mapped, never extrapolated) via an order-invariant, median-based axis-tie-conditioned lookup table for TVDSS↔OWT/TWT inversion (Increment 4.1 — every tied value on the axis being inverted is grouped and registered, never resolved by an order-dependent "first wins" tie-break); LAS MD-to-checkshot-time mapping for Poseidon 2 within validated coverage only; and a Poseidon-2-only sonic-checkshot drift diagnostic (a local, version-independent trapezoidal integration of sonic slowness over the longest valid continuous MD interval, compared against the checkshot-interpolated OWT increment over the same interval — this diagnostic never modifies `VP_m_s`, `DTCO`, checkshot OWT, or the time-depth curve itself).
- `p2mem/io/checkshot_inventory.py` — deterministic, metadata-only inventory/QC-table builders for the Increment 4 / 4.1 outputs (file inventory, ingestion issues, Depth-axis duplicate-tie register, the Increment 4.1 TVDSS/OWT-axis tie register, depth-tie QC, velocity summary, sonic-checkshot drift summary, time-depth mapping summary, JSON manifest) — never raw per-sample checkshot or LAS arrays, and never a full environment-dependent build path, only a basename.
- `config/checkshot_contracts.yml` — the human-authored, human-reviewable per-file checkshot contract for each of the three approved files, including each well's `model_use_status` (`primary_model` for Poseidon 2 only; `qc_only` for Boreas 1 and Proteus 1ST2) and `identity_evidence_status` (`verified` for Poseidon 2 and Boreas 1; `inferred_unverified` for Proteus 1ST2).
- **Key real-data findings (disclosed):** Poseidon 2's raw checkshot file contains 5 repeated Depth ties, 6 non-increasing TVDSS steps, and 4 non-increasing OWT steps, all independently detected and conditioned (never silently smoothed); its checkshot-vs-survey TVDSS comparison shows a maximum absolute residual of ≈0.089 m with no systematic offset pattern. Boreas 1 shows 3 repeated Depth ties and 4 non-increasing TVDSS steps (OWT strictly increasing throughout) and a checkshot-vs-survey comparison with a near-constant offset of ≈−0.69 m, reported as an observed datum-like offset pattern — not a proven datum error. Proteus 1ST2's file is strictly increasing in all three columns with no repeated ties, and shows a near-constant offset of ≈+0.30 m against the locked survey trajectory; its association with Proteus 1ST2 remains `inferred_unverified` throughout every output. Poseidon 2's sonic-checkshot drift over its longest valid continuous sonic interval (≈MD 2449–4064 m) is ≈+16.7 ms (≈+4.4% of the checkshot-interpolated one-way transit time (OWT) increment over that interval — NOT two-way time; the diagnostic compares the integrated sonic transit time directly against the checkshot's OWT increment over the identical interval), reported as a diagnostic only. Poseidon North 1 has no approved checkshot file; this is recorded as `checkshot_availability: NOT_AVAILABLE`, a factual data gap, not an ingestion failure. Depth-tie conditioning alone does NOT guarantee TVDSS or OWT is itself strictly increasing (required for TVDSS↔OWT/TWT inversion) — see the Increment 4.1 update below for the corrected, order-invariant handling of this. See `INCREMENT_04_MANIFEST.md` for the full statistics, tables, and figures.
- Still not implemented: formation-top correction, lithology interpretation, density modelling, sonic/checkshot drift *correction*, synthetic extension of the time-depth relationship beyond measured checkshot coverage, pore-pressure prediction, elastic-property calculation, rock-strength estimation, overburden/horizontal stresses, or wellbore-stability calculations. Those remain explicitly out of scope for this increment.

**Increment 3 / 3.1 / 3.1.1: Deviation-Survey Ingestion, Minimum-Curvature Validation, and MD–TVD–TVDSS Depth Framework.** Builds on the LOCKED Increment 2.1.1 LAS-ingestion layer (unmodified - see below) by adding Petrel deviation-survey parsing, an explicit per-file survey contract, a standard minimum-curvature trajectory engine, an explicit depth-reference (MD/TVD/TVDSS) framework, and MD-to-TVD/TVDSS mapping of the existing LAS `MD_m` arrays. See `INCREMENT_03_MANIFEST.md` for the Increment 3 technical design and real four-well integration results, `INCREMENT_03_1_MANIFEST.md` for the Increment 3.1 corrective patch (four audit findings: source-filenames-with-spaces, absolute-path leakage, DLS-normalization disclosure, dogleg numerical stability — see "Increment 3.1 update" below), and `INCREMENT_03_1_1_MANIFEST.md` for the Increment 3.1.1 packaging-only corrective patch (notebook/source `%%writefile` synchronization; no scientific or numerical change). The Proteus 1ST2 trajectory-discrepancy finding is disclosed, not resolved, in any of these releases. This layer, and the Increment 2.1.1 LAS-ingestion layer beneath it, are LOCKED as of Increment 4 and reused unmodified.

Locked from Increment 2.1.1 (unmodified in Increment 3 unless a blocking defect is documented - none was found):
- `p2mem/units.py` — an explicit, NumPy-based unit-conversion layer (no external unit-registry dependency such as Pint) implementing 21 public conversion functions between oilfield and SI-internal units. Unchanged since Increment 1.1. See the module docstring and `tests/test_units.py`.
- `p2mem/models.py`, `p2mem/io/las.py`, `p2mem/io/inventory.py`, `config/las_curve_contracts.yml` — the auditable LAS 2.0 parser, per-file curve-contract resolver, and inventory builders for the four approved wells, corrected and independently re-verified through Increment 2.1.1 (158 tests passing, 4/4 real wells loading with zero ingestion errors). See `INCREMENT_02_v2.1.1_MANIFEST.md`.

New in Increment 3:
- `p2mem/deviation_models.py` — typed, frozen dataclasses for every deviation-survey/trajectory/depth-mapping result object (header info, per-file contract, raw station data, minimum-curvature result, trajectory-validation residuals, depth-basis selection, typed batch failures, LAS depth-mapping result), mirroring the LAS layer's design philosophy. Every source (`_source_`) array is kept explicitly separate from every independently computed (`_mc_`) array — never overwritten, never mixed.
- `p2mem/io/deviation.py` — an auditable Petrel deviation-survey (well-trace) parser and per-file contract resolver for the same four wells, keyed by their exact, literal source filenames (which contain spaces, e.g. `"Poseidon 2_dev.txt"` — corrected in Increment 3.1; see below). Extracts and preserves the full header block (well/survey identity, wellhead X/Y, datum and its MSL reference, coordinate-reference-system text, declared angle/depth/coordinate conventions) and the exact 11-column station table, with file-identity checks (filename, SHA-256, well/survey identifier, wellhead/datum values, coordinate system, column order, station count, MD coverage) all enforced as blocking `ERROR`s before any trajectory computation is attempted. Also raises two disclosure `WARNING`s for every successfully loaded file: the pre-existing `MD_UNIT_NOT_EXPLICITLY_DECLARED`, and the Increment 3.1 `DLS_NORMALIZATION_INFERRED_AS_DEG_PER_30M` (the supplied `DLS` column's degrees-per-30-m normalization is inferred, not header-declared, and is independently verified per file against a recomputation from that file's own inclination/azimuth).
- `p2mem/trajectory.py` — the standard minimum-curvature method (a numerically stable `arctan2(||cross||, dot)` dogleg-angle formulation — Increment 3.1 correction, see below — a ratio factor with an explicit Taylor-series limit as the dogleg approaches zero, TVD/northing/easting displacement, dogleg severity in degrees per 30 m), implemented explicitly and transparently with no third-party survey-computation library.
- `p2mem/depth_mapping.py` — MD-to-TVD/TVDSS interpolation of the locked LAS `MD_m` array against the explicitly selected depth-trajectory basis, using a documented, deterministic piecewise-linear station interpolation (never a per-sample minimum-curvature recomputation) that never extrapolates silently.
- `p2mem/io/deviation_inventory.py` — deterministic, metadata-only inventory-table builders for the Increment 3 outputs (file inventory, trajectory-validation summary, depth-reference register, LAS depth-mapping summary, ingestion issues, JSON manifest) — never raw per-sample station or LAS arrays, and (Increment 3.1 correction) never a full environment-dependent build path, only a basename.
- `config/deviation_survey_contracts.yml` — the human-authored, human-reviewable per-file deviation-survey contract for each of the four wells, keyed by the exact literal source filename (`"Poseidon 2_dev.txt"`, `"Boreas 1_dev.txt"`, `"Poseidon North 1_dev.txt"`, `"Proteus 1ST2_dev.txt"` — corrected in Increment 3.1), including an explicit, uniformly applied `depth_basis_policy` (`petrel_source_trace`, the conservative default given the Proteus 1ST2 finding below) and residual-comparison tolerances declared once and applied identically to every well (never tuned per well to force a pass/fail outcome).
- **Key real-data finding (disclosed, not resolved):** independent minimum-curvature reconstruction of Poseidon 2, Boreas 1, and Poseidon North 1 agrees with their Petrel-supplied TVD to approximately millimetre scale. Proteus 1ST2 shows a materially larger discrepancy (~0.19 m TVD, ~1.6 m easting at maximum) concentrated in its deeper section (below ~MD 4200 m), even though its own supplied dogleg-severity column is internally consistent with an independent recomputation from its own inclination/azimuth at every station. This is reported as a visible trajectory-validation `WARNING`, not corrected, hidden, or used to justify loosening every well's tolerance — see `INCREMENT_03_MANIFEST.md` Section 6 for the full investigation and the evidence pattern observed. Unaffected by the Increment 3.1 patch.
- Still not implemented (as of Increment 3/3.1/3.1.1): checkshot ingestion, time-depth conversion, formation-top correction, petrophysical interpretation, gamma-ray normalization, shale-volume calculation, lithology classification, normal-compaction-trend fitting, pore-pressure prediction, elastic-property calculation, rock-strength estimation, overburden/horizontal stresses, or wellbore-stability calculations. Those were explicitly out of scope for this increment. Checkshot ingestion and the time-depth framework were subsequently added in Increment 4 (see above); the remainder are added one gated increment at a time in later releases.

## Installation

Requires Python 3.9 or later.

```bash
# from the project root (the directory containing pyproject.toml)
pip install -e .
```

This installs the `p2mem` package in editable mode along with its runtime dependencies: NumPy (`numpy>=1.24`) and, as of Increment 2, PyYAML (`pyyaml>=6.0`) — used for parsing the human-authored curve contracts in `config/las_curve_contracts.yml`, `config/deviation_survey_contracts.yml` (Increment 3), and (new in Increment 4) `config/checkshot_contracts.yml`. No new runtime dependency was added in Increment 3 or Increment 4: the minimum-curvature engine, depth-mapping interpolation, duplicate-tie conditioning, velocity diagnostics, and time-depth interpolation all use only NumPy (including a local, version-independent trapezoidal-integration helper in `p2mem/time_depth.py`, added because `numpy.trapz`/`numpy.trapezoid` are not consistently available across supported NumPy versions). Matplotlib and pandas are used only for notebook display and QC-figure generation (`run_integration_03.py`/`run_integration_04.py`, the Increment 3/4 notebooks) — never imported by the installable `p2mem` package itself. To also install the test dependency:

```bash
pip install -e ".[dev]"
```

## Running the tests

```bash
pytest -v
```

The suite in `tests/test_units.py` validates `p2mem/units.py` (unchanged since Increment 1.1) against analytical reference values, round-trip consistency, scalar/array inputs, NaN preservation, and rejection of invalid/nonphysical/ambiguous inputs. The suite in `tests/test_las.py` validates `p2mem/io/las.py` (locked since Increment 2.1.1) against small synthetic LAS fixtures. The suites in `tests/test_trajectory.py`, `tests/test_deviation.py`, and `tests/test_depth_mapping.py` (new in Increment 3; extended in Increment 3.1 with dogleg numerical-stability, DLS-normalization-disclosure, and real-filename-with-spaces regression/negative tests) validate the minimum-curvature engine, the Petrel deviation-survey parser/contract resolver, and the MD-to-TVD/TVDSS mapping respectively, against small synthetic fixtures under `tests/fixtures/` and in-memory synthetic data (this layer is locked, unmodified, as of Increment 4). `tests/test_deviation_inventory.py` (new in Increment 3.1) validates that no exported inventory/issues/manifest row, for a successful or a failed well, ever embeds a full environment-dependent build path. The suites in `tests/test_checkshot.py`, `tests/test_time_depth.py`, and `tests/test_checkshot_inventory.py` (new in Increment 4) validate the checkshot parser/contract resolver, the duplicate-tie conditioning/velocity-diagnostic/time-depth-interpolation/sonic-drift numerical layer, and the deterministic inventory/QC-table builders respectively, against small synthetic fixtures under `tests/fixtures/` (including a CRLF fixture used to verify line-ending detection) and in-memory synthetic/analytic data — including an analytic constant-velocity case used to independently verify the trapezoidal-integration helper. None of these suites require the private/raw project LAS, deviation, or checkshot files, so the full suite runs the same way for anyone who clones this repository. Run the command above and read the reported pass/fail count directly — this document does not assert a fixed expected count, since that must always be read from the actual `pytest` output for the code currently on disk. Real integration validation (which DOES require the raw LAS/deviation/checkshot files, not included in this repository) is a separate notebook run — see `02_LAS_Ingestion_and_Curve_Contracts.ipynb`, `03_Deviation_Survey_and_Depth_Framework.ipynb`, and `04_Checkshot_QC_and_Time_Depth_Framework.ipynb`.

## Directory structure

```
Poseidon_1D_MEM/
├── README.md
├── pyproject.toml
├── p2mem/
│   ├── __init__.py
│   ├── models.py
│   ├── units.py
│   ├── deviation_models.py
│   ├── trajectory.py
│   ├── depth_mapping.py
│   ├── checkshot_models.py
│   ├── time_depth.py
│   └── io/
│       ├── __init__.py
│       ├── las.py
│       ├── inventory.py
│       ├── deviation.py
│       ├── deviation_inventory.py
│       ├── checkshot.py
│       └── checkshot_inventory.py
├── tests/
│   ├── test_units.py
│   ├── test_las.py
│   ├── test_trajectory.py
│   ├── test_deviation.py
│   ├── test_depth_mapping.py
│   ├── test_deviation_inventory.py
│   ├── test_checkshot.py
│   ├── test_time_depth.py
│   ├── test_checkshot_inventory.py
│   └── fixtures/         (small synthetic LAS + deviation-survey + checkshot files; no project raw data)
├── config/
│   ├── las_curve_contracts.yml
│   ├── deviation_survey_contracts.yml
│   └── checkshot_contracts.yml
├── data/
│   └── raw/
│       ├── logs/         (the four raw LAS files - NOT included in this repository; immutable inputs)
│       ├── deviation/    (the four raw deviation-survey files, exact filenames contain spaces, e.g. "Poseidon 2_dev.txt" - NOT included in this repository; immutable inputs)
│       └── checkshot/    (the three raw checkshot files, exact filenames e.g. "Poseidon2-Checkshot.txt" - NOT included in this repository; immutable inputs, never rewritten/renamed/"cleaned")
├── notebooks/  (reserved for later increments)
└── outputs/
    ├── 02_las_inventory/            (Increment 2.1.1 real four-well run: CSV/JSON metadata only, no raw log samples)
    ├── 03_deviation_depth/          (Increment 3 real four-well run: CSV/JSON metadata + QC figures, no raw station/log samples)
    └── 04_checkshot_time_depth/     (Increment 4 real three-file checkshot run: CSV/JSON metadata + QC figures, no raw per-sample checkshot/LAS arrays)
```

`notebooks/` is created empty by the project-setup notebook cell and is not yet populated in-repo (the increment notebooks themselves are delivered as top-level files, e.g. `02_LAS_Ingestion_and_Curve_Contracts.ipynb`, `03_Deviation_Survey_and_Depth_Framework.ipynb`, `04_Checkshot_QC_and_Time_Depth_Framework.ipynb`, and are meant to be run from Google Drive per their own directory-setup cells).

## Scientific limitations

These limitations are specific to the Poseidon 2 dataset and this project's current increment, and are carried forward here so they are visible outside the conversation in which they were identified:

- **RHOB (bulk density) coverage in Poseidon 2 ends at approximately 5,296.85 m MD.** Sonic and other curves continue deeper, so Vp/Vs and dynamic Poisson's ratio remain computable below that depth, but density-dependent properties (Young's modulus, shear modulus, bulk modulus, acoustic impedance, shear impedance) are unavailable below it unless density is explicitly estimated and flagged as such — never silently substituted.
- **No reliable shale-based normal compaction trend (NCT) exists from Poseidon 2 alone.** A provisional, transferred candidate NCT identified in offset well Poseidon North 1 is a *candidate*, not a validated trend, and must not be presented as calibrated.
- **Independently measured Vp/Vs quality flags:** approximately 3.38% of Poseidon 2 Vp/Vs values fall below 1.5, and approximately 0.53% fall below the physical validity cutoff of √2 (≈1.4142) required for a non-negative dynamic Poisson's ratio.
- **No independent calibration data (RFT/MDT pressure points, LOT/XLOT tests, or core data) has been supplied or incorporated.** Any pore-pressure or stress output in later increments must be presented as a bounded or theoretical estimate, not a validated field prediction.
- **Empirical/correlation equations are not implemented until their governing equation, units, applicability range, and calibration status are recorded in the project's method-and-citation register.** Several candidate methods remain in "pending" status and are intentionally absent from the codebase for that reason, not because they were overlooked.
- Additional open items (offset-well GR/ECGR scale adjudication, missing formation tops for one offset well) are tracked in the project's design-review documentation and gate specific later phases (lithology and pore-pressure), not this increment.
- **Increment 2 update:** LAS ingestion independently reconfirms (does not newly discover, and does not act on) two previously-flagged anomalies from the Rev 1 design review: Boreas 1's ECGR curve (canonical name `ECGR_api`) ranges from approximately −0.0001 to 519.18 API (vs. roughly 5–205 API for the other three wells' GR-family curves) with 96.98% valid coverage; and Proteus 1ST2's LAS log file places its neutron-porosity curve (canonical name `NPHI_pct`) at column position 5 rather than the last position (8) used by the other three wells. Both are reported as ingestion facts (see `outputs/02_las_inventory/`); neither is rescaled, reinterpreted, or otherwise acted on by this increment.
- **Increment 2.1 / 2.1.1 update:** corrective patches addressing independent audits' naming, reporting, contract-validation, and notebook/source-synchronization findings — see `INCREMENT_02_v2.1_MANIFEST.md` and `INCREMENT_02_v2.1.1_MANIFEST.md`. No new scientific finding was made in either patch; the two anomalies above are unaffected and remain open items for a later, explicitly-scoped increment.
- **Increment 3 update:** deviation-survey ingestion independently reconfirms the Petrel-supplied trajectory for Poseidon 2, Boreas 1, and Poseidon North 1 to approximately millimetre scale via minimum curvature, and additionally DISCOVERS (not merely reconfirms) a real trajectory-reconstruction discrepancy in Proteus 1ST2's deeper section (~0.19 m TVD, ~1.6 m easting at maximum, concentrated below ~MD 4200 m) — see "Current implementation status" above and `INCREMENT_03_MANIFEST.md` Section 6 for the full investigation. This is disclosed as an open item, not corrected or hidden; downstream MD-to-TVD/TVDSS mapping for Proteus 1ST2 conservatively uses the Petrel-supplied source trajectory (not the disagreeing minimum-curvature trajectory) as a result.
- **Increment 3.1 update:** a corrective patch addressing an independent audit's findings on source-filename handling, output environment-independence, an undisclosed normalization inference, and dogleg-angle numerical conditioning — see `INCREMENT_03_1_MANIFEST.md` for the full audit and re-verification record. No new scientific finding was made in this patch; the Proteus 1ST2 discrepancy above is unaffected, remains disclosed exactly as before, and was neither corrected nor concealed. The only numerical changes are at the floating-point noise floor of the diagnostic `dogleg_deg`/`dls_deg_per_30m`/TVD-and-offset-residual fields (at most ~9×10⁻¹³ m for TVD, ~7×10⁻¹⁵ m for easting/northing, across all four real wells), with zero change to any well's PASS/WARNING status.
- **Increment 3.1.1 update:** a packaging-only corrective patch (three notebook `%%writefile` cells that had drifted from their packaged source files, mirroring the earlier Increment 2.1.1 finding). No scientific, numerical, or real-data change of any kind — see `INCREMENT_03_1_1_MANIFEST.md`.
- **Increment 4 update:** checkshot ingestion and the time-depth framework independently reproduce every raw-data anomaly the project design anticipated (Poseidon 2: 5 repeated Depth ties, 6 non-increasing TVDSS steps, 4 non-increasing OWT steps, and a ≈257 m gap in Depth coverage between 1313.1 m and 1570.1 m; Boreas 1: 3 repeated Depth ties and 4 non-increasing TVDSS steps with OWT strictly increasing; Proteus 1ST2: strictly increasing in all three raw columns) and DISCLOSES (not resolves) two further items: (1) Boreas 1's and Proteus 1ST2's checkshot-vs-locked-survey TVDSS comparisons each show a near-constant offset (≈−0.69 m and ≈+0.30 m respectively) — reported as an observed datum-like offset pattern, not a proven datum error, with both source references preserved unmodified; (2) `Proteus1-Checkshot.txt` carries no embedded well identifier, so its association with Proteus 1ST2 is recorded as `identity_status: inferred_unverified` and used for QC only, never as a substitute time-depth model for any other well. Poseidon 2's sonic-checkshot drift over its longest valid continuous sonic interval is a diagnostic finding only (≈+16.7 ms, ≈+4.4%) and does not trigger any correction of `VP_m_s`, `DTCO`, checkshot OWT, or the time-depth curve. Poseidon North 1 has no approved checkshot file and is recorded as a factual data gap (`checkshot_availability: NOT_AVAILABLE`), not substituted with another well's data. See `INCREMENT_04_MANIFEST.md` for the full statistics, tables, and figures. **NOTE:** `INCREMENT_04_MANIFEST.md` incorrectly stated TVDSS is strictly increasing after Depth-tie conditioning for all three wells; this was corrected by Increment 4.1 (see below) — do not rely on that original statement.
- **Increment 4.1 update:** a narrowly scoped corrective patch to Increment 4, applied after an independent numerical-method audit, that does NOT begin Increment 5 or any later-phase work. It corrected an order-dependent tie-break: `tvdss_to_owt`/`owt_to_tvdss` (and their TWT equivalents) previously resolved a repeated value on the axis being inverted by silently keeping whichever tied row was encountered first in the Depth-conditioned table and discarding the other, with no audit trail beyond a bare count. This is replaced by an explicit, order-invariant policy (`p2mem.time_depth.build_axis_conditioned_lookup_table`/`build_axis_conditioned_tables_for_well`): every tied value is grouped by exact equality regardless of parse order, every tied row is registered in a new audit register (`checkshot_time_axis_tie_register.csv` — separate from, and never confused with, the pre-existing Depth-axis `checkshot_duplicate_tie_register.csv`), and the group's dependent-value MEDIAN becomes the conditioned representative (order-invariant; a screening-level choice, not proof the original TVDSS↔OWT relationship was single-valued at that tied value). A genuine reversal (not a tie) raises a typed error rather than being sorted or forced monotonic. Independently reproduced real-data counts: Poseidon 2 has two TVDSS-axis and two OWT-axis tie groups after Depth-tie conditioning; Boreas 1 has one TVDSS-axis tie group and zero OWT-axis tie groups; Proteus 1ST2 has none of either — correcting `INCREMENT_04_MANIFEST.md`'s original, incorrect "strictly increasing for all three wells" statement. This patch also hardens `trapezoidal_integrate` and `compute_sonic_checkshot_drift` to validate their numerical preconditions (finite, one-dimensional, equal-length, strictly increasing MD/x where required) — most notably, a decreasing or duplicate MD run can no longer silently produce a physically invalid negative transit time; it now raises a typed error instead. None of this hardening changes the already-verified real Poseidon 2 sonic-drift result, which is bit-for-bit unchanged. See `INCREMENT_04_1_MANIFEST.md` for the full audit, corrected statistics, and re-verification record.

- **Increment 4.1.1 update:** a narrowly scoped numerical-validation corrective patch to Increment 4.1, applied after an independent numerical-method/software-QA audit, that does NOT begin Increment 5 or any later-phase work, and does NOT alter any previously verified real Poseidon 2/Boreas 1/Proteus 1ST2 result. It corrected four blocking defects and one input-safety gap. (1) `build_axis_conditioned_lookup_table` grouped ALL occurrences of an identical axis value together GLOBALLY before checking for a reversal, so a reversal that returned to an already-seen value (e.g. `[100.0, 200.0, 100.0]`) was silently hidden rather than raising a typed error; it now evaluates the ORIGINAL, ungrouped sequence's successive differences for negativity BEFORE any grouping is attempted — provably equivalent to the 4.1 behavior for every legitimate adjacent tie, and strictly stronger against a non-adjacent reversal. (2) `compute_sonic_checkshot_drift`/`find_longest_finite_positive_run` validated MD monotonicity only within the selected finite-positive-VP run, so a decreasing or duplicate MD value outside that run (e.g. at a NaN-VP station) could pass silently; the COMPLETE canonical `md_m` array is now required finite and strictly increasing before run-selection (the real Poseidon 2 MD array, 31,897 samples, was independently re-verified to already satisfy this — the real sonic-drift result is bit-for-bit unchanged). (3) `compare_checkshot_to_survey` reached an untyped NumPy `ValueError` ("zero-size array to reduction operation") if every checkshot Depth row fell outside the locked survey's own MD coverage; it now raises a typed error naming the well, the checkshot Depth range, and the survey MD coverage. (4) `p2mem.io.checkshot.load_checkshot_surveys` did not catch the typed numerical-conditioning error, so a defect in one well's data could stop the entire batch; it is now caught per well (never via a blanket exception handler) and recorded as a typed, isolated per-well failure, exactly like every other expected failure mode. (5) `seconds_to_milliseconds`/`milliseconds_to_seconds` coerced their input directly, unlike `p2mem.units`'s Increment-1 input-safety policy, so a boolean, numeric-looking string, or complex value would be silently reinterpreted rather than rejected; both now reject such input with a typed error via a local, documented copy of `p2mem.units`'s identical private dtype check (`p2mem/units.py` itself remains LOCKED and unmodified). See `INCREMENT_04_1_1_MANIFEST.md` for the full audit, the regression-test list, and the re-verification record.

## Screening-level statement

**This 1D Mechanical Earth Model is a screening-level, uncalibrated, educational work product.** It has not been validated against independent field measurements and does not carry the assurance level required for drilling engineering, well design, casing/mud-weight selection, or any other operational decision. Any numerical result produced by this codebase should be read as illustrative of a defensible methodology applied to the available data, not as a certified or field-ready prediction.


#### `p2mem/checkshot_models.py` — typed dataclasses for the checkshot/time-depth layer

In [ ]:
%%writefile p2mem/checkshot_models.py
"""
p2mem.checkshot_models - Typed, documented result/schema objects for the
Increment 4 checkshot-ingestion and time-depth layer.

Design rationale
-----------------
Mirrors `p2mem.deviation_models` (Increment 3): every object here is a
frozen `dataclass`, nothing performs I/O or numerical computation, and
every array/field name is explicit about what it holds (raw vs.
conditioned, source vs. survey-derived, seconds vs. milliseconds - see
"Naming discipline" below). This module collects every typed object used
by `p2mem.io.checkshot`, `p2mem.io.checkshot_inventory`, and
`p2mem.time_depth` in one place.

Naming discipline (explicit, unit-suffixed names - project-wide policy)
-------------------------------------------------------------------------
* `Depth_source_m`   - the checkshot file's own first column, EXACTLY as
                        labelled in its header ("Depth"). This name is
                        deliberately NOT `MD_source_m`: the header never
                        states this column is measured depth, so it is
                        never silently renamed to imply that. Whether it
                        behaves as MD is evaluated empirically (see
                        `CheckshotSurveyDepthComparisonResult`) and the
                        evidence is recorded, never assumed.
* `TVDSS_source_m`   - the file's own supplied TVDSS column, unmodified.
* `OWT_source_s`     - the file's own supplied one-way-time column
                        (seconds), unmodified, WITH every raw duplicate
                        row preserved (see `CheckshotStationData`).
* `OWT_conditioned_s`, `TVDSS_conditioned_m`, `Depth_conditioned_m` - the
                        SEPARATE, explicitly named conditioned-tie
                        representation built by `p2mem.time_depth`. Never
                        confused with the `_source_` arrays above; the
                        conditioning rule that produced them is always
                        recorded (see `ConditionedCheckshotData`).
* `TWT_s`, `TWT_ms`  - two-way time, always derived via the LOCKED
                        Increment 1 `p2mem.units.owt_to_twt` (never
                        reimplemented here). `_s` and `_ms` are never
                        mixed in one computation without an explicit,
                        named conversion step.
* `Vavg_m_s`, `Vint_m_s` - average and interval velocity (see
                        `p2mem.time_depth` for their exact definitions
                        and NaN-on-invalid-interval semantics).

Axis-tie conditioning (Increment 4.1 addition)
------------------------------------------------
Increment 4's original `_build_strictly_increasing_table` (removed in this
patch) resolved a repeated TVDSS or OWT value in the Depth-conditioned
table by keeping whichever tied row happened to come first and silently
dropping the other - deterministic, but an order-dependent tie-break: a
file whose two tied rows were supplied in the opposite order would have
produced a different, silently different, inverse mapping. This is
corrected by `p2mem.time_depth.build_axis_conditioned_lookup_table`, which
groups every value tied on the axis being inverted (TVDSS, for
`owt_to_tvdss`'s independent variable... note the OPPOSITE naming
direction below), takes the group's dependent-value MEDIAN as a
representative (order-invariant), and registers every group -
`AxisTimeDepthTieRegisterEntry` - independently of the pre-existing
Depth-tie register (`DuplicateTieRegisterEntry`), which conditions the
raw rows against the Depth axis and is unaffected by this patch. A
genuine reversal (not a tie - a group's axis value less than the
preceding group's) raises `TimeDepthError` rather than being sorted,
discarded, or forced monotonic. See `p2mem.time_depth` module docstring,
section "Order-invariant axis-tie conditioning" for full detail.

Evidence-status vocabulary (Increment 4 addition)
----------------------------------------------------
Extends the project's established measured / derived / correlation-
derived / assumed / unavailable disclosure discipline with three new,
explicit fields used only for checkshot association and model role:

* `well_identity_evidence_status` - "verified" (the file's own content,
  or an independently verified cross-reference, proves which well it
  belongs to) or "inferred_unverified" (association rests on filename,
  project context, and numerical depth-tie plausibility only - see
  Proteus 1ST2 in `config/checkshot_contracts.yml`). Never described as
  "verified" when it is not.
* `model_use_status` - "primary_model" (this well's checkshot may define
  the primary time-depth relationship used elsewhere) or "qc_only"
  (supporting QC/comparison data only - must never be transferred into
  another well's time-depth model).
* `checkshot_availability` - "AVAILABLE" or "NOT_AVAILABLE" (a factual
  data-gap statement for a well with no approved checkshot file at all,
  e.g. Poseidon North 1 - never treated as, or reported alongside, an
  ingestion failure).
"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Optional, Tuple

import numpy as np

__all__ = [
    "STATUS_PASS",
    "STATUS_WARNING",
    "STATUS_FAIL",
    "VALID_IDENTITY_EVIDENCE_STATUSES",
    "VALID_MODEL_USE_STATUSES",
    "VALID_CHECKSHOT_AVAILABILITY_STATUSES",
    "VALID_DEPTH_BASIS_INTERPRETATION_STATUSES",
    "CheckshotHeaderInfo",
    "CheckshotFileContract",
    "CheckshotIngestionIssue",
    "CheckshotStationData",
    "DuplicateTieRegisterEntry",
    "ConditionedCheckshotData",
    "AxisTimeDepthTieRegisterEntry",
    "AxisConditionedLookupTable",
    "VelocityDiagnosticsResult",
    "CheckshotSurveyDepthComparisonResult",
    "SonicCheckshotDriftResult",
    "TimeDepthMappingSummary",
    "CheckshotIngestionFailure",
    "CheckshotWellResult",
    "CheckshotAvailabilityRecord",
]

# ---------------------------------------------------------------------------
# Shared status vocabulary (mirrors p2mem.deviation_models)
# ---------------------------------------------------------------------------
STATUS_PASS = "PASS"
STATUS_WARNING = "WARNING"
STATUS_FAIL = "FAIL"

VALID_IDENTITY_EVIDENCE_STATUSES = ("verified", "inferred_unverified")
VALID_MODEL_USE_STATUSES = ("primary_model", "qc_only")
VALID_CHECKSHOT_AVAILABILITY_STATUSES = ("AVAILABLE", "NOT_AVAILABLE")
VALID_DEPTH_BASIS_INTERPRETATION_STATUSES = (
    "candidate_md_evaluated_against_locked_survey",
)


# ---------------------------------------------------------------------------
# Raw header / provenance
# ---------------------------------------------------------------------------
@dataclass(frozen=True)
class CheckshotHeaderInfo:
    """
    The literal, structurally parsed header of one checkshot file. Every
    approved file in this project shares the identical two-line header
    format (one free-text survey statement, one tab-separated column-
    header line) - this dataclass stores what was actually parsed, not an
    assumed constant, so a genuinely different header is caught by
    contract resolution rather than silently accepted.
    """

    source_filename: str
    sha256: str
    survey_statement: str
    column_header_line: str
    column_names: Tuple[str, ...]
    line_ending_convention: str  # e.g. "CRLF" - disclosed, never "corrected"


# ---------------------------------------------------------------------------
# Per-file contract
# ---------------------------------------------------------------------------
@dataclass(frozen=True)
class CheckshotFileContract:
    """
    One file's complete, human-authored expectation set, resolved against
    the actual file by `p2mem.io.checkshot.resolve_checkshot_contract`.
    Mirrors `p2mem.deviation_models.DeviationFileContract`'s "declare
    everything, verify everything" pattern.
    """

    source_filename: str
    expected_sha256: str
    project_well_key: str
    well_identity_evidence_status: str
    well_identity_evidence_notes: str
    model_use_status: str
    expected_survey_statement: str
    expected_column_header_line: str
    expected_column_order: Tuple[str, ...]
    expected_column_count: int
    expected_time_type: str
    expected_time_unit: str
    expected_vertical_correction_fragment: str
    expected_srd_reference_fragment: str
    expected_row_count: int
    expected_depth_min_m: float
    expected_depth_max_m: float
    expected_tvdss_min_m: float
    expected_tvdss_max_m: float
    expected_owt_min_s: float
    expected_owt_max_s: float
    numeric_range_tolerance: float
    depth_basis_interpretation_status: str
    duplicate_tie_policy: str
    interpolation_policy: str
    extrapolation_policy: str
    notes: str


# ---------------------------------------------------------------------------
# Issues
# ---------------------------------------------------------------------------
@dataclass(frozen=True)
class CheckshotIngestionIssue:
    """One ERROR (blocking) or WARNING (non-blocking, disclosed) fact."""

    severity: str  # "ERROR" or "WARNING"
    code: str
    message: str
    context: str


# ---------------------------------------------------------------------------
# Raw station data (never overwritten; every raw row preserved exactly)
# ---------------------------------------------------------------------------
@dataclass(frozen=True)
class CheckshotStationData:
    """
    The exact, raw parsed checkshot rows, column-ordered as
    (Depth, TVDSS, OWT), with EVERY row preserved exactly as read -
    including any repeated/duplicate Depth, TVDSS, or OWT value. Nothing
    here is deduplicated, reordered, or averaged; see
    `ConditionedCheckshotData` for the separate, explicitly named
    conditioned representation.
    """

    Depth_source_m: np.ndarray
    TVDSS_source_m: np.ndarray
    OWT_source_s: np.ndarray


# ---------------------------------------------------------------------------
# Duplicate / repeated-tie register
# ---------------------------------------------------------------------------
@dataclass(frozen=True)
class DuplicateTieRegisterEntry:
    """
    One auditable record of a repeated (tied) independent-axis (Depth)
    value found in a file's raw data, per the project's "never silently
    average ties" policy. `source_row_indices` are 0-indexed positions
    into that well's `CheckshotStationData` arrays.
    """

    well_key: str
    axis: str  # "Depth" (the only independent-axis tie type detected)
    tie_value_m: float
    source_row_indices: Tuple[int, ...]
    original_tvdss_m: Tuple[float, ...]
    original_owt_s: Tuple[float, ...]
    duplicate_type: str  # e.g. "repeated_depth_distinct_tvdss_owt"
    group_size: int
    tvdss_value_spread_m: float
    owt_value_spread_s: float
    selected_representative_tvdss_m: float
    selected_representative_owt_s: float
    conditioning_rule: str
    affected_downstream_outputs: Tuple[str, ...]


# ---------------------------------------------------------------------------
# Conditioned (tie-collapsed) representation
# ---------------------------------------------------------------------------
@dataclass(frozen=True)
class ConditionedCheckshotData:
    """
    The SEPARATE, explicitly named conditioned-tie representation used
    for all downstream interpolation/velocity work. Built by collapsing
    each repeated-Depth tie group in `CheckshotStationData` to one
    deterministic representative row (see `DuplicateTieRegisterEntry
    .conditioning_rule`) - never by silently averaging, never by
    injecting artificial epsilon separations to force strict
    monotonicity. `Depth_conditioned_m` is therefore strictly increasing
    by construction (every distinct raw Depth value contributes exactly
    one conditioned row).
    """

    Depth_conditioned_m: np.ndarray
    TVDSS_conditioned_m: np.ndarray
    OWT_conditioned_s: np.ndarray
    n_raw_rows: int
    n_conditioned_rows: int
    n_tie_groups: int
    conditioning_method: str


# ---------------------------------------------------------------------------
# Axis-tie register and axis-conditioned lookup table (Increment 4.1)
# ---------------------------------------------------------------------------
@dataclass(frozen=True)
class AxisTimeDepthTieRegisterEntry:
    """
    One auditable record of a repeated (tied) value on the AXIS being
    inverted (TVDSS, for `tvdss_to_owt`; OWT, for `owt_to_tvdss`) found in
    the Depth-conditioned table - separate from, and never confused with,
    `DuplicateTieRegisterEntry` (which registers repeated Depth values in
    the RAW table). `conditioned_row_indices` are 0-indexed positions into
    that well's `ConditionedCheckshotData` arrays (Depth_conditioned_m /
    TVDSS_conditioned_m / OWT_conditioned_s) - NOT into the raw arrays.

    `tie_kind` is `"identical_pair"` when every tied row shares an
    IDENTICAL dependent value too (spread == 0.0 exactly - the group
    collapses without changing any value, but is still counted and
    registered per the project's "never silently average, never silently
    drop" policy), or `"genuinely_non_unique"` when the dependent values
    differ (the axis alone cannot distinguish these rows - the median
    representative is a screening-level choice, not proof the original
    relationship was single-valued at that axis value).
    """

    well_key: str
    interpolation_direction: str  # "tvdss_to_owt" | "owt_to_tvdss"
    axis: str  # "TVDSS_conditioned_m" | "OWT_conditioned_s" - the independent axis being grouped
    dependent_axis: str  # the other axis - the one whose value is conditioned per group
    tie_axis_value: float
    conditioned_row_indices: Tuple[int, ...]
    associated_depth_m: Tuple[float, ...]
    original_dependent_values: Tuple[float, ...]
    group_size: int
    dependent_value_spread: float
    selected_representative_dependent_value: float
    conditioning_rule: str
    tie_kind: str  # "identical_pair" | "genuinely_non_unique"
    affected_downstream_outputs: Tuple[str, ...]


@dataclass(frozen=True)
class AxisConditionedLookupTable:
    """
    An order-invariant, axis-tie-conditioned lookup table used ONLY for
    inverse/forward interpolation FROM the named independent axis (TVDSS
    or OWT) TO the named dependent axis. This is explicitly NOT the raw
    checkshot data and NOT the Depth-conditioned table
    (`ConditionedCheckshotData`) - it is a further-conditioned view of the
    latter, built by grouping ties on `independent_axis_name` and taking
    each group's dependent-value median (see `AxisTimeDepthTieRegisterEntry`
    for the per-group audit trail). `axis_values` is guaranteed strictly
    increasing (a genuine reversal raises `TimeDepthError` at construction
    - see `p2mem.time_depth.build_axis_conditioned_lookup_table`).

    Never described as raw, uniquely measured, or unconditioned - the
    median representative used here is a screening-level choice, not
    evidence that the original TVDSS<->OWT relationship was single-valued
    at every tied axis value.
    """

    well_key: str
    interpolation_direction: str  # "tvdss_to_owt" | "owt_to_tvdss"
    independent_axis_name: str
    dependent_axis_name: str
    axis_values: np.ndarray
    dependent_values: np.ndarray
    n_input_points: int
    n_output_points: int
    n_axis_tie_groups: int
    n_collapsed_points: int
    n_identical_pairs: int
    n_genuinely_nonunique_groups: int
    conditioning_method: str


# ---------------------------------------------------------------------------
# Velocity diagnostics
# ---------------------------------------------------------------------------
@dataclass(frozen=True)
class VelocityDiagnosticsResult:
    """
    Average velocity (Vavg = TVDSS / OWT) at every conditioned row, and
    interval velocity (Vint = dTVDSS / dOWT) between consecutive
    conditioned rows. `Vint_m_s` has length `n_conditioned_rows - 1`;
    index k is the interval between conditioned row k and k+1.
    Non-positive dTVDSS or dOWT yields NaN (never inf or a negative
    velocity) at that index, flagged in `vint_invalid_mask`.
    """

    well_key: str
    Depth_conditioned_m: np.ndarray
    Vavg_m_s: np.ndarray
    Vint_m_s: np.ndarray
    vint_invalid_mask: np.ndarray
    n_vint_intervals: int
    n_vint_invalid: int
    vint_invalid_reason_counts: dict


# ---------------------------------------------------------------------------
# Checkshot-vs-locked-survey depth-reference comparison
# ---------------------------------------------------------------------------
@dataclass(frozen=True)
class CheckshotSurveyDepthComparisonResult:
    """
    Compares checkshot `Depth_source_m` (treated as a CANDIDATE measured-
    depth axis - see the module docstring's naming-discipline note)
    against TVDSS interpolated from the LOCKED Increment 3/3.1.1
    `petrel_source_trace` survey MD->TVD relationship for the same well
    (TVDSS_survey_m = TVD_survey_m - datum_elevation_m).

    Sign convention (explicit, never implicit): `residual_m` at each
    compared row is defined as

        residual_m = TVDSS_survey_interpolated_m - TVDSS_source_m

    i.e. positive means the locked survey trajectory places that depth
    DEEPER (more positive TVDSS) than the checkshot file's own supplied
    TVDSS at the same Depth value. This is the convention used
    consistently for all three admitted files' recomputed residuals in
    `INCREMENT_04_MANIFEST.md`.
    """

    well_key: str
    n_compared: int
    n_outside_survey_md_coverage: int
    min_residual_m: float
    max_residual_m: float
    max_abs_residual_m: float
    mean_residual_m: float
    median_residual_m: float
    rmse_m: float
    first_residual_m: float
    last_residual_m: float
    residual_trend_description: str
    depth_basis_interpretation_status: str
    residual_sign_convention: str


# ---------------------------------------------------------------------------
# Poseidon 2 sonic-checkshot drift diagnostic
# ---------------------------------------------------------------------------
@dataclass(frozen=True)
class SonicCheckshotDriftResult:
    """
    Poseidon-2-only diagnostic comparing integrated sonic one-way transit
    time (trapezoidal integration of slowness = 1/VP_m_s against MD, over
    an algorithmically identified continuous-coverage sonic interval)
    against the checkshot-interpolated OWT increment over the identical
    MD/Depth endpoints. Diagnostic only - no correction is applied to
    VP_m_s, DTCO, checkshot OWT, or the time-depth curve as a result of
    this comparison (see `limitations`).
    """

    well_key: str
    md_interval_start_m: float
    md_interval_end_m: float
    n_sonic_samples: int
    selection_criteria: str
    integration_method: str
    sonic_transit_time_s: float
    checkshot_owt_increment_s: float
    sonic_minus_checkshot_ms: float
    checkshot_minus_sonic_ms: float
    sonic_minus_checkshot_percent: float
    limitations: Tuple[str, ...]


# ---------------------------------------------------------------------------
# LAS MD -> checkshot time mapping summary (Poseidon 2 only; no raw arrays)
# ---------------------------------------------------------------------------
@dataclass(frozen=True)
class TimeDepthMappingSummary:
    """
    Deterministic SUMMARY (never the full per-sample array - see the
    module docstring's "do not package full proprietary...arrays" policy)
    of mapping a well's canonical LAS MD onto checkshot-derived OWT/TWT,
    within validated checkshot coverage only. Samples outside checkshot
    Depth coverage are never extrapolated; `n_extrapolated` must be 0 by
    construction (see `p2mem.time_depth.map_las_md_to_checkshot_time`).
    """

    well_key: str
    n_las_samples: int
    n_inside_coverage: int
    n_shallower_than_coverage: int
    n_deeper_than_coverage: int
    mapped_fraction: float
    checkshot_depth_min_m: float
    checkshot_depth_max_m: float
    las_md_min_m: float
    las_md_max_m: float
    interpolation_method: str
    n_extrapolated: int


# ---------------------------------------------------------------------------
# Ingestion failure (typed, mirrors DeviationIngestionFailure)
# ---------------------------------------------------------------------------
@dataclass(frozen=True)
class CheckshotIngestionFailure:
    well_key: str
    source_path: str
    error_type: str
    message: str
    exception: BaseException


# ---------------------------------------------------------------------------
# Combined per-well result
# ---------------------------------------------------------------------------
@dataclass(frozen=True)
class CheckshotWellResult:
    """The complete, typed result of successfully loading and contract-
    resolving, conditioning, and diagnosing one checkshot file.

    `axis_tie_entries` (Increment 4.1) is the FLAT tuple of every
    `AxisTimeDepthTieRegisterEntry` found across BOTH inversion directions
    (`tvdss_to_owt` and `owt_to_tvdss`) - separate from, and never
    confused with, `duplicate_ties` (the pre-existing Depth-axis register).
    `axis_tables` maps `"tvdss_to_owt"` / `"owt_to_tvdss"` to the
    corresponding `AxisConditionedLookupTable`, built once per well at
    load time (see `p2mem.time_depth.build_axis_conditioned_lookup_table`)
    so every caller (interpolation, the CSV/JSON exporters, the notebook)
    shares one order-invariant table rather than each silently rebuilding
    its own."""

    header: CheckshotHeaderInfo
    contract: CheckshotFileContract
    raw: CheckshotStationData
    duplicate_ties: Tuple[DuplicateTieRegisterEntry, ...]
    conditioned: ConditionedCheckshotData
    velocity: VelocityDiagnosticsResult
    depth_comparison: Optional[CheckshotSurveyDepthComparisonResult]
    issues: Tuple[CheckshotIngestionIssue, ...] = field(default_factory=tuple)
    contract_status: str = "PASSED"
    axis_tie_entries: Tuple[AxisTimeDepthTieRegisterEntry, ...] = field(default_factory=tuple)
    axis_tables: dict = field(default_factory=dict)


# ---------------------------------------------------------------------------
# Data-availability record (e.g. Poseidon North 1 - a factual gap)
# ---------------------------------------------------------------------------
@dataclass(frozen=True)
class CheckshotAvailabilityRecord:
    """
    Records that a project well has NO approved checkshot file, as a
    factual data gap - never as, or alongside, an ingestion failure, and
    never filled by substituting another well's file.
    """

    well_key: str
    checkshot_availability: str  # "NOT_AVAILABLE"
    notes: str


#### `p2mem/time_depth.py` — duplicate-tie conditioning, velocity diagnostics, forward/inverse time-depth interpolation, checkshot-vs-survey comparison, sonic-checkshot drift

In [ ]:
%%writefile p2mem/time_depth.py
"""
p2mem.time_depth - Numerical checkshot conditioning, velocity diagnostics,
forward/inverse time-depth interpolation, checkshot-vs-survey depth
comparison, and the Poseidon-2-only sonic-checkshot drift diagnostic
(Increment 4).

Scope
-----
This module is pure numerical computation on already-parsed checkshot
arrays (see `p2mem.io.checkshot` for file I/O and contract resolution) and
already-loaded LAS/deviation-survey arrays (see `p2mem.io.las` /
`p2mem.io.deviation`, both LOCKED and unmodified). It does not read files
and does not decide which files are approved for which well - it takes
typed/plain NumPy inputs and returns typed, documented results.

Why this module exists separately from `p2mem.depth_mapping`
-----------------------------------------------------------------
The locked Increment 3 `p2mem.depth_mapping.map_las_md_to_tvd_tvdss` is an
ALL-OR-NOTHING design: `ExtrapolationRejectedError` is raised if even one
LAS sample falls outside survey MD coverage. That is the correct policy
for MD->TVD/TVDSS mapping against a full-well deviation survey (extending
a trajectory beyond its last surveyed station is not defensible).
Checkshot coverage is different in kind: Poseidon 2's checkshot Depth
range (1267.8-4700.0 m) is a genuine PARTIAL subset of its LAS MD
interval, by the nature of when/how checkshot surveys are acquired - this
is an expected, disclosed coverage gap, not an error condition. Increment
4 therefore requires LAS samples outside checkshot coverage to be
reported as out-of-coverage (via an explicit mask and count) and mapped
to NaN, never rejected outright and never extrapolated. `map_las_md_to_
checkshot_time` below is consequently a NEW, separate function with a
coverage-masking contract, deliberately different from `depth_mapping
.py`'s reject-on-any-extrapolation contract - not a modification of the
locked module.

Interpolation method
----------------------
Every interpolation in this module is transparent PIECEWISE-LINEAR
(`numpy.interp`), never a higher-order spline, per the project's stated
preference for transparency over curve smoothness. No interpolation here
ever extrapolates: every function reports (and NaNs) samples outside the
validated coverage of the axis being interpolated against, explicitly.

Duplicate-tie conditioning policy
------------------------------------
Real checkshot files in this project contain REPEATED Depth values
(distinct rows sharing an identical Depth but slightly different TVDSS/
OWT - see `detect_and_condition_depth_ties`). These are acquisition
artifacts (e.g. a repeated tie-in shot), not measurement noise to be
smoothed. Every raw row is preserved exactly in `CheckshotStationData`;
this module never overwrites the raw arrays. A SEPARATE "conditioned"
representation is built by collapsing each repeated-Depth group to one
deterministic representative row before any interpolation is attempted -
`numpy.interp` requires a monotonic (here, strictly increasing) x-axis,
which raw duplicate-Depth data does not provide.

The representative for a tied group's TVDSS (and, independently, its OWT)
is that group's MEDIAN value. Median is used, rather than an arbitrary
"first" or "last" row, because it is order-independent (does not silently
depend on which duplicate happened to be typed/exported first) and
generalizes correctly to a group of any size. It is disclosed here that
for a group of exactly 2 rows (every duplicate group observed in this
project's three approved checkshot files has exactly 2 rows), the median
of 2 values equals their arithmetic mean - this is a documented,
deterministic consequence of the chosen rule, not an undisclosed
"averaging" of ties: the raw rows remain fully preserved and individually
registered (see `DuplicateTieRegisterEntry`), and TVDSS/OWT are each
conditioned independently (so the conditioned row's TVDSS and OWT are not
guaranteed to have both come from the same original raw row - disclosed
here explicitly). No artificial epsilon separation is ever injected to
force strict monotonicity beyond what the median rule itself produces;
where the median rule still leaves a non-strictly-increasing step in a
TVDSS or OWT axis used for TIME-domain (inverse) interpolation, that is
handled and disclosed separately - see "Locally non-unique inversion"
below.

Order-invariant axis-tie conditioning (Increment 4.1)
---------------------------------------------------------
Depth-domain forward interpolation (`depth_to_tvdss`, `depth_to_owt`,
`depth_to_twt`, `map_las_md_to_checkshot_time`) is always well-defined
after Depth-tie conditioning, because `Depth_conditioned_m` is strictly
increasing by construction (one representative row per distinct Depth
value). TVDSS and OWT are each conditioned independently of Depth,
however, and are NOT guaranteed strictly increasing after Depth-tie
conditioning alone: real, distinct Depth values in this project's actual
files carry an equal TVDSS or OWT at adjacent depths (e.g. Poseidon 2 has
two such TVDSS ties and two such OWT ties after Depth-tie conditioning;
Boreas 1 has one TVDSS tie and zero OWT ties; Proteus 1ST2 has none of
either - see `INCREMENT_04_1_MANIFEST.md` for the exact, independently
reproduced counts and the corrected statement of this fact. An earlier
statement in `INCREMENT_04_MANIFEST.md` that TVDSS was strictly
increasing after conditioning for all three wells was INCORRECT and has
been corrected there and here.)

TIME-domain interpolation (`tvdss_to_owt`, `owt_to_tvdss`, and their TWT
equivalents) requires the axis being interpolated FROM (TVDSS or OWT) to
itself be strictly increasing - `numpy.interp` does not detect a
non-monotonic axis and will silently return nonsense for one. Increment
4's original resolution (`_build_strictly_increasing_table`, REMOVED in
this patch) kept whichever tied row happened to appear first in the
Depth-conditioned table and silently discarded the other: deterministic,
but an ORDER-DEPENDENT tie-break - a file whose two tied rows were
supplied in the opposite relative order would silently resolve to a
different inverse value, and the discarded row's information vanished
with no audit trail beyond a bare count.

`build_axis_conditioned_lookup_table` replaces this with an explicit,
ORDER-INVARIANT policy: every value tied on the axis being inverted is
grouped (regardless of which tied row was parsed first), the group's
COMPLETE original rows are preserved in a typed audit register
(`AxisTimeDepthTieRegisterEntry` - separate from, and never confused
with, the raw checkshot data or the Depth-conditioned table), and the
group's dependent-value MEDIAN becomes the conditioned representative
(order-invariant, and disclosed as reducing to the arithmetic mean for a
tied group of size 2 - the only size observed in this project's real
data). A group whose tied rows also share an IDENTICAL dependent value
(`tie_kind="identical_pair"`) collapses without changing the value, but
is still counted and registered, never silently merged away unrecorded.
A GENUINE reversal in the axis being inverted (a group's value less than
the preceding group's - not a tie) raises `TimeDepthError`: this function
never sorts, greedily discards, or forces monotonicity to make a
reversal disappear. The resulting `AxisConditionedLookupTable` is never
described as raw, uniquely measured, or unconditioned - the median
representative is a screening-level choice, not proof that the original
TVDSS<->OWT relationship was single-valued at every tied axis value.

Numerical-validation corrective patch (Increment 4.1.1)
-------------------------------------------------------
An independent numerical-validation audit of Increment 4.1 found four
blocking defects in the reversal/validation logic described above and in
`compute_sonic_checkshot_drift`/`compare_checkshot_to_survey`, plus an
input-safety gap in `seconds_to_milliseconds`/`milliseconds_to_seconds`.
None of these defects altered any previously verified REAL Poseidon
2/Boreas 1/Proteus 1ST2 result - all such results are reproduced exactly
in this patch - they were latent gaps that a specific, previously
untested input shape could have triggered.

1. Hidden reversal via global exact-value grouping (`build_axis_
   conditioned_lookup_table`). The Increment 4.1 implementation grouped
   ALL occurrences of an identical axis value together, globally, before
   checking for a reversal - so `[100.0, 200.0, 100.0]` was incorrectly
   accepted (the trailing `100.0` silently merged into the first group,
   producing an apparently-valid `[100.0, 200.0]` and hiding the genuine
   `200 -> 100` reversal). Fixed by evaluating `np.diff` of the ORIGINAL,
   ungrouped axis sequence for any negative step BEFORE grouping is
   attempted; only once the original sequence is confirmed non-decreasing
   are exact ADJACENT ties grouped and median-conditioned, exactly as in
   4.1. This is provably equivalent to the 4.1 behavior for every
   legitimate adjacent tie (once no negative diff exists anywhere in the
   original sequence, any two occurrences of the same value are
   necessarily adjacent - a non-adjacent repeat would require descending
   back down to that value after a strictly larger one, which is exactly
   the negative diff already excluded) and strictly stronger against a
   reversal that returns to an already-seen value. `depth_conditioned_m`
   is also now validated finite and strictly increasing on entry.
2. Incomplete MD validation in the sonic-checkshot integration path
   (`compute_sonic_checkshot_drift`, `find_longest_finite_positive_run`).
   Increment 4.1 validated strict MD monotonicity only WITHIN the
   selected finite-positive-VP run, so a decreasing or duplicate MD value
   at a station outside that run (e.g. one with `VP_m_s=NaN`) could pass
   silently. Fixed by requiring the COMPLETE canonical `md_m` array to be
   one-dimensional, finite, strictly increasing, and free of duplicates
   BEFORE run-selection; `find_longest_finite_positive_run` also now
   validates compatible one-dimensional shapes when called directly. The
   real Poseidon 2 canonical MD array (31,897 samples) was independently
   re-verified fully finite and strictly increasing across its ENTIRE
   length, so this stricter check does not alter the real sonic-drift
   result.
3. Untyped crash on zero survey-coverage overlap (`compare_checkshot_to_
   survey`). If every checkshot Depth row fell outside the locked
   survey's own MD coverage, NumPy's `min`/`max`/`mean` reductions on the
   resulting empty residual array raised an untyped
   `ValueError: zero-size array to reduction operation ...` rather than a
   documented, typed failure. Fixed by explicitly checking for zero
   in-coverage rows and raising `TimeDepthError` naming the well key, the
   checkshot Depth range, and the survey MD coverage, and stating plainly
   that no comparison was performed and no extrapolation was attempted.
4. Missing batch isolation for numerical failures (`p2mem.io.checkshot.
   load_checkshot_surveys`). Only file/parsing/contract errors were
   caught per well; a `TimeDepthError` raised during Depth-tie
   conditioning, axis-tie conditioning, or survey comparison for one well
   would propagate out of the batch loader and stop every other well from
   loading, contradicting this project's documented batch-isolation
   guarantee. Fixed by catching `TimeDepthError` per well (never via a
   blanket `except Exception`) and recording a typed
   `CheckshotIngestionFailure(error_type="numerical_conditioning_
   failure")`, exactly like every other expected per-well failure mode.
5. Unit-helper input-safety gap (`seconds_to_milliseconds`,
   `milliseconds_to_seconds`). These Increment-4 helpers coerced their
   input with `np.asarray(..., dtype=np.float64)` directly, unlike
   `p2mem.units`'s Increment-1 policy, so a boolean, a numeric-looking
   string (or string array), or a complex value would be silently
   reinterpreted rather than rejected. Fixed with a local
   `_reject_ambiguous_dtype` check (a deliberate, documented LOCAL copy of
   `p2mem.units`'s identical private check - `p2mem/units.py` is LOCKED
   and this project's convention is never to import a private/
   underscore-prefixed name across a module boundary) applied to the raw
   input's dtype before any numeric coercion.

See `INCREMENT_04_1_1_MANIFEST.md` for the full defect writeup, the
required regression-test list, and the distinction this patch draws
between a valid adjacent tie, a globally repeated (non-adjacent) value, a
genuine negative successive-axis step, median conditioning, and numerical
rejection.
"""

from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, Tuple

import numpy as np

from p2mem.checkshot_models import (
    AxisConditionedLookupTable,
    AxisTimeDepthTieRegisterEntry,
    ConditionedCheckshotData,
    CheckshotSurveyDepthComparisonResult,
    DuplicateTieRegisterEntry,
    SonicCheckshotDriftResult,
    TimeDepthMappingSummary,
    VelocityDiagnosticsResult,
)
from p2mem.units import owt_to_twt, twt_to_owt

__all__ = [
    "TimeDepthError",
    "CONDITIONING_METHOD",
    "AXIS_TIE_CONDITIONING_METHOD",
    "INTERPOLATION_METHOD",
    "INTEGRATION_METHOD",
    "seconds_to_milliseconds",
    "milliseconds_to_seconds",
    "trapezoidal_integrate",
    "detect_and_condition_depth_ties",
    "compute_velocity_diagnostics",
    "compare_checkshot_to_survey",
    "depth_to_tvdss",
    "depth_to_owt",
    "depth_to_twt",
    "build_axis_conditioned_lookup_table",
    "build_axis_conditioned_tables_for_well",
    "tvdss_to_owt",
    "tvdss_to_twt",
    "owt_to_tvdss",
    "twt_to_tvdss",
    "map_las_md_to_checkshot_time",
    "find_longest_finite_positive_run",
    "compute_sonic_checkshot_drift",
]

CONDITIONING_METHOD = "duplicate_depth_median_representative_v1"
AXIS_TIE_CONDITIONING_METHOD = "axis_tie_median_representative_order_invariant_v1"
INTERPOLATION_METHOD = "piecewise_linear_no_extrapolation"
INTEGRATION_METHOD = "trapezoidal_rule_on_slowness_vs_md"


class TimeDepthError(ValueError):
    """
    Raised for a structural/numerical precondition failure in this module:
    fewer than 2 conditioned rows, a non-finite input array, a shape/
    length mismatch, a non-strictly-increasing axis where one is required
    (including a genuine reversal in an axis-tie-conditioned lookup table
    - see `build_axis_conditioned_lookup_table` - which this module never
    resolves by sorting, discarding, or forcing monotonicity), a sonic
    interval too short (or insufficiently well-behaved - decreasing/
    duplicate/non-finite MD) to integrate, a non-positive checkshot OWT
    increment, or a drift-comparison endpoint falling outside checkshot
    Depth coverage (this module never extrapolates to satisfy a request).
    """


# ---------------------------------------------------------------------------
# Shared numerical-validation helpers (Increment 4.1 hardening)
# ---------------------------------------------------------------------------
def _as_1d_finite(label: str, arr) -> np.ndarray:
    """Coerce `arr` to a NumPy float64 array and require it be
    one-dimensional and fully finite (no NaN/inf); raise `TimeDepthError`
    (naming `label`) otherwise."""
    a = np.asarray(arr, dtype=np.float64)
    if a.ndim != 1:
        raise TimeDepthError(f"{label} must be a one-dimensional array; got shape {a.shape}.")
    if a.size == 0:
        raise TimeDepthError(f"{label} must not be empty.")
    if not np.all(np.isfinite(a)):
        raise TimeDepthError(f"{label} contains a non-finite value (NaN or inf).")
    return a


def _require_equal_length(pairs) -> None:
    """`pairs` is an iterable of (label, array). Raise `TimeDepthError`
    naming every label if the arrays do not all share one length."""
    lengths = {label: int(np.shape(a)[0]) if np.ndim(a) else 0 for label, a in pairs}
    if len(set(lengths.values())) > 1:
        raise TimeDepthError(f"arrays must share one length; got {lengths}.")


def _reject_ambiguous_dtype(raw: np.ndarray, context: str) -> None:
    """
    Reject input kinds that are not genuine numeric scalars/arrays,
    mirroring the input-safety policy `p2mem.units` established in
    Increment 1 (Increment 4.1.1 audit finding: `seconds_to_milliseconds`/
    `milliseconds_to_seconds`, added in Increment 4 as thin local helpers
    - see the module docstring - had never been checked against that
    policy and would otherwise silently coerce a boolean to 1.0/0.0, a
    numeric-looking string to a float, or a complex value with its
    imaginary part discarded).

    This is a LOCAL copy of the identical dtype-kind check performed by
    `p2mem.units._reject_ambiguous_dtype` (that module's own version is
    private and is not imported here, since `p2mem/units.py` is LOCKED
    and this project's convention is never to import a private/
    underscore-prefixed name across module boundaries) - not a
    reimplementation of different behavior. Only integer- or floating-
    dtype input is accepted; boolean, string/bytes, complex, and object
    dtypes all raise `TypeError`.
    """
    kind = raw.dtype.kind
    if kind == "b":
        raise TypeError(
            f"{context}: boolean input is not accepted (True/False would be "
            f"silently reinterpreted as 1.0/0.0). Pass a numeric value instead."
        )
    if kind in ("U", "S"):
        raise TypeError(
            f"{context}: string input is not accepted, even a numeric-looking "
            f"string (e.g. \"3.5\"). Pass a numeric value instead."
        )
    if kind == "c":
        raise TypeError(
            f"{context}: complex input is not accepted (this module handles "
            f"only real-valued physical quantities). Pass a real numeric value instead."
        )
    if kind not in ("i", "u", "f"):
        raise TypeError(
            f"{context}: unsupported input type (dtype={raw.dtype!r}). "
            f"Only Python numeric scalars and NumPy integer/floating arrays are accepted."
        )


def _require_strictly_increasing(label: str, arr: np.ndarray) -> None:
    """Raise `TimeDepthError` (naming `label` and the offending interval
    index/indices) if `arr` (size >= 2) is not strictly increasing.
    Never sorts, discards, or otherwise repairs `arr` - a genuine
    reversal or a tie in a context requiring strict monotonicity is
    always reported, never silently resolved."""
    if arr.size < 2:
        return
    diffs = np.diff(arr)
    if not np.all(diffs > 0.0):
        bad = np.where(diffs <= 0.0)[0].tolist()
        raise TimeDepthError(
            f"{label} is not strictly increasing at interval index(es) {bad}; this function "
            f"never sorts, discards, or forces monotonicity to resolve a tie or reversal here."
        )


# ---------------------------------------------------------------------------
# Trivial, locally-scoped unit helpers (seconds <-> milliseconds).
# Not added to the LOCKED p2mem.units module - see project instructions:
# only p2mem/__init__.py, pyproject.toml, README.md may be touched among
# existing files for Increment 4.
# ---------------------------------------------------------------------------
def seconds_to_milliseconds(seconds: np.ndarray) -> np.ndarray:
    """
    TWT_ms = TWT_s * 1000 (exact scaling; NaN preserved).

    Validated (Increment 4.1.1 audit finding - unit-helper input
    consistency): this helper was added in Increment 4 as a thin local
    scaling function and, unlike `p2mem.units`, had never been checked
    against the input-safety policy established in Increment 1. Before
    this fix, `np.asarray(seconds, dtype=np.float64)` would silently
    coerce a boolean (`True`/`False` -> `1.0`/`0.0`), a numeric-looking
    string or string array (e.g. `"3.5"`), or a complex value (silently
    discarding its imaginary part) into a float, rather than raising.
    `_reject_ambiguous_dtype` (a local copy of `p2mem.units`'s identical,
    private, un-imported check - see that function's docstring) is now
    called on the raw input's dtype before any numeric coercion, so
    boolean/string/complex input raises `TypeError`. Python numeric
    scalars, NumPy integer/floating arrays, and NaN values are accepted
    exactly as before.
    """
    raw = np.asarray(seconds)
    _reject_ambiguous_dtype(raw, "seconds_to_milliseconds: seconds")
    return raw.astype(np.float64) * 1000.0


def milliseconds_to_seconds(milliseconds: np.ndarray) -> np.ndarray:
    """
    TWT_s = TWT_ms / 1000 (exact scaling; NaN preserved).

    Validated (Increment 4.1.1 audit finding - unit-helper input
    consistency): see `seconds_to_milliseconds`'s docstring above for the
    full rationale. The same `_reject_ambiguous_dtype` guard is applied
    here before numeric coercion, so boolean/string/complex input raises
    `TypeError` while valid Python numeric scalars, NumPy integer/
    floating arrays, and NaN values are accepted exactly as before.
    """
    raw = np.asarray(milliseconds)
    _reject_ambiguous_dtype(raw, "milliseconds_to_seconds: milliseconds")
    return raw.astype(np.float64) / 1000.0


def trapezoidal_integrate(y: np.ndarray, x: np.ndarray) -> float:
    """
    Manual trapezoidal-rule integration of `y` against `x`
    (sum of (y[i]+y[i+1])/2 * (x[i+1]-x[i]) over all i), implemented
    locally rather than via `numpy.trapz`/`numpy.trapezoid` so this
    module's numerical behavior does not depend on which of those two
    (mutually exclusive across NumPy versions - `trapz` was removed in
    NumPy 2.0 in favor of `trapezoid`) happens to be available in a given
    execution environment (including Google Colab).

    Validated (Increment 4.1 hardening - the pre-4.1 implementation
    performed none of this and could silently integrate garbage,
    including returning a physically invalid NEGATIVE "transit time" for
    a decreasing `x`): `x` and `y` must each be one-dimensional, finite,
    and of equal length; at least 2 samples are required; and `x` must be
    STRICTLY increasing (a decreasing or duplicate-value `x` raises
    `TimeDepthError` rather than being silently integrated, sorted, or
    otherwise repaired). This validation does not change the arithmetic
    performed on already-valid input - the real Poseidon 2 sonic-drift
    result in `INCREMENT_04_1_MANIFEST.md` is bit-for-bit unchanged from
    Increment 4's.
    """
    x = _as_1d_finite("trapezoidal_integrate: x", x)
    y = _as_1d_finite("trapezoidal_integrate: y", y)
    _require_equal_length([("x", x), ("y", y)])
    if x.size < 2:
        raise TimeDepthError("trapezoidal_integrate: at least 2 samples are required to integrate.")
    _require_strictly_increasing("trapezoidal_integrate: x", x)
    dx = np.diff(x)
    avg = (y[:-1] + y[1:]) / 2.0
    return float(np.sum(avg * dx))


# ---------------------------------------------------------------------------
# Duplicate-tie detection and conditioning
# ---------------------------------------------------------------------------
def detect_and_condition_depth_ties(
    well_key: str,
    depth_source_m: np.ndarray,
    tvdss_source_m: np.ndarray,
    owt_source_s: np.ndarray,
) -> Tuple[Tuple[DuplicateTieRegisterEntry, ...], ConditionedCheckshotData]:
    """
    Group raw rows by exact `Depth_source_m` value (in order of first
    appearance), register every group of size > 1 as a
    `DuplicateTieRegisterEntry`, and build the conditioned representation
    (one row per distinct Depth, using the tied group's median TVDSS/OWT
    as its representative - see module docstring). Singleton groups pass
    through unchanged; `Depth_conditioned_m` is returned strictly
    increasing by construction whenever `depth_source_m` visits each
    distinct value in non-decreasing order overall (verified: raises
    `TimeDepthError` if a distinct Depth value re-appears out of the
    surrounding sort order, which would indicate a genuinely unsorted or
    corrupted source file, not an ordinary repeated tie-in).
    """
    depth_source_m = np.asarray(depth_source_m, dtype=np.float64)
    tvdss_source_m = np.asarray(tvdss_source_m, dtype=np.float64)
    owt_source_s = np.asarray(owt_source_s, dtype=np.float64)
    n = depth_source_m.size
    if not (tvdss_source_m.size == n and owt_source_s.size == n):
        raise TimeDepthError(
            f"{well_key}: Depth/TVDSS/OWT source arrays must share one length; got "
            f"{n}, {tvdss_source_m.size}, {owt_source_s.size}."
        )
    if n == 0:
        raise TimeDepthError(f"{well_key}: checkshot source arrays are empty.")
    for _name, _arr in (
        ("Depth_source_m", depth_source_m),
        ("TVDSS_source_m", tvdss_source_m),
        ("OWT_source_s", owt_source_s),
    ):
        if not np.all(np.isfinite(_arr)):
            raise TimeDepthError(
                f"{well_key}: {_name} contains a non-finite value (NaN or inf); this function "
                f"never conditions or interpolates against non-finite source data."
            )

    groups: Dict[float, list] = {}
    order: list = []
    for i, d in enumerate(depth_source_m):
        if d not in groups:
            groups[d] = []
            order.append(d)
        groups[d].append(i)

    ties: list = []
    cond_depth: list = []
    cond_tvdss: list = []
    cond_owt: list = []

    for d in order:
        idxs = tuple(groups[d])
        tvdss_group = tvdss_source_m[list(idxs)]
        owt_group = owt_source_s[list(idxs)]
        rep_tvdss = float(np.median(tvdss_group))
        rep_owt = float(np.median(owt_group))
        cond_depth.append(d)
        cond_tvdss.append(rep_tvdss)
        cond_owt.append(rep_owt)

        if len(idxs) > 1:
            spread_tvdss = float(np.max(tvdss_group) - np.min(tvdss_group))
            spread_owt = float(np.max(owt_group) - np.min(owt_group))
            ties.append(
                DuplicateTieRegisterEntry(
                    well_key=well_key,
                    axis="Depth",
                    tie_value_m=float(d),
                    source_row_indices=idxs,
                    original_tvdss_m=tuple(float(v) for v in tvdss_group),
                    original_owt_s=tuple(float(v) for v in owt_group),
                    duplicate_type="repeated_depth_distinct_tvdss_owt",
                    group_size=len(idxs),
                    tvdss_value_spread_m=spread_tvdss,
                    owt_value_spread_s=spread_owt,
                    selected_representative_tvdss_m=rep_tvdss,
                    selected_representative_owt_s=rep_owt,
                    conditioning_rule=(
                        "median of the tied group's TVDSS values (independently, median of "
                        "its OWT values); for this group's size "
                        f"({len(idxs)}), median "
                        + ("equals the arithmetic mean." if len(idxs) % 2 == 0 else "is the middle value.")
                    ),
                    affected_downstream_outputs=(
                        "conditioned_time_depth_curve",
                        "velocity_diagnostics",
                        "forward_inverse_interpolation",
                    ),
                )
            )

    cond_depth_arr = np.asarray(cond_depth, dtype=np.float64)
    cond_tvdss_arr = np.asarray(cond_tvdss, dtype=np.float64)
    cond_owt_arr = np.asarray(cond_owt, dtype=np.float64)

    if cond_depth_arr.size > 1 and not np.all(np.diff(cond_depth_arr) > 0.0):
        bad = np.where(np.diff(cond_depth_arr) <= 0.0)[0].tolist()
        raise TimeDepthError(
            f"{well_key}: conditioned Depth values are not strictly increasing at "
            f"interval index(es) {bad}; this indicates the source file's distinct Depth "
            f"values are not presented in non-decreasing order (not an ordinary repeated "
            f"tie-in) and cannot be safely conditioned by this function."
        )

    conditioned = ConditionedCheckshotData(
        Depth_conditioned_m=cond_depth_arr,
        TVDSS_conditioned_m=cond_tvdss_arr,
        OWT_conditioned_s=cond_owt_arr,
        n_raw_rows=n,
        n_conditioned_rows=int(cond_depth_arr.size),
        n_tie_groups=len(ties),
        conditioning_method=CONDITIONING_METHOD,
    )
    return tuple(ties), conditioned


# ---------------------------------------------------------------------------
# Velocity diagnostics
# ---------------------------------------------------------------------------
def compute_velocity_diagnostics(
    well_key: str, conditioned: ConditionedCheckshotData
) -> VelocityDiagnosticsResult:
    """
    Vavg_m_s = TVDSS_conditioned_m / OWT_conditioned_s at every conditioned
    row (NaN if OWT <= 0, which does not occur in any admitted file but is
    guarded rather than assumed).

    Vint_m_s[k] = (TVDSS[k+1]-TVDSS[k]) / (OWT[k+1]-OWT[k]) for the
    interval between conditioned rows k and k+1. NaN (never inf or a
    negative value) whenever either increment is <= 0 - this is the
    project's explicit "zero/negative delta-OWT must yield NaN + QC
    reason, never infinite/negative velocity" requirement. No smoothing is
    applied for visual/aesthetic purposes.
    """
    depth = conditioned.Depth_conditioned_m
    tvdss = conditioned.TVDSS_conditioned_m
    owt = conditioned.OWT_conditioned_s
    n = depth.size

    with np.errstate(divide="ignore", invalid="ignore"):
        vavg = np.where(owt > 0.0, tvdss / owt, np.nan)

    if n < 2:
        vint = np.zeros(0, dtype=np.float64)
        invalid_mask = np.zeros(0, dtype=bool)
    else:
        d_tvdss = np.diff(tvdss)
        d_owt = np.diff(owt)
        valid = (d_tvdss > 0.0) & (d_owt > 0.0)
        with np.errstate(divide="ignore", invalid="ignore"):
            vint = np.where(valid, d_tvdss / d_owt, np.nan)
        invalid_mask = ~valid

    reason_counts = {
        "non_positive_delta_owt": int(np.sum((np.diff(owt) <= 0.0)) if n >= 2 else 0),
        "non_positive_delta_tvdss": int(np.sum((np.diff(tvdss) <= 0.0)) if n >= 2 else 0),
    }

    return VelocityDiagnosticsResult(
        well_key=well_key,
        Depth_conditioned_m=depth,
        Vavg_m_s=vavg,
        Vint_m_s=vint,
        vint_invalid_mask=invalid_mask,
        n_vint_intervals=int(vint.size),
        n_vint_invalid=int(np.sum(invalid_mask)),
        vint_invalid_reason_counts=reason_counts,
    )


# ---------------------------------------------------------------------------
# Checkshot-vs-locked-survey depth-reference comparison
# ---------------------------------------------------------------------------
def _describe_residual_trend(depth: np.ndarray, residual: np.ndarray) -> str:
    if residual.size < 2:
        return "insufficient compared points to characterize a trend"
    std_resid = float(np.std(residual))
    rng = float(np.max(residual) - np.min(residual))
    if np.std(depth) > 0.0 and np.std(residual) > 0.0:
        r = float(np.corrcoef(depth, residual)[0, 1])
        slope = float(np.polyfit(depth, residual, 1)[0])
    else:
        r = 0.0
        slope = 0.0
    if std_resid < 0.02:
        return (
            f"near-constant residual across the compared depth interval "
            f"(standard deviation {std_resid:.4f} m, range {rng:.4f} m)"
        )
    if abs(r) > 0.6:
        direction = "increases" if slope > 0 else "decreases"
        return (
            f"residual {direction} approximately linearly with depth "
            f"(slope {slope:.3e} m/m, correlation r={r:.3f}, standard deviation {std_resid:.4f} m)"
        )
    return (
        f"residual does not show a strong monotonic trend with depth "
        f"(standard deviation {std_resid:.4f} m, correlation r={r:.3f})"
    )


def compare_checkshot_to_survey(
    well_key: str,
    depth_source_m: np.ndarray,
    tvdss_source_m: np.ndarray,
    survey_md_m: np.ndarray,
    survey_tvd_m: np.ndarray,
    datum_elevation_m: float,
    *,
    depth_basis_interpretation_status: str,
) -> CheckshotSurveyDepthComparisonResult:
    """
    Compare checkshot `Depth_source_m` (as a CANDIDATE MD axis) against
    TVDSS interpolated from the locked survey MD->TVD relationship for the
    same well (`survey_md_m`, `survey_tvd_m` - the Petrel `petrel_source_
    trace`, e.g. `DeviationWellResult.raw.MD_source_m` /
    `.TVD_source_m`), converting TVD to TVDSS via `datum_elevation_m`
    (`DeviationWellResult.header.datum_elevation_m`).

    Sign convention: residual_m = TVDSS_survey_interpolated_m -
    TVDSS_source_m (see `CheckshotSurveyDepthComparisonResult` docstring).
    Only checkshot rows whose Depth falls within the survey's own MD
    coverage are compared; rows outside that coverage are counted
    (`n_outside_survey_md_coverage`) but never extrapolated against.

    Validated (Increment 4.1.1 hardening - Blocking Defect 3): if EVERY
    checkshot `depth_source_m` row falls outside the survey's own MD
    coverage, there are zero rows left to compare and no residual
    statistic (min/max/mean/median/RMSE) can be computed - the pre-4.1.1
    implementation reached NumPy's `min`/`max`/`mean` reductions on an
    empty array in this case, raising an untyped `ValueError` ("zero-size
    array to reduction operation minimum which has no identity") instead
    of this module's own typed `TimeDepthError`. This is now detected
    explicitly and raises `TimeDepthError` naming the well, the
    checkshot's own Depth range, the survey's MD coverage, and stating
    plainly that no comparison was performed and no extrapolation was
    attempted - never a bare NumPy exception, and never a silently
    empty/NaN result.
    """
    depth_source_m = _as_1d_finite(f"{well_key}: compare_checkshot_to_survey: depth_source_m", depth_source_m)
    tvdss_source_m = _as_1d_finite(f"{well_key}: compare_checkshot_to_survey: tvdss_source_m", tvdss_source_m)
    survey_md_m = _as_1d_finite(f"{well_key}: compare_checkshot_to_survey: survey_md_m", survey_md_m)
    survey_tvd_m = _as_1d_finite(f"{well_key}: compare_checkshot_to_survey: survey_tvd_m", survey_tvd_m)
    _require_equal_length(
        [("depth_source_m", depth_source_m), ("tvdss_source_m", tvdss_source_m)]
    )
    _require_equal_length([("survey_md_m", survey_md_m), ("survey_tvd_m", survey_tvd_m)])

    if survey_md_m.size < 2 or not np.all(np.diff(survey_md_m) > 0.0):
        raise TimeDepthError(
            f"{well_key}: locked survey MD array is not strictly increasing; cannot "
            f"interpolate a well-defined MD->TVDSS relationship."
        )

    survey_tvdss_m = survey_tvd_m - float(datum_elevation_m)
    md_min, md_max = float(survey_md_m[0]), float(survey_md_m[-1])
    inside = (depth_source_m >= md_min) & (depth_source_m <= md_max)
    n_outside = int(np.sum(~inside))

    if not np.any(inside):
        checkshot_min, checkshot_max = float(np.min(depth_source_m)), float(np.max(depth_source_m))
        raise TimeDepthError(
            f"{well_key}: compare_checkshot_to_survey: every checkshot Depth_source_m row "
            f"falls outside the locked survey's own MD coverage - no comparison was "
            f"performed and no extrapolation was attempted. Checkshot Depth range: "
            f"[{checkshot_min:.4f}, {checkshot_max:.4f}] m. Survey MD coverage: "
            f"[{md_min:.4f}, {md_max:.4f}] m."
        )

    depth_in = depth_source_m[inside]
    tvdss_survey_interp = np.interp(depth_in, survey_md_m, survey_tvdss_m)
    residual = tvdss_survey_interp - tvdss_source_m[inside]

    return CheckshotSurveyDepthComparisonResult(
        well_key=well_key,
        n_compared=int(depth_in.size),
        n_outside_survey_md_coverage=n_outside,
        min_residual_m=float(np.min(residual)),
        max_residual_m=float(np.max(residual)),
        max_abs_residual_m=float(np.max(np.abs(residual))),
        mean_residual_m=float(np.mean(residual)),
        median_residual_m=float(np.median(residual)),
        rmse_m=float(np.sqrt(np.mean(residual**2))),
        first_residual_m=float(residual[0]) if residual.size else float("nan"),
        last_residual_m=float(residual[-1]) if residual.size else float("nan"),
        residual_trend_description=_describe_residual_trend(depth_in, residual),
        depth_basis_interpretation_status=depth_basis_interpretation_status,
        residual_sign_convention="residual_m = TVDSS_survey_interpolated_m - TVDSS_source_m",
    )


# ---------------------------------------------------------------------------
# Depth-domain forward interpolation (always well-defined post-conditioning)
# ---------------------------------------------------------------------------
def _coverage_mask(query: np.ndarray, axis_min: float, axis_max: float) -> np.ndarray:
    return (query >= axis_min) & (query <= axis_max)


def _interp_with_coverage(
    query: np.ndarray, x: np.ndarray, y: np.ndarray
) -> Tuple[np.ndarray, np.ndarray]:
    """Piecewise-linear interpolate `query` against strictly increasing `x`/`y`,
    returning (values with NaN outside [x[0], x[-1]], the coverage mask).

    Defensive validation (Increment 4.1): `x` and `y` must be finite,
    one-dimensional, equal-length, and `x` strictly increasing. Every
    caller in this module already guarantees this by construction
    (`ConditionedCheckshotData.Depth_conditioned_m` from
    `detect_and_condition_depth_ties`; `AxisConditionedLookupTable
    .axis_values` from `build_axis_conditioned_lookup_table`) - this is a
    second, independent line of defense against a future caller violating
    that contract, not a change to any currently-produced value.
    """
    x = _as_1d_finite("_interp_with_coverage: x", x)
    y = _as_1d_finite("_interp_with_coverage: y", y)
    _require_equal_length([("x", x), ("y", y)])
    _require_strictly_increasing("_interp_with_coverage: x", x)
    query = np.asarray(query, dtype=np.float64)
    mask = _coverage_mask(query, float(x[0]), float(x[-1]))
    out = np.full(query.shape, np.nan, dtype=np.float64)
    if np.any(mask):
        out[mask] = np.interp(query[mask], x, y)
    return out, mask


def depth_to_tvdss(depth_query, conditioned: ConditionedCheckshotData):
    """Depth -> TVDSS, piecewise-linear, NaN outside conditioned Depth coverage."""
    return _interp_with_coverage(
        np.atleast_1d(depth_query), conditioned.Depth_conditioned_m, conditioned.TVDSS_conditioned_m
    )


def depth_to_owt(depth_query, conditioned: ConditionedCheckshotData):
    """Depth -> OWT_source_s, piecewise-linear, NaN outside conditioned Depth coverage."""
    return _interp_with_coverage(
        np.atleast_1d(depth_query), conditioned.Depth_conditioned_m, conditioned.OWT_conditioned_s
    )


def depth_to_twt(depth_query, conditioned: ConditionedCheckshotData):
    """Depth -> TWT_s = 2 * (Depth -> OWT), via the LOCKED owt_to_twt conversion."""
    owt, mask = depth_to_owt(depth_query, conditioned)
    return owt_to_twt(owt), mask


# ---------------------------------------------------------------------------
# Time-domain forward/inverse interpolation: order-invariant axis-tie
# conditioning (Increment 4.1 - replaces the removed, order-dependent
# `_build_strictly_increasing_table` "keep first, drop later" logic)
# ---------------------------------------------------------------------------
def _group_by_exact_value(values: np.ndarray) -> Tuple[list, Dict[float, list]]:
    """Group indices of `values` by exact float equality, in order of
    first appearance (mirrors `detect_and_condition_depth_ties`'s own
    grouping logic, applied here to an axis-conditioned array instead of
    raw source data)."""
    groups: Dict[float, list] = {}
    order: list = []
    for i, v in enumerate(values):
        v = float(v)
        if v not in groups:
            groups[v] = []
            order.append(v)
        groups[v].append(i)
    return order, groups


def build_axis_conditioned_lookup_table(
    well_key: str,
    interpolation_direction: str,
    independent_axis_name: str,
    dependent_axis_name: str,
    depth_conditioned_m: np.ndarray,
    independent_values: np.ndarray,
    dependent_values: np.ndarray,
) -> Tuple[Tuple[AxisTimeDepthTieRegisterEntry, ...], AxisConditionedLookupTable]:
    """
    Build an order-invariant, axis-tie-conditioned lookup table for
    interpolating FROM `independent_values` (TVDSS or OWT, already
    Depth-tie conditioned - see `ConditionedCheckshotData`) TO
    `dependent_values` (the other of the two).

    Every value tied on `independent_values` is grouped by exact equality
    (regardless of which tied row appears first - this is the fix for the
    Increment 4 "keep first, drop later" order-dependence defect), its
    COMPLETE original rows are registered as one `AxisTimeDepthTieRegisterEntry`
    (never discarded unrecorded), and the group's `dependent_values`
    MEDIAN becomes that group's single conditioned representative -
    order-invariant, and disclosed as reducing to the arithmetic mean for
    a group of size 2. A group whose members ALSO share an identical
    dependent value (`tie_kind="identical_pair"`) still collapses to one
    row (its value is unchanged by taking the median of identical values)
    but is still counted and registered.

    Validated BEFORE any grouping is attempted (Increment 4.1.1 hardening
    - see "Reversal detection on the ORIGINAL sequence" below): the
    original, ungrouped `independent_values` sequence's successive
    differences must be non-negative everywhere (a tie, i.e. a zero
    difference, is allowed; a negative difference is a genuine reversal
    and raises `TimeDepthError` immediately, before grouping ever runs).
    After grouping, the resulting per-group `independent_values` sequence
    is additionally required to be STRICTLY increasing as a second,
    independent check; if it is not (which cannot actually occur once the
    pre-grouping check above has passed, since every tied value in a
    non-decreasing sequence is necessarily contiguous - see below), this
    function raises `TimeDepthError` rather than sorting, discarding, or
    forcing monotonicity. Real data from all three approved checkshot
    files, independently verified, contains only ties (zero-width
    intervals) at this stage, never a genuine reversal - see
    `INCREMENT_04_1_MANIFEST.md` / `INCREMENT_04_1_1_MANIFEST.md`.

    Reversal detection on the ORIGINAL sequence (Increment 4.1.1 fix)
    -------------------------------------------------------------------
    Increment 4.1's `_group_by_exact_value` groups every occurrence of a
    given axis value by EXACT equality regardless of its position in the
    array (this is what makes the median representative order-invariant
    for a genuine, adjacent tie). This has a defect: if the SAME axis
    value reappears after a LARGER value has already been seen (e.g.
    `[100.0, 200.0, 100.0]`), the global-equality grouping silently merges
    the first and third rows into one group, and the interior `200.0`
    simply disappears from the post-grouping array - `[100.0, 100.0,
    100.0]`'s grouped result is `[100.0]`, and `[100.0, 200.0, 100.0]`'s
    grouped result is `[100.0, 200.0]`, with no diff between adjacent
    output values ever exposing the `200 -> 100` reversal that existed in
    the original data. A reversal HIDDEN this way is never "a tie" - it
    is a decreasing step in the original measurement order, and this
    function must never sort, group away, or otherwise make it
    disappear. The fix checks `np.diff` of the ORIGINAL, ungrouped
    `independent_values` for any negative step BEFORE grouping is
    attempted, and raises immediately if one is found. This is safe and
    non-restrictive for every legitimate case: in a sequence that has
    already passed this check (no negative step anywhere), every group of
    equal values is guaranteed to be a CONTIGUOUS run (a non-adjacent
    repeat of the same value would require passing back down to it after
    a strictly larger value, which is exactly the negative step this
    check forbids) - so ordinary adjacent ties (e.g. `[100.0, 100.0,
    200.0]` or `[100.0, 200.0, 200.0, 300.0]`) are entirely unaffected and
    continue to be grouped and median-conditioned exactly as before.
    """
    depth_conditioned_m = _as_1d_finite(
        f"{well_key}: {interpolation_direction}: depth_conditioned_m", depth_conditioned_m
    )
    independent_values = _as_1d_finite(
        f"{well_key}: {interpolation_direction}: {independent_axis_name}", independent_values
    )
    dependent_values = _as_1d_finite(
        f"{well_key}: {interpolation_direction}: {dependent_axis_name}", dependent_values
    )
    _require_equal_length(
        [
            ("depth_conditioned_m", depth_conditioned_m),
            (independent_axis_name, independent_values),
            (dependent_axis_name, dependent_values),
        ]
    )
    _require_strictly_increasing(
        f"{well_key}: {interpolation_direction}: depth_conditioned_m", depth_conditioned_m
    )

    # Increment 4.1.1 fix (Blocking Defect 1): check the ORIGINAL,
    # ungrouped sequence for a genuine reversal BEFORE any grouping is
    # attempted - grouping by global exact-value equality can otherwise
    # silently hide a reversal that passes back down to an
    # already-seen value (see the docstring section above). A zero
    # difference (an ordinary tie) is explicitly allowed here; only a
    # strictly negative difference is a reversal.
    if independent_values.size > 1:
        _orig_diffs = np.diff(independent_values)
        _bad = np.where(_orig_diffs < 0.0)[0]
        if _bad.size > 0:
            raise TimeDepthError(
                f"{well_key}: {interpolation_direction}: {independent_axis_name} contains a "
                f"genuine reversal (a decreasing step, not a tie) at original interval "
                f"index(es) {_bad.tolist()}; this function never sorts, groups away, or "
                f"otherwise hides a reversal by merging a repeated value with an earlier, "
                f"non-adjacent occurrence of the same value."
            )

    order, groups = _group_by_exact_value(independent_values)

    entries: list = []
    out_axis: list = []
    out_dependent: list = []
    n_collapsed = 0
    n_identical_pairs = 0
    n_nonunique = 0

    for v in order:
        idxs = tuple(groups[v])
        dep_group = dependent_values[list(idxs)]
        depth_group = depth_conditioned_m[list(idxs)]
        rep_dependent = float(np.median(dep_group))
        out_axis.append(v)
        out_dependent.append(rep_dependent)

        if len(idxs) > 1:
            spread = float(np.max(dep_group) - np.min(dep_group))
            is_identical = spread == 0.0
            tie_kind = "identical_pair" if is_identical else "genuinely_non_unique"
            if is_identical:
                n_identical_pairs += 1
            else:
                n_nonunique += 1
            n_collapsed += len(idxs) - 1
            entries.append(
                AxisTimeDepthTieRegisterEntry(
                    well_key=well_key,
                    interpolation_direction=interpolation_direction,
                    axis=independent_axis_name,
                    dependent_axis=dependent_axis_name,
                    tie_axis_value=float(v),
                    conditioned_row_indices=idxs,
                    associated_depth_m=tuple(float(x) for x in depth_group),
                    original_dependent_values=tuple(float(x) for x in dep_group),
                    group_size=len(idxs),
                    dependent_value_spread=spread,
                    selected_representative_dependent_value=rep_dependent,
                    conditioning_rule=(
                        f"median of the tied group's {dependent_axis_name} values, computed "
                        f"independently of which tied row appeared first (order-invariant); "
                        f"for this group's size ({len(idxs)}), median "
                        + ("equals the arithmetic mean." if len(idxs) % 2 == 0 else "is the middle value.")
                    ),
                    tie_kind=tie_kind,
                    affected_downstream_outputs=(
                        f"{interpolation_direction}_lookup_table",
                        "forward_inverse_time_depth_interpolation",
                    ),
                )
            )

    out_axis_arr = np.asarray(out_axis, dtype=np.float64)
    out_dependent_arr = np.asarray(out_dependent, dtype=np.float64)

    _require_strictly_increasing(
        f"{well_key}: {interpolation_direction}: {independent_axis_name} after axis-tie conditioning",
        out_axis_arr,
    )

    table = AxisConditionedLookupTable(
        well_key=well_key,
        interpolation_direction=interpolation_direction,
        independent_axis_name=independent_axis_name,
        dependent_axis_name=dependent_axis_name,
        axis_values=out_axis_arr,
        dependent_values=out_dependent_arr,
        n_input_points=int(independent_values.size),
        n_output_points=int(out_axis_arr.size),
        n_axis_tie_groups=len(entries),
        n_collapsed_points=n_collapsed,
        n_identical_pairs=n_identical_pairs,
        n_genuinely_nonunique_groups=n_nonunique,
        conditioning_method=AXIS_TIE_CONDITIONING_METHOD,
    )
    return tuple(entries), table


def build_axis_conditioned_tables_for_well(
    well_key: str, conditioned: ConditionedCheckshotData
) -> Tuple[Tuple[AxisTimeDepthTieRegisterEntry, ...], Dict[str, AxisConditionedLookupTable]]:
    """
    Build BOTH axis-conditioned lookup tables (`tvdss_to_owt` and
    `owt_to_tvdss`) for one well in a single call, returning the combined,
    flat tuple of every `AxisTimeDepthTieRegisterEntry` found across both
    directions plus a dict of the two `AxisConditionedLookupTable`
    objects keyed by direction name. This is the function
    `p2mem.io.checkshot.load_checkshot_file` calls once per well so every
    downstream consumer (interpolation, the CSV/JSON exporters, the
    notebook) shares the same order-invariant tables.
    """
    entries_to_owt, table_to_owt = build_axis_conditioned_lookup_table(
        well_key,
        "tvdss_to_owt",
        "TVDSS_conditioned_m",
        "OWT_conditioned_s",
        conditioned.Depth_conditioned_m,
        conditioned.TVDSS_conditioned_m,
        conditioned.OWT_conditioned_s,
    )
    entries_to_tvdss, table_to_tvdss = build_axis_conditioned_lookup_table(
        well_key,
        "owt_to_tvdss",
        "OWT_conditioned_s",
        "TVDSS_conditioned_m",
        conditioned.Depth_conditioned_m,
        conditioned.OWT_conditioned_s,
        conditioned.TVDSS_conditioned_m,
    )
    return (
        tuple(entries_to_owt) + tuple(entries_to_tvdss),
        {"tvdss_to_owt": table_to_owt, "owt_to_tvdss": table_to_tvdss},
    )


def tvdss_to_owt(tvdss_query, table: AxisConditionedLookupTable):
    """
    TVDSS -> OWT_source_s, piecewise-linear against the order-invariant
    `tvdss_to_owt` `AxisConditionedLookupTable` (see
    `build_axis_conditioned_lookup_table` /
    `build_axis_conditioned_tables_for_well`). Returns (values with NaN
    outside coverage, coverage_mask).
    """
    if table.interpolation_direction != "tvdss_to_owt":
        raise TimeDepthError(
            f"tvdss_to_owt requires a table built with interpolation_direction="
            f"'tvdss_to_owt'; got {table.interpolation_direction!r}."
        )
    return _interp_with_coverage(np.atleast_1d(tvdss_query), table.axis_values, table.dependent_values)


def owt_to_tvdss(owt_query, table: AxisConditionedLookupTable):
    """
    OWT_source_s -> TVDSS, piecewise-linear against the order-invariant
    `owt_to_tvdss` `AxisConditionedLookupTable`. Returns (values with NaN
    outside coverage, coverage_mask).
    """
    if table.interpolation_direction != "owt_to_tvdss":
        raise TimeDepthError(
            f"owt_to_tvdss requires a table built with interpolation_direction="
            f"'owt_to_tvdss'; got {table.interpolation_direction!r}."
        )
    return _interp_with_coverage(np.atleast_1d(owt_query), table.axis_values, table.dependent_values)


def tvdss_to_twt(tvdss_query, table: AxisConditionedLookupTable):
    """TVDSS -> TWT_s, via TVDSS -> OWT (the supplied `tvdss_to_owt` table)
    then the LOCKED owt_to_twt conversion."""
    owt, mask = tvdss_to_owt(tvdss_query, table)
    return owt_to_twt(owt), mask


def twt_to_tvdss(twt_query, table: AxisConditionedLookupTable):
    """TWT_s -> TVDSS, via the LOCKED twt_to_owt conversion then OWT -> TVDSS
    (the supplied `owt_to_tvdss` table)."""
    owt_query = twt_to_owt(np.atleast_1d(twt_query))
    return owt_to_tvdss(owt_query, table)


# ---------------------------------------------------------------------------
# LAS MD -> checkshot-derived OWT/TWT (partial-coverage NaN masking)
# ---------------------------------------------------------------------------
def map_las_md_to_checkshot_time(
    well_key: str, las_md_source_m: np.ndarray, conditioned: ConditionedCheckshotData
) -> Tuple[np.ndarray, np.ndarray, TimeDepthMappingSummary]:
    """
    Map a well's canonical LAS `MD_m` onto checkshot-derived OWT/TWT,
    treating LAS MD directly as the checkshot Depth axis (both wells'
    Depth-basis interpretation is `candidate_md_evaluated_against_locked_
    survey` - see `config/checkshot_contracts.yml`). Samples outside
    checkshot Depth coverage are NaN, never extrapolated
    (`n_extrapolated` is always 0 by construction). Returns the full
    per-sample arrays (for notebook plotting only - never packaged as a
    per-sample CSV, per project policy) plus the deterministic
    `TimeDepthMappingSummary`.
    """
    las_md = np.asarray(las_md_source_m, dtype=np.float64)
    if las_md.ndim != 1 or las_md.size == 0 or not np.all(np.isfinite(las_md)):
        raise TimeDepthError(f"{well_key}: LAS MD array must be a non-empty, finite 1-D array.")

    owt_mapped, inside_mask = depth_to_owt(las_md, conditioned)
    twt_mapped = owt_to_twt(owt_mapped)

    depth_min = float(conditioned.Depth_conditioned_m[0])
    depth_max = float(conditioned.Depth_conditioned_m[-1])
    n_shallower = int(np.sum(las_md < depth_min))
    n_deeper = int(np.sum(las_md > depth_max))
    n_inside = int(np.sum(inside_mask))

    summary = TimeDepthMappingSummary(
        well_key=well_key,
        n_las_samples=int(las_md.size),
        n_inside_coverage=n_inside,
        n_shallower_than_coverage=n_shallower,
        n_deeper_than_coverage=n_deeper,
        mapped_fraction=float(n_inside) / float(las_md.size),
        checkshot_depth_min_m=depth_min,
        checkshot_depth_max_m=depth_max,
        las_md_min_m=float(np.min(las_md)),
        las_md_max_m=float(np.max(las_md)),
        interpolation_method=INTERPOLATION_METHOD,
        n_extrapolated=0,
    )
    return owt_mapped, twt_mapped, summary


# ---------------------------------------------------------------------------
# Poseidon 2 sonic-checkshot drift diagnostic
# ---------------------------------------------------------------------------
def find_longest_finite_positive_run(md_m: np.ndarray, values: np.ndarray) -> Tuple[int, int]:
    """
    Return the (start_index, end_index_inclusive) of the longest maximal
    contiguous run of finite, strictly positive `values` in `md_m` order
    (`md_m` assumed already in ascending log order, as every canonical LAS
    `MD_m` array in this project is). Raises `TimeDepthError` if no finite
    positive sample exists at all.

    Validated (Increment 4.1.1 hardening): `md_m` and `values` must each
    be one-dimensional and of equal, compatible shape when this function
    is called directly (`compute_sonic_checkshot_drift` already validates
    both arrays more strictly - full MD finiteness/monotonicity - before
    calling this function, but this function is part of the module's
    public API and must not silently misbehave on a malformed shape when
    called on its own).
    """
    md_m = np.asarray(md_m)
    values = np.asarray(values)
    if md_m.ndim != 1:
        raise TimeDepthError(f"find_longest_finite_positive_run: md_m must be one-dimensional; got shape {md_m.shape}.")
    if values.ndim != 1:
        raise TimeDepthError(f"find_longest_finite_positive_run: values must be one-dimensional; got shape {values.shape}.")
    if md_m.shape != values.shape:
        raise TimeDepthError(
            f"find_longest_finite_positive_run: md_m and values must share one shape; got "
            f"{md_m.shape} and {values.shape}."
        )
    valid = np.isfinite(values) & (values > 0.0)
    idx_valid = np.where(valid)[0]
    if idx_valid.size == 0:
        raise TimeDepthError("no finite, strictly positive sample found to select a run from.")
    splits = np.where(np.diff(idx_valid) > 1)[0]
    runs = np.split(idx_valid, splits + 1)
    longest = max(runs, key=lambda r: r.size)
    return int(longest[0]), int(longest[-1])


def compute_sonic_checkshot_drift(
    well_key: str,
    md_m: np.ndarray,
    vp_m_s: np.ndarray,
    conditioned: ConditionedCheckshotData,
) -> SonicCheckshotDriftResult:
    """
    Poseidon-2-only diagnostic: integrate sonic slowness (1/VP_m_s)
    trapezoidally over MD across the longest contiguous finite-VP interval
    (`find_longest_finite_positive_run`), and compare it against the
    checkshot-interpolated OWT increment across the identical MD/Depth
    endpoints (`depth_to_owt`, applied to `conditioned` - i.e. Depth is
    used directly as the comparison axis for the same reason documented
    in `map_las_md_to_checkshot_time`).

    Both signed differences are reported (`sonic_minus_checkshot_ms`,
    `checkshot_minus_sonic_ms`) plus a signed percent
    (`sonic_minus_checkshot_percent`, relative to the checkshot
    increment). This is a DIAGNOSTIC comparison only - see `limitations`;
    VP_m_s, DTCO, checkshot OWT, and the conditioned time-depth curve are
    never modified as a result of this computation.

    Validated (Increment 4.1 hardening, EXTENDED in Increment 4.1.1 - see
    below): `md_m` and `vp_m_s` must each be one-dimensional, finite*, and
    of equal shape (*`vp_m_s` may contain non-finite/non-positive samples
    anywhere - those are exactly what `find_longest_finite_positive_run`
    screens for - but `md_m` itself, the log's own canonical depth index,
    must be entirely finite); the resulting checkshot OWT increment over
    the selected interval must be strictly positive (a non-positive
    increment would make the percent-difference calculation meaningless
    or divide-by-zero).

    Full-MD monotonicity requirement (Increment 4.1.1 fix - Blocking
    Defect 2). The Increment 4.1 implementation checked strict MD
    monotonicity only WITHIN the selected finite-positive-VP run, not
    across the complete canonical `md_m` array. This allowed a decreasing
    or duplicate MD value OUTSIDE the selected run (e.g. at a station
    whose `VP_m_s` happens to be NaN, and is therefore excluded from the
    run-selection step entirely) to pass silently - the canonical MD
    index itself is a structural precondition of this function's
    contract, not just of the one sub-interval it happens to select for
    integration. This function now requires the COMPLETE `md_m` array to
    be strictly increasing (no decreasing step and no duplicate MD value
    anywhere in the log, not only inside the selected run) BEFORE the run
    is even selected, raising `TimeDepthError` otherwise. This validation
    does not alter the arithmetic path for already-valid input - the real
    Poseidon 2 canonical `MD_m` array (31,897 samples) was independently
    confirmed entirely finite and strictly increasing throughout before
    this hardening was added, so the real Poseidon 2 result is bit-for-
    bit unchanged from Increment 4.1's.
    """
    md_m = _as_1d_finite(f"{well_key}: compute_sonic_checkshot_drift: md_m", md_m)
    vp_m_s = np.asarray(vp_m_s, dtype=np.float64)
    if vp_m_s.ndim != 1:
        raise TimeDepthError(f"{well_key}: vp_m_s must be a one-dimensional array; got shape {vp_m_s.shape}.")
    if md_m.shape != vp_m_s.shape:
        raise TimeDepthError(
            f"{well_key}: MD and VP arrays must share one shape; got {md_m.shape} and {vp_m_s.shape}."
        )
    # Increment 4.1.1 fix (Blocking Defect 2): the COMPLETE canonical MD
    # index must be strictly increasing, not only the sub-run this
    # function later selects for integration - a decreasing or duplicate
    # MD value outside the selected run (e.g. at a NaN-VP station) must
    # not be allowed to pass silently.
    _require_strictly_increasing(f"{well_key}: compute_sonic_checkshot_drift: md_m (complete canonical array)", md_m)

    start_idx, end_idx = find_longest_finite_positive_run(md_m, vp_m_s)
    md_run = md_m[start_idx : end_idx + 1]
    vp_run = vp_m_s[start_idx : end_idx + 1]
    if md_run.size < 2:
        raise TimeDepthError(f"{well_key}: selected sonic interval has fewer than 2 samples.")
    _require_strictly_increasing(f"{well_key}: selected sonic interval's MD_m", md_run)

    slowness = 1.0 / vp_run
    md1, md2 = float(md_run[0]), float(md_run[-1])
    sonic_transit_time_s = trapezoidal_integrate(slowness, md_run)

    depth_min = float(conditioned.Depth_conditioned_m[0])
    depth_max = float(conditioned.Depth_conditioned_m[-1])
    if md1 < depth_min or md2 > depth_max:
        raise TimeDepthError(
            f"{well_key}: sonic interval [{md1:.4f}, {md2:.4f}] extends beyond checkshot Depth "
            f"coverage [{depth_min:.4f}, {depth_max:.4f}]; this diagnostic never extrapolates "
            f"checkshot time beyond its validated coverage."
        )

    owt_at_md1 = float(np.interp(md1, conditioned.Depth_conditioned_m, conditioned.OWT_conditioned_s))
    owt_at_md2 = float(np.interp(md2, conditioned.Depth_conditioned_m, conditioned.OWT_conditioned_s))
    checkshot_owt_increment_s = owt_at_md2 - owt_at_md1
    if checkshot_owt_increment_s <= 0.0:
        raise TimeDepthError(
            f"{well_key}: checkshot OWT increment over [{md1:.4f}, {md2:.4f}] is non-positive "
            f"({checkshot_owt_increment_s:.6g} s); this diagnostic requires a strictly positive "
            f"checkshot time increment to compare against the integrated sonic transit time."
        )

    diff_ms = (sonic_transit_time_s - checkshot_owt_increment_s) * 1000.0
    pct = (
        (sonic_transit_time_s - checkshot_owt_increment_s) / checkshot_owt_increment_s * 100.0
        if checkshot_owt_increment_s != 0.0
        else float("nan")
    )

    return SonicCheckshotDriftResult(
        well_key=well_key,
        md_interval_start_m=md1,
        md_interval_end_m=md2,
        n_sonic_samples=int(md_run.size),
        selection_criteria=(
            "longest maximal contiguous run of finite, strictly positive VP_m_s samples in "
            "canonical LAS MD order"
        ),
        integration_method=INTEGRATION_METHOD,
        sonic_transit_time_s=sonic_transit_time_s,
        checkshot_owt_increment_s=checkshot_owt_increment_s,
        sonic_minus_checkshot_ms=diff_ms,
        checkshot_minus_sonic_ms=-diff_ms,
        sonic_minus_checkshot_percent=pct,
        limitations=(
            "Sonic transit time is integrated along the borehole (MD) path; checkshot OWT is a "
            "vertically corrected (near-vertical raypath) travel time - the two are not measured "
            "along an identical raypath for a deviated interval.",
            "No acquisition or environmental corrections (temperature, pressure, tool eccentering, "
            "cycle-skip editing) are supplied with either the sonic log or the checkshot survey; "
            "none are applied here.",
            "No run-merging metadata is available to confirm the sonic log is a single, "
            "continuously calibrated logging run over the selected interval.",
            "This is a diagnostic comparison only, not a calibration: no drift correction is "
            "applied to VP_m_s, DTCO, checkshot OWT, or the conditioned time-depth curve.",
        ),
    )


#### `p2mem/io/checkshot.py` — checkshot file parser and per-file contract resolver

In [ ]:
%%writefile p2mem/io/checkshot.py
"""
p2mem.io.checkshot - Auditable checkshot (velocity survey) file parser and
per-file contract resolution (Increment 4).

Why this module exists
------------------------
The three approved checkshot files (Poseidon 2, Boreas 1, Proteus 1ST2)
are plain-text Schlumberger-format velocity surveys with a minimal,
fixed two-line header (one free-text survey statement, one tab-separated
column-header line: "Depth\\tTVDSS\\tOWT(sec)") followed by a three-column
data table. This is structurally much simpler than the Petrel deviation-
survey format (`p2mem.io.deviation`), but the same discipline is applied
at reduced complexity: STRUCTURAL PARSING (can the file even be
tokenized - is the two-line header present, does every data row have
exactly 3 numeric tokens) is kept separate from CONTRACT RESOLUTION (does
this specific file's actual header/data agree with what
`config/checkshot_contracts.yml` says it should be: filename, SHA-256,
exact header text, column order, row count, numeric ranges). A structural
defect is a `CheckshotParsingError`; a contract mismatch is a
`CheckshotContractError`. Both are always ERROR-severity and blocking.

Raw file identity is never altered: this module never rewrites, renames,
or "cleans" a raw checkshot file; it only reads it (recording identity by
SHA-256) and reports what it finds.

Depth-basis disclosure (DEPTH_BASIS_NOT_EXPLICITLY_DECLARED)
------------------------------------------------------------------
Every approved file's header declares OWT (seconds), "vertically
corrected, relative to SRD (MSL)", and supplies a TVDSS column - but its
first column is labelled only "Depth", never "MD". This module never
silently renames that column to `MD_m`; it is stored as `Depth_source_m`
(see `p2mem.checkshot_models`), and a WARNING-severity
`CheckshotIngestionIssue` (this code) is always raised on a successful
load, disclosing that the column's MD-like behavior must be, and was,
evaluated empirically (see `p2mem.time_depth.compare_checkshot_to_survey`,
invoked by `load_checkshot_file` whenever a locked survey trajectory is
supplied) rather than assumed from the header alone.

Well-identity evidence disclosure
------------------------------------
`Proteus1-Checkshot.txt` carries no embedded well identifier of its own;
its association with Proteus 1ST2 is declared in
`config/checkshot_contracts.yml` as `well_identity_evidence_status:
inferred_unverified` (filename, project context, and depth-tie
plausibility only - never independently proven by file content). This
module raises a dedicated, always-WARNING
`WELL_IDENTITY_INFERRED_UNVERIFIED` issue for any file contract declaring
that status, and never upgrades that status to "verified" on the strength
of a successful contract resolution (a clean structural/numeric match
does not, by itself, prove which physical well a file came from).
"""

from __future__ import annotations

import hashlib
from pathlib import Path
from typing import Dict, Tuple

import numpy as np
import yaml

from p2mem.checkshot_models import (
    VALID_CHECKSHOT_AVAILABILITY_STATUSES,
    VALID_IDENTITY_EVIDENCE_STATUSES,
    VALID_MODEL_USE_STATUSES,
    CheckshotFileContract,
    CheckshotHeaderInfo,
    CheckshotIngestionFailure,
    CheckshotIngestionIssue,
    CheckshotStationData,
    CheckshotWellResult,
)
from p2mem.time_depth import (
    TimeDepthError,
    build_axis_conditioned_tables_for_well,
    compare_checkshot_to_survey,
    compute_velocity_diagnostics,
    detect_and_condition_depth_ties,
)

__all__ = [
    "CheckshotFileNotFoundError",
    "CheckshotParsingError",
    "CheckshotContractDefinitionError",
    "CheckshotContractError",
    "DEPTH_BASIS_NOT_EXPLICITLY_DECLARED_CODE",
    "WELL_IDENTITY_INFERRED_UNVERIFIED_CODE",
    "load_checkshot_contract_config",
    "parse_checkshot_header",
    "read_checkshot_rows",
    "resolve_checkshot_contract",
    "load_checkshot_file",
    "load_checkshot_surveys",
]

DEPTH_BASIS_NOT_EXPLICITLY_DECLARED_CODE = "DEPTH_BASIS_NOT_EXPLICITLY_DECLARED"
WELL_IDENTITY_INFERRED_UNVERIFIED_CODE = "WELL_IDENTITY_INFERRED_UNVERIFIED"

_REQUIRED_COLUMN_COUNT = 3


# ---------------------------------------------------------------------------
# Exceptions
# ---------------------------------------------------------------------------
class CheckshotFileNotFoundError(FileNotFoundError):
    """The checkshot file path given to `load_checkshot_file` does not exist."""


class CheckshotParsingError(ValueError):
    """
    A structural defect in the checkshot text file itself: fewer than 2
    header lines, a data row without exactly 3 tab-separated numeric
    tokens, a non-finite (NaN/Inf) literal token, or no data rows at all.
    Raised before any contract is consulted.
    """


class CheckshotContractDefinitionError(ValueError):
    """
    `config/checkshot_contracts.yml` itself is malformed: a duplicate
    top-level file key, a missing required field, an invalid type, an
    unsupported enum value, or an internally inconsistent expected-value
    set (e.g. min > max).
    """


class CheckshotContractError(RuntimeError):
    """
    A specific file's actual parsed header or data does not agree with
    its `CheckshotFileContract`: wrong filename, SHA-256 mismatch, header
    text mismatch, column-order/count mismatch, row-count mismatch, or a
    numeric range outside the contract's declared tolerance. Carries
    `.issues` (the full tuple, ERROR and WARNING alike).
    """

    def __init__(self, message: str, issues: Tuple[CheckshotIngestionIssue, ...]):
        super().__init__(message)
        self.issues = issues


# ---------------------------------------------------------------------------
# Contract configuration (config/checkshot_contracts.yml)
# ---------------------------------------------------------------------------
class _NoDuplicateKeySafeLoader(yaml.SafeLoader):
    """
    A `yaml.SafeLoader` subclass that raises on a duplicate mapping key
    rather than silently keeping only the last occurrence. File-scoped to
    this module (mirrors, but does not import, the identical private
    pattern in `p2mem.io.deviation` - that class is private to its own
    module and not exported for reuse).
    """


def _construct_mapping_no_duplicates(loader: yaml.SafeLoader, node, deep: bool = False):
    mapping: Dict = {}
    for key_node, value_node in node.value:
        key = loader.construct_object(key_node, deep=deep)
        if key in mapping:
            raise CheckshotContractDefinitionError(
                f"Duplicate key {key!r} found while parsing checkshot contract YAML "
                f"(line {key_node.start_mark.line + 1}); duplicate contract keys are "
                f"rejected rather than silently keeping only the last one."
            )
        value = loader.construct_object(value_node, deep=deep)
        mapping[key] = value
    return mapping


_NoDuplicateKeySafeLoader.add_constructor(
    yaml.resolver.BaseResolver.DEFAULT_MAPPING_TAG, _construct_mapping_no_duplicates
)

_REQUIRED_FILE_FIELDS = (
    "expected_sha256",
    "project_well_key",
    "well_identity_evidence_status",
    "well_identity_evidence_notes",
    "model_use_status",
    "expected_survey_statement",
    "expected_column_header_line",
    "expected_column_order",
    "expected_time_type",
    "expected_time_unit",
    "expected_vertical_correction_fragment",
    "expected_srd_reference_fragment",
    "expected_row_count",
    "expected_depth_min_m",
    "expected_depth_max_m",
    "expected_tvdss_min_m",
    "expected_tvdss_max_m",
    "expected_owt_min_s",
    "expected_owt_max_s",
    "numeric_range_tolerance",
    "depth_basis_interpretation_status",
    "duplicate_tie_policy",
    "interpolation_policy",
    "extrapolation_policy",
    "notes",
)


def _num(value, field_name: str, filename: str):
    if isinstance(value, bool) or not isinstance(value, (int, float)):
        raise CheckshotContractDefinitionError(
            f"{filename}: field {field_name!r} must be numeric, got {value!r}."
        )
    return float(value)


def load_checkshot_contract_config(yaml_path: str) -> Dict[str, CheckshotFileContract]:
    """
    Load and validate `config/checkshot_contracts.yml`, returning a dict
    keyed by exact source filename (e.g. "Poseidon2-Checkshot.txt").
    Raises `CheckshotContractDefinitionError` for any structural or
    internal-consistency problem in the YAML itself, before any checkshot
    file is opened.
    """
    with open(yaml_path, "r", encoding="utf-8") as fh:
        raw = yaml.load(fh, Loader=_NoDuplicateKeySafeLoader)

    if not isinstance(raw, dict) or "files" not in raw:
        raise CheckshotContractDefinitionError(
            f"{yaml_path}: top-level YAML must be a mapping with a 'files' key."
        )
    files_section = raw["files"]
    if not isinstance(files_section, dict) or not files_section:
        raise CheckshotContractDefinitionError(f"{yaml_path}: 'files' must be a non-empty mapping.")

    contracts: Dict[str, CheckshotFileContract] = {}
    for filename, entry in files_section.items():
        if not isinstance(entry, dict):
            raise CheckshotContractDefinitionError(f"{filename}: contract entry must be a mapping.")
        missing = [f for f in _REQUIRED_FILE_FIELDS if f not in entry]
        if missing:
            raise CheckshotContractDefinitionError(f"{filename}: missing required field(s) {missing}.")

        col_order = entry["expected_column_order"]
        if (
            not isinstance(col_order, list)
            or not col_order
            or len(set(col_order)) != len(col_order)
            or not all(isinstance(c, str) for c in col_order)
        ):
            raise CheckshotContractDefinitionError(
                f"{filename}: expected_column_order must be a list of unique strings."
            )
        if len(col_order) != _REQUIRED_COLUMN_COUNT:
            raise CheckshotContractDefinitionError(
                f"{filename}: expected_column_order must have exactly {_REQUIRED_COLUMN_COUNT} "
                f"entries (Depth, TVDSS, OWT); got {len(col_order)}."
            )

        identity_status = entry["well_identity_evidence_status"]
        if identity_status not in VALID_IDENTITY_EVIDENCE_STATUSES:
            raise CheckshotContractDefinitionError(
                f"{filename}: well_identity_evidence_status must be one of "
                f"{VALID_IDENTITY_EVIDENCE_STATUSES}, got {identity_status!r}."
            )
        model_use = entry["model_use_status"]
        if model_use not in VALID_MODEL_USE_STATUSES:
            raise CheckshotContractDefinitionError(
                f"{filename}: model_use_status must be one of {VALID_MODEL_USE_STATUSES}, "
                f"got {model_use!r}."
            )

        row_count = entry["expected_row_count"]
        if isinstance(row_count, bool) or not isinstance(row_count, int) or row_count < 1:
            raise CheckshotContractDefinitionError(
                f"{filename}: expected_row_count must be a positive integer."
            )

        depth_min = _num(entry["expected_depth_min_m"], "expected_depth_min_m", filename)
        depth_max = _num(entry["expected_depth_max_m"], "expected_depth_max_m", filename)
        tvdss_min = _num(entry["expected_tvdss_min_m"], "expected_tvdss_min_m", filename)
        tvdss_max = _num(entry["expected_tvdss_max_m"], "expected_tvdss_max_m", filename)
        owt_min = _num(entry["expected_owt_min_s"], "expected_owt_min_s", filename)
        owt_max = _num(entry["expected_owt_max_s"], "expected_owt_max_s", filename)
        tolerance = _num(entry["numeric_range_tolerance"], "numeric_range_tolerance", filename)
        if tolerance < 0.0:
            raise CheckshotContractDefinitionError(f"{filename}: numeric_range_tolerance must be >= 0.")
        for lo, hi, name in (
            (depth_min, depth_max, "depth"),
            (tvdss_min, tvdss_max, "tvdss"),
            (owt_min, owt_max, "owt"),
        ):
            if lo > hi:
                raise CheckshotContractDefinitionError(f"{filename}: expected {name} min > max.")

        for f in (
            "expected_sha256",
            "project_well_key",
            "well_identity_evidence_notes",
            "expected_survey_statement",
            "expected_column_header_line",
            "expected_time_type",
            "expected_time_unit",
            "expected_vertical_correction_fragment",
            "expected_srd_reference_fragment",
            "depth_basis_interpretation_status",
            "duplicate_tie_policy",
            "interpolation_policy",
            "extrapolation_policy",
            "notes",
        ):
            if not isinstance(entry[f], str) or not entry[f].strip():
                raise CheckshotContractDefinitionError(f"{filename}: field {f!r} must be a non-empty string.")

        contracts[filename] = CheckshotFileContract(
            source_filename=filename,
            expected_sha256=entry["expected_sha256"],
            project_well_key=entry["project_well_key"],
            well_identity_evidence_status=identity_status,
            well_identity_evidence_notes=entry["well_identity_evidence_notes"],
            model_use_status=model_use,
            expected_survey_statement=entry["expected_survey_statement"],
            expected_column_header_line=entry["expected_column_header_line"],
            expected_column_order=tuple(col_order),
            expected_column_count=_REQUIRED_COLUMN_COUNT,
            expected_time_type=entry["expected_time_type"],
            expected_time_unit=entry["expected_time_unit"],
            expected_vertical_correction_fragment=entry["expected_vertical_correction_fragment"],
            expected_srd_reference_fragment=entry["expected_srd_reference_fragment"],
            expected_row_count=row_count,
            expected_depth_min_m=depth_min,
            expected_depth_max_m=depth_max,
            expected_tvdss_min_m=tvdss_min,
            expected_tvdss_max_m=tvdss_max,
            expected_owt_min_s=owt_min,
            expected_owt_max_s=owt_max,
            numeric_range_tolerance=tolerance,
            depth_basis_interpretation_status=entry["depth_basis_interpretation_status"],
            duplicate_tie_policy=entry["duplicate_tie_policy"],
            interpolation_policy=entry["interpolation_policy"],
            extrapolation_policy=entry["extrapolation_policy"],
            notes=entry["notes"],
        )
    return contracts


# ---------------------------------------------------------------------------
# Structural parsing
# ---------------------------------------------------------------------------
def parse_checkshot_header(path: str) -> CheckshotHeaderInfo:
    """
    Parse the fixed two-line checkshot header: line 1 is a free-text
    survey statement, line 2 is the tab-separated column-header line.
    Raises `CheckshotFileNotFoundError` if the path does not exist, or
    `CheckshotParsingError` if fewer than 2 lines are present.

    Raw files in this project use CRLF line endings; opened in universal-
    newlines text mode, Python normalizes this transparently. That is
    disclosed here (`line_ending_convention`), not "corrected" - the raw
    file bytes on disk are never altered by this module.
    """
    p = Path(path)
    if not p.exists():
        raise CheckshotFileNotFoundError(f"Checkshot file not found: {path}")

    raw_bytes = p.read_bytes()
    sha256 = hashlib.sha256(raw_bytes).hexdigest()
    line_ending = "CRLF" if b"\r\n" in raw_bytes else ("LF" if b"\n" in raw_bytes else "NONE")

    text = raw_bytes.decode("utf-8")
    lines = text.splitlines()
    if len(lines) < 2:
        raise CheckshotParsingError(
            f"{path}: expected at least 2 header lines (survey statement, column-header "
            f"line); found {len(lines)}."
        )
    survey_statement = lines[0]
    column_header_line = lines[1]
    column_names = tuple(column_header_line.split("\t"))
    if len(column_names) != _REQUIRED_COLUMN_COUNT:
        raise CheckshotParsingError(
            f"{path}: column-header line must have exactly {_REQUIRED_COLUMN_COUNT} "
            f"tab-separated fields; found {len(column_names)} in {column_header_line!r}."
        )

    return CheckshotHeaderInfo(
        source_filename=p.name,
        sha256=sha256,
        survey_statement=survey_statement,
        column_header_line=column_header_line,
        column_names=column_names,
        line_ending_convention=line_ending,
    )


def read_checkshot_rows(path: str) -> CheckshotStationData:
    """
    Parse the data rows (all non-empty lines after the 2-line header) into
    raw `CheckshotStationData`. Every row must have exactly 3 tab-
    separated numeric tokens; a blank line is permitted only as a trailing
    end-of-file artifact (all 3 approved files end this way) - a blank
    line found BEFORE the last non-empty line is a structural defect.
    """
    text = Path(path).read_text(encoding="utf-8")
    lines = text.splitlines()
    data_lines = lines[2:]

    last_nonblank = -1
    for i, ln in enumerate(data_lines):
        if ln.strip():
            last_nonblank = i
    if last_nonblank < 0:
        raise CheckshotParsingError(f"{path}: no data rows found after the 2-line header.")

    depth_vals, tvdss_vals, owt_vals = [], [], []
    for i, ln in enumerate(data_lines[: last_nonblank + 1]):
        if not ln.strip():
            raise CheckshotParsingError(
                f"{path}: blank line found at data row {i + 1} before the last data row; "
                f"only a trailing blank line (end-of-file artifact) is permitted."
            )
        parts = ln.split("\t")
        if len(parts) != _REQUIRED_COLUMN_COUNT:
            raise CheckshotParsingError(
                f"{path}: data row {i + 1} has {len(parts)} tab-separated field(s), "
                f"expected {_REQUIRED_COLUMN_COUNT}: {ln!r}"
            )
        try:
            d, t, o = float(parts[0]), float(parts[1]), float(parts[2])
        except ValueError as exc:
            raise CheckshotParsingError(
                f"{path}: data row {i + 1} contains a non-numeric token: {ln!r} ({exc})"
            ) from exc
        if not (np.isfinite(d) and np.isfinite(t) and np.isfinite(o)):
            raise CheckshotParsingError(
                f"{path}: data row {i + 1} contains a non-finite (NaN/Inf) value: {ln!r}"
            )
        depth_vals.append(d)
        tvdss_vals.append(t)
        owt_vals.append(o)

    return CheckshotStationData(
        Depth_source_m=np.asarray(depth_vals, dtype=np.float64),
        TVDSS_source_m=np.asarray(tvdss_vals, dtype=np.float64),
        OWT_source_s=np.asarray(owt_vals, dtype=np.float64),
    )


# ---------------------------------------------------------------------------
# Contract resolution
# ---------------------------------------------------------------------------
def resolve_checkshot_contract(
    header: CheckshotHeaderInfo,
    stations: CheckshotStationData,
    contract: CheckshotFileContract,
) -> Tuple[CheckshotIngestionIssue, ...]:
    """
    Compare the actual parsed header/data against `contract`, returning
    the complete tuple of issues (ERROR and WARNING). Never raises itself
    - `load_checkshot_file` raises `CheckshotContractError` if any ERROR
    is present. Blocking (ERROR) checks: filename, SHA-256, header text
    (both lines), column order/count, row count, and each numeric range
    beyond `numeric_range_tolerance`. Non-blocking (WARNING) disclosures:
    depth-basis-not-explicitly-declared (always) and well-identity-
    inferred-unverified (only for a contract declaring that status).
    """
    issues: list = []

    def err(code: str, message: str) -> None:
        issues.append(CheckshotIngestionIssue("ERROR", code, message, header.source_filename))

    def warn(code: str, message: str) -> None:
        issues.append(CheckshotIngestionIssue("WARNING", code, message, header.source_filename))

    if header.source_filename != contract.source_filename:
        err(
            "FILENAME_MISMATCH",
            f"Actual filename {header.source_filename!r} does not match contract key "
            f"{contract.source_filename!r}.",
        )
    if header.sha256 != contract.expected_sha256:
        err(
            "SHA256_MISMATCH",
            f"Actual SHA-256 {header.sha256!r} does not match expected "
            f"{contract.expected_sha256!r}; raw file identity could not be confirmed.",
        )
    if header.survey_statement != contract.expected_survey_statement:
        err(
            "SURVEY_STATEMENT_MISMATCH",
            f"Actual survey-statement line {header.survey_statement!r} does not match "
            f"expected {contract.expected_survey_statement!r}.",
        )
    if header.column_header_line != contract.expected_column_header_line:
        err(
            "COLUMN_HEADER_LINE_MISMATCH",
            f"Actual column-header line {header.column_header_line!r} does not match "
            f"expected {contract.expected_column_header_line!r}.",
        )
    if header.column_names != contract.expected_column_order:
        err(
            "COLUMN_ORDER_MISMATCH",
            f"Actual column order {header.column_names!r} does not match expected "
            f"{contract.expected_column_order!r}.",
        )
    if len(header.column_names) != contract.expected_column_count:
        err(
            "COLUMN_COUNT_MISMATCH",
            f"Actual column count {len(header.column_names)} does not match expected "
            f"{contract.expected_column_count}.",
        )
    if contract.expected_vertical_correction_fragment not in header.survey_statement:
        err(
            "VERTICAL_CORRECTION_STATEMENT_MISSING",
            f"Expected vertical-correction statement fragment "
            f"{contract.expected_vertical_correction_fragment!r} not found in survey "
            f"statement {header.survey_statement!r}.",
        )
    if contract.expected_srd_reference_fragment not in header.survey_statement:
        err(
            "SRD_REFERENCE_STATEMENT_MISSING",
            f"Expected SRD-reference statement fragment "
            f"{contract.expected_srd_reference_fragment!r} not found in survey statement "
            f"{header.survey_statement!r}.",
        )

    n_rows = int(stations.Depth_source_m.size)
    if n_rows != contract.expected_row_count:
        err(
            "ROW_COUNT_MISMATCH",
            f"Actual row count {n_rows} does not match expected {contract.expected_row_count}.",
        )

    tol = contract.numeric_range_tolerance
    for label, actual_min, actual_max, exp_min, exp_max in (
        ("Depth_source_m", float(stations.Depth_source_m.min()), float(stations.Depth_source_m.max()),
         contract.expected_depth_min_m, contract.expected_depth_max_m),
        ("TVDSS_source_m", float(stations.TVDSS_source_m.min()), float(stations.TVDSS_source_m.max()),
         contract.expected_tvdss_min_m, contract.expected_tvdss_max_m),
        ("OWT_source_s", float(stations.OWT_source_s.min()), float(stations.OWT_source_s.max()),
         contract.expected_owt_min_s, contract.expected_owt_max_s),
    ):
        if abs(actual_min - exp_min) > tol or abs(actual_max - exp_max) > tol:
            err(
                "NUMERIC_RANGE_MISMATCH",
                f"{label}: actual range [{actual_min}, {actual_max}] does not match expected "
                f"[{exp_min}, {exp_max}] within tolerance {tol}.",
            )

    # Non-blocking disclosures.
    warn(
        DEPTH_BASIS_NOT_EXPLICITLY_DECLARED_CODE,
        "This file's header labels its first column only 'Depth', never explicitly 'MD'. "
        "'Depth_source_m' is preserved under that literal name (never silently renamed to "
        "MD_m); whether it behaves as measured depth is evaluated empirically against the "
        "locked deviation-survey MD->TVDSS relationship for this well "
        "(see p2mem.time_depth.compare_checkshot_to_survey / "
        "checkshot_depth_tie_qc.csv), not assumed from the header.",
    )
    if contract.well_identity_evidence_status == "inferred_unverified":
        warn(
            WELL_IDENTITY_INFERRED_UNVERIFIED_CODE,
            f"This file's association with project well {contract.project_well_key!r} is "
            f"NOT independently proven by file content (no embedded well identifier); it "
            f"is inferred from filename, project context, and depth-tie plausibility only. "
            f"{contract.well_identity_evidence_notes} model_use_status="
            f"{contract.model_use_status!r} - never treated as verified, and never "
            f"transferred into another well's primary time-depth model.",
        )

    return tuple(issues)


# ---------------------------------------------------------------------------
# High-level load functions
# ---------------------------------------------------------------------------
def load_checkshot_file(
    path: str,
    contract: CheckshotFileContract,
    *,
    survey_md_m: "np.ndarray | None" = None,
    survey_tvd_m: "np.ndarray | None" = None,
    datum_elevation_m: "float | None" = None,
) -> CheckshotWellResult:
    """
    Parse, contract-resolve, condition, and diagnose one checkshot file,
    returning a complete `CheckshotWellResult`.

    If `survey_md_m`/`survey_tvd_m`/`datum_elevation_m` are supplied (the
    locked `petrel_source_trace` survey for the SAME well, per
    `config/deviation_survey_contracts.yml`), the checkshot-vs-survey
    depth-reference comparison is computed and attached as
    `.depth_comparison`; otherwise it is `None` (no survey available for
    this well - never fabricated).

    Also builds (Increment 4.1) BOTH order-invariant axis-conditioned
    lookup tables (`.axis_tables["tvdss_to_owt"]`,
    `.axis_tables["owt_to_tvdss"]`) and the combined axis-tie register
    (`.axis_tie_entries`) - see `p2mem.time_depth
    .build_axis_conditioned_tables_for_well`. TVDSS and OWT are NOT
    guaranteed strictly increasing after Depth-tie conditioning alone (a
    real, distinct Depth value can carry an equal TVDSS or OWT to its
    neighbor); this call raises `TimeDepthError` only for a genuine
    reversal (never observed in any of the three approved real files),
    not for an ordinary tie.

    Raises `CheckshotFileNotFoundError`, `CheckshotParsingError`,
    `CheckshotContractError`, or (only for a genuine axis reversal after
    Depth-tie conditioning, not for an ordinary tie) `TimeDepthError`
    (never returns a partially valid result).

    "Genuine axis reversal" here (Increment 4.1.1 clarification) means any
    strictly negative step in the ORIGINAL, ungrouped axis sequence,
    including a reversal that returns to an already-seen value (e.g.
    `[100.0, 200.0, 100.0]`) - not only a reversal that survives global
    exact-value grouping. See `p2mem.time_depth
    .build_axis_conditioned_lookup_table` for the full corrected
    detection logic and `INCREMENT_04_1_1_MANIFEST.md` for the audit
    finding this corrects.
    """
    header = parse_checkshot_header(path)
    stations = read_checkshot_rows(path)
    issues = resolve_checkshot_contract(header, stations, contract)

    error_issues = tuple(i for i in issues if i.severity == "ERROR")
    if error_issues:
        raise CheckshotContractError(
            f"{path}: {len(error_issues)} contract-resolution ERROR(s): "
            + "; ".join(f"[{i.code}] {i.message}" for i in error_issues),
            issues,
        )

    ties, conditioned = detect_and_condition_depth_ties(
        contract.project_well_key,
        stations.Depth_source_m,
        stations.TVDSS_source_m,
        stations.OWT_source_s,
    )
    velocity = compute_velocity_diagnostics(contract.project_well_key, conditioned)

    # Increment 4.1: build BOTH order-invariant axis-conditioned lookup
    # tables (tvdss_to_owt, owt_to_tvdss) once per well here, so every
    # downstream consumer (interpolation calls, the CSV/JSON exporters,
    # the notebook) shares the same tables and tie register rather than
    # each silently rebuilding its own - see p2mem.time_depth module
    # docstring, "Order-invariant axis-tie conditioning".
    axis_tie_entries, axis_tables = build_axis_conditioned_tables_for_well(
        contract.project_well_key, conditioned
    )

    depth_comparison = None
    if survey_md_m is not None and survey_tvd_m is not None and datum_elevation_m is not None:
        depth_comparison = compare_checkshot_to_survey(
            contract.project_well_key,
            stations.Depth_source_m,
            stations.TVDSS_source_m,
            survey_md_m,
            survey_tvd_m,
            datum_elevation_m,
            depth_basis_interpretation_status=contract.depth_basis_interpretation_status,
        )

    return CheckshotWellResult(
        header=header,
        contract=contract,
        raw=stations,
        duplicate_ties=ties,
        conditioned=conditioned,
        velocity=velocity,
        depth_comparison=depth_comparison,
        issues=issues,
        contract_status="PASSED",
        axis_tie_entries=axis_tie_entries,
        axis_tables=axis_tables,
    )


def load_checkshot_surveys(
    file_paths: Dict[str, str],
    contracts: Dict[str, CheckshotFileContract],
    *,
    survey_trajectories: "Dict[str, tuple] | None" = None,
) -> Tuple[Dict[str, CheckshotWellResult], Dict[str, CheckshotIngestionFailure]]:
    """
    Load a batch of checkshot files keyed by well key (e.g. "Poseidon_2"),
    isolating expected per-well ingestion failures as typed
    `CheckshotIngestionFailure` records. One well's failure never stops
    the others from loading - this now explicitly includes a
    `TimeDepthError` raised anywhere inside `load_checkshot_file` (Depth-
    tie conditioning, axis-tie conditioning, or checkshot-vs-survey
    comparison), which is caught per well and recorded with
    `error_type="numerical_conditioning_failure"` (Increment 4.1.1 fix -
    the Increment 4.1 implementation caught only file/parsing/contract
    errors, so a numerical defect in one well's data could previously
    stop the entire batch). Never caught with a blanket `except
    Exception` - only the specific typed exceptions this module and
    `p2mem.time_depth` are documented to raise.

    `contracts` is keyed by SOURCE FILENAME (as in
    `config/checkshot_contracts.yml`); `file_paths` is keyed by WELL KEY.
    `survey_trajectories`, if given, maps well key ->
    (survey_md_m, survey_tvd_m, datum_elevation_m) for wells that have a
    locked deviation survey available for the depth-reference comparison.
    """
    results: Dict[str, CheckshotWellResult] = {}
    failures: Dict[str, CheckshotIngestionFailure] = {}
    survey_trajectories = survey_trajectories or {}

    for well_key, path in file_paths.items():
        basename = Path(path).name
        contract = contracts.get(basename)
        if contract is None:
            raise CheckshotContractDefinitionError(
                f"No checkshot contract found for file {basename!r} (well key {well_key!r}); "
                f"contracts are keyed by source filename and must be authored before ingestion."
            )
        survey_md_m = survey_tvd_m = datum_elevation_m = None
        if well_key in survey_trajectories:
            survey_md_m, survey_tvd_m, datum_elevation_m = survey_trajectories[well_key]
        try:
            results[well_key] = load_checkshot_file(
                path, contract,
                survey_md_m=survey_md_m, survey_tvd_m=survey_tvd_m, datum_elevation_m=datum_elevation_m,
            )
        except CheckshotFileNotFoundError as exc:
            failures[well_key] = CheckshotIngestionFailure(
                well_key=well_key, source_path=path, error_type="file_not_found",
                message=str(exc), exception=exc,
            )
        except CheckshotParsingError as exc:
            failures[well_key] = CheckshotIngestionFailure(
                well_key=well_key, source_path=path, error_type="parsing_failure",
                message=str(exc), exception=exc,
            )
        except CheckshotContractError as exc:
            failures[well_key] = CheckshotIngestionFailure(
                well_key=well_key, source_path=path, error_type="contract_failure",
                message=str(exc), exception=exc,
            )
        except TimeDepthError as exc:
            # Increment 4.1.1 fix (Blocking Defect 4): a numerical/
            # structural conditioning failure (Depth-tie conditioning,
            # axis-tie conditioning, checkshot-vs-survey comparison, or
            # any other TimeDepthError-raising validation inside
            # load_checkshot_file) must isolate exactly like every other
            # expected per-well failure above - it must NOT be allowed to
            # propagate out of this batch loader and stop every other
            # well from loading. TimeDepthError messages are built from
            # well_key and numeric values only (never a file path), so
            # str(exc) here carries no absolute-path leakage risk.
            failures[well_key] = CheckshotIngestionFailure(
                well_key=well_key, source_path=path, error_type="numerical_conditioning_failure",
                message=str(exc), exception=exc,
            )

    return results, failures


#### `p2mem/io/checkshot_inventory.py` — deterministic inventory-table builders

In [ ]:
%%writefile p2mem/io/checkshot_inventory.py
"""
p2mem.io.checkshot_inventory - Deterministic, metadata-only inventory-table
builders for the Increment 4 checkshot / time-depth layer.

Mirrors the design of `p2mem.io.deviation_inventory` (Increment 3, LOCKED):
every function here returns a list of plain dicts (one per output row),
ready for `csv.DictWriter`, or a single JSON-serializable manifest dict -
never a full per-sample checkshot or LAS array (see "Package deterministic
summaries and QC metadata only" in the Increment 4 specification). Every
row uses only a file's BASENAME for any path-shaped field, and every
numeric field is a plain Python float/int (never a NumPy scalar) so the
output is stable JSON/CSV regardless of environment - this is the same
environment-independence discipline established in Increment 3.1's
`_sanitize_message` correction.
"""

from __future__ import annotations

from pathlib import Path
from typing import Dict, List, Tuple

from p2mem.checkshot_models import (
    CheckshotAvailabilityRecord,
    CheckshotIngestionFailure,
    CheckshotWellResult,
    SonicCheckshotDriftResult,
    TimeDepthMappingSummary,
)

__all__ = [
    "build_checkshot_file_inventory_rows",
    "build_checkshot_issues_rows",
    "build_duplicate_tie_register_rows",
    "build_checkshot_time_axis_tie_register_rows",
    "build_checkshot_depth_tie_qc_rows",
    "build_checkshot_velocity_summary_rows",
    "build_sonic_checkshot_drift_rows",
    "build_time_depth_mapping_rows",
    "build_checkshot_time_depth_manifest",
]


def _sanitize_message(message: str, source_path: str) -> str:
    """Replace a literal full path inside `message` with its basename
    (mirrors `p2mem.io.deviation_inventory._sanitize_message`)."""
    basename = Path(source_path).name
    return message.replace(source_path, basename)


def build_checkshot_file_inventory_rows(
    results: Dict[str, CheckshotWellResult],
    failures: Dict[str, CheckshotIngestionFailure],
    availability: Dict[str, CheckshotAvailabilityRecord] = None,
) -> List[dict]:
    availability = availability or {}
    rows: List[dict] = []
    for well_key in sorted(set(results) | set(failures) | set(availability)):
        if well_key in results:
            r = results[well_key]
            n_errors = sum(1 for i in r.issues if i.severity == "ERROR")
            n_warnings = sum(1 for i in r.issues if i.severity == "WARNING")
            rows.append(
                {
                    "well_key": well_key,
                    "source_filename": r.header.source_filename,
                    "sha256": r.header.sha256,
                    "project_well_key": r.contract.project_well_key,
                    "well_identity_evidence_status": r.contract.well_identity_evidence_status,
                    "model_use_status": r.contract.model_use_status,
                    "n_raw_rows": int(r.raw.Depth_source_m.size),
                    "n_conditioned_rows": r.conditioned.n_conditioned_rows,
                    "n_tie_groups": r.conditioned.n_tie_groups,
                    "depth_min_m": float(r.raw.Depth_source_m.min()),
                    "depth_max_m": float(r.raw.Depth_source_m.max()),
                    "tvdss_min_m": float(r.raw.TVDSS_source_m.min()),
                    "tvdss_max_m": float(r.raw.TVDSS_source_m.max()),
                    "owt_min_s": float(r.raw.OWT_source_s.min()),
                    "owt_max_s": float(r.raw.OWT_source_s.max()),
                    "contract_status": r.contract_status,
                    "n_errors": n_errors,
                    "n_warnings": n_warnings,
                    "error_type": "",
                    "error_message": "",
                }
            )
        elif well_key in failures:
            f = failures[well_key]
            rows.append(
                {
                    "well_key": well_key,
                    "source_filename": Path(f.source_path).name,
                    "sha256": "",
                    "project_well_key": "",
                    "well_identity_evidence_status": "",
                    "model_use_status": "",
                    "n_raw_rows": "",
                    "n_conditioned_rows": "",
                    "n_tie_groups": "",
                    "depth_min_m": "",
                    "depth_max_m": "",
                    "tvdss_min_m": "",
                    "tvdss_max_m": "",
                    "owt_min_s": "",
                    "owt_max_s": "",
                    "contract_status": "FAILED",
                    "n_errors": "",
                    "n_warnings": "",
                    "error_type": f.error_type,
                    "error_message": _sanitize_message(f.message, f.source_path),
                }
            )
        else:
            a = availability[well_key]
            rows.append(
                {
                    "well_key": well_key,
                    "source_filename": "",
                    "sha256": "",
                    "project_well_key": well_key,
                    "well_identity_evidence_status": "",
                    "model_use_status": "",
                    "n_raw_rows": "",
                    "n_conditioned_rows": "",
                    "n_tie_groups": "",
                    "depth_min_m": "",
                    "depth_max_m": "",
                    "tvdss_min_m": "",
                    "tvdss_max_m": "",
                    "owt_min_s": "",
                    "owt_max_s": "",
                    "contract_status": a.checkshot_availability,
                    "n_errors": "",
                    "n_warnings": "",
                    "error_type": "",
                    "error_message": a.notes,
                }
            )
    return rows


def build_checkshot_issues_rows(
    results: Dict[str, CheckshotWellResult], failures: Dict[str, CheckshotIngestionFailure]
) -> List[dict]:
    rows: List[dict] = []
    for well_key in sorted(results):
        r = results[well_key]
        for issue in r.issues:
            rows.append(
                {
                    "well_key": well_key,
                    "source_filename": r.header.source_filename,
                    "severity": issue.severity,
                    "code": issue.code,
                    "message": issue.message,
                    "context": issue.context,
                }
            )
    for well_key in sorted(failures):
        f = failures[well_key]
        rows.append(
            {
                "well_key": well_key,
                "source_filename": Path(f.source_path).name,
                "severity": "ERROR",
                "code": f.error_type.upper(),
                "message": _sanitize_message(f.message, f.source_path),
                "context": Path(f.source_path).name,
            }
        )
    return rows


def build_duplicate_tie_register_rows(results: Dict[str, CheckshotWellResult]) -> List[dict]:
    rows: List[dict] = []
    for well_key in sorted(results):
        r = results[well_key]
        for tie in r.duplicate_ties:
            rows.append(
                {
                    "well_key": tie.well_key,
                    "axis": tie.axis,
                    "tie_value_m": tie.tie_value_m,
                    "source_row_indices": ";".join(str(i) for i in tie.source_row_indices),
                    "original_tvdss_m": ";".join(str(v) for v in tie.original_tvdss_m),
                    "original_owt_s": ";".join(str(v) for v in tie.original_owt_s),
                    "duplicate_type": tie.duplicate_type,
                    "group_size": tie.group_size,
                    "tvdss_value_spread_m": tie.tvdss_value_spread_m,
                    "owt_value_spread_s": tie.owt_value_spread_s,
                    "selected_representative_tvdss_m": tie.selected_representative_tvdss_m,
                    "selected_representative_owt_s": tie.selected_representative_owt_s,
                    "conditioning_rule": tie.conditioning_rule,
                    "affected_downstream_outputs": ";".join(tie.affected_downstream_outputs),
                }
            )
    return rows


def build_checkshot_time_axis_tie_register_rows(results: Dict[str, CheckshotWellResult]) -> List[dict]:
    """
    One row per `AxisTimeDepthTieRegisterEntry` (Increment 4.1) - the
    order-invariant TVDSS/OWT-axis tie register, separate from (and never
    confused with) `build_duplicate_tie_register_rows`'s Depth-axis
    register. `conditioned_row_indices` index into that well's
    Depth-conditioned arrays (`ConditionedCheckshotData`), NOT the raw
    arrays - see `checkshot_duplicate_tie_register.csv` for the raw-row
    (Depth-axis) register.
    """
    rows: List[dict] = []
    for well_key in sorted(results):
        r = results[well_key]
        for tie in r.axis_tie_entries:
            rows.append(
                {
                    "well_key": tie.well_key,
                    "interpolation_direction": tie.interpolation_direction,
                    "axis": tie.axis,
                    "dependent_axis": tie.dependent_axis,
                    "tie_axis_value": tie.tie_axis_value,
                    "conditioned_row_indices": ";".join(str(i) for i in tie.conditioned_row_indices),
                    "associated_depth_m": ";".join(str(v) for v in tie.associated_depth_m),
                    "original_dependent_values": ";".join(str(v) for v in tie.original_dependent_values),
                    "group_size": tie.group_size,
                    "dependent_value_spread": tie.dependent_value_spread,
                    "selected_representative_dependent_value": tie.selected_representative_dependent_value,
                    "conditioning_rule": tie.conditioning_rule,
                    "tie_kind": tie.tie_kind,
                    "affected_downstream_outputs": ";".join(tie.affected_downstream_outputs),
                }
            )
    return rows


def build_checkshot_depth_tie_qc_rows(results: Dict[str, CheckshotWellResult]) -> List[dict]:
    rows: List[dict] = []
    for well_key in sorted(results):
        c = results[well_key].depth_comparison
        if c is None:
            continue
        rows.append(
            {
                "well_key": c.well_key,
                "n_compared": c.n_compared,
                "n_outside_survey_md_coverage": c.n_outside_survey_md_coverage,
                "min_residual_m": c.min_residual_m,
                "max_residual_m": c.max_residual_m,
                "max_abs_residual_m": c.max_abs_residual_m,
                "mean_residual_m": c.mean_residual_m,
                "median_residual_m": c.median_residual_m,
                "rmse_m": c.rmse_m,
                "first_residual_m": c.first_residual_m,
                "last_residual_m": c.last_residual_m,
                "residual_trend_description": c.residual_trend_description,
                "depth_basis_interpretation_status": c.depth_basis_interpretation_status,
                "residual_sign_convention": c.residual_sign_convention,
            }
        )
    return rows


def build_checkshot_velocity_summary_rows(results: Dict[str, CheckshotWellResult]) -> List[dict]:
    rows: List[dict] = []
    for well_key in sorted(results):
        v = results[well_key].velocity
        import numpy as np  # local import: only used for this summary reduction

        vavg_finite = v.Vavg_m_s[np.isfinite(v.Vavg_m_s)]
        vint_valid = v.Vint_m_s[~v.vint_invalid_mask] if v.n_vint_intervals else np.array([])
        rows.append(
            {
                "well_key": v.well_key,
                "n_conditioned_rows": int(v.Depth_conditioned_m.size),
                "vavg_min_m_s": float(vavg_finite.min()) if vavg_finite.size else "",
                "vavg_max_m_s": float(vavg_finite.max()) if vavg_finite.size else "",
                "vavg_mean_m_s": float(vavg_finite.mean()) if vavg_finite.size else "",
                "n_vint_intervals": v.n_vint_intervals,
                "n_vint_invalid": v.n_vint_invalid,
                "vint_valid_min_m_s": float(vint_valid.min()) if vint_valid.size else "",
                "vint_valid_max_m_s": float(vint_valid.max()) if vint_valid.size else "",
                "vint_valid_mean_m_s": float(vint_valid.mean()) if vint_valid.size else "",
                "non_positive_delta_owt_count": v.vint_invalid_reason_counts.get(
                    "non_positive_delta_owt", 0
                ),
                "non_positive_delta_tvdss_count": v.vint_invalid_reason_counts.get(
                    "non_positive_delta_tvdss", 0
                ),
            }
        )
    return rows


def build_sonic_checkshot_drift_rows(
    drift_results: Dict[str, SonicCheckshotDriftResult]
) -> List[dict]:
    rows: List[dict] = []
    for well_key in sorted(drift_results):
        d = drift_results[well_key]
        rows.append(
            {
                "well_key": d.well_key,
                "md_interval_start_m": d.md_interval_start_m,
                "md_interval_end_m": d.md_interval_end_m,
                "n_sonic_samples": d.n_sonic_samples,
                "selection_criteria": d.selection_criteria,
                "integration_method": d.integration_method,
                "sonic_transit_time_s": d.sonic_transit_time_s,
                "checkshot_owt_increment_s": d.checkshot_owt_increment_s,
                "sonic_minus_checkshot_ms": d.sonic_minus_checkshot_ms,
                "checkshot_minus_sonic_ms": d.checkshot_minus_sonic_ms,
                "sonic_minus_checkshot_percent": d.sonic_minus_checkshot_percent,
                "limitations": " | ".join(d.limitations),
            }
        )
    return rows


def build_time_depth_mapping_rows(mappings: Dict[str, TimeDepthMappingSummary]) -> List[dict]:
    rows: List[dict] = []
    for well_key in sorted(mappings):
        m = mappings[well_key]
        rows.append(
            {
                "well_key": m.well_key,
                "n_las_samples": m.n_las_samples,
                "n_inside_coverage": m.n_inside_coverage,
                "n_shallower_than_coverage": m.n_shallower_than_coverage,
                "n_deeper_than_coverage": m.n_deeper_than_coverage,
                "mapped_fraction": m.mapped_fraction,
                "checkshot_depth_min_m": m.checkshot_depth_min_m,
                "checkshot_depth_max_m": m.checkshot_depth_max_m,
                "las_md_min_m": m.las_md_min_m,
                "las_md_max_m": m.las_md_max_m,
                "interpolation_method": m.interpolation_method,
                "n_extrapolated": m.n_extrapolated,
            }
        )
    return rows


def _axis_tie_conditioning_summary(r: CheckshotWellResult) -> dict:
    """
    Per-well summary (Increment 4.1) of the order-invariant axis-tie
    conditioning applied to each inversion direction's lookup table -
    NEVER confused with `n_tie_groups` above, which counts Depth-axis
    (raw-row) ties only. `axis_tie_conditioning_policy` names the
    deterministic representative rule actually used (median, per group,
    order-invariant) so a reader never has to infer it from the counts
    alone.
    """
    to_owt = r.axis_tables.get("tvdss_to_owt") if r.axis_tables else None
    to_tvdss = r.axis_tables.get("owt_to_tvdss") if r.axis_tables else None
    return {
        "axis_tie_conditioning_policy": (
            "order-invariant: ties on the axis being inverted are grouped by exact value "
            "(regardless of which tied row was parsed first), and each group's dependent-value "
            "MEDIAN becomes the group's single conditioned representative; a genuine reversal "
            "(not a tie) raises TimeDepthError rather than being sorted or forced monotonic."
        ),
        "n_tvdss_axis_tie_groups": to_owt.n_axis_tie_groups if to_owt is not None else 0,
        "n_tvdss_axis_collapsed_points": to_owt.n_collapsed_points if to_owt is not None else 0,
        "n_tvdss_axis_identical_pairs": to_owt.n_identical_pairs if to_owt is not None else 0,
        "n_tvdss_axis_genuinely_nonunique_groups": (
            to_owt.n_genuinely_nonunique_groups if to_owt is not None else 0
        ),
        "n_owt_axis_tie_groups": to_tvdss.n_axis_tie_groups if to_tvdss is not None else 0,
        "n_owt_axis_collapsed_points": to_tvdss.n_collapsed_points if to_tvdss is not None else 0,
        "n_owt_axis_identical_pairs": to_tvdss.n_identical_pairs if to_tvdss is not None else 0,
        "n_owt_axis_genuinely_nonunique_groups": (
            to_tvdss.n_genuinely_nonunique_groups if to_tvdss is not None else 0
        ),
    }


def build_checkshot_time_depth_manifest(
    results: Dict[str, CheckshotWellResult],
    failures: Dict[str, CheckshotIngestionFailure],
    availability: Dict[str, CheckshotAvailabilityRecord],
    drift_results: Dict[str, SonicCheckshotDriftResult],
    mappings: Dict[str, TimeDepthMappingSummary],
) -> dict:
    wells = {}
    for well_key in sorted(results):
        r = results[well_key]
        wells[well_key] = {
            "checkshot_availability": "AVAILABLE",
            "source_filename": r.header.source_filename,
            "sha256": r.header.sha256,
            "project_well_key": r.contract.project_well_key,
            "well_identity_evidence_status": r.contract.well_identity_evidence_status,
            "model_use_status": r.contract.model_use_status,
            "n_raw_rows": int(r.raw.Depth_source_m.size),
            "n_conditioned_rows": r.conditioned.n_conditioned_rows,
            "n_tie_groups": r.conditioned.n_tie_groups,
            "contract_status": r.contract_status,
            "n_issue_errors": sum(1 for i in r.issues if i.severity == "ERROR"),
            "n_issue_warnings": sum(1 for i in r.issues if i.severity == "WARNING"),
            "depth_comparison": (
                {
                    "n_compared": r.depth_comparison.n_compared,
                    "max_abs_residual_m": r.depth_comparison.max_abs_residual_m,
                    "mean_residual_m": r.depth_comparison.mean_residual_m,
                    "residual_trend_description": r.depth_comparison.residual_trend_description,
                    "residual_sign_convention": r.depth_comparison.residual_sign_convention,
                }
                if r.depth_comparison is not None
                else None
            ),
            "axis_tie_conditioning": _axis_tie_conditioning_summary(r),
        }
    for well_key in sorted(availability):
        a = availability[well_key]
        wells[well_key] = {
            "checkshot_availability": a.checkshot_availability,
            "notes": a.notes,
        }

    failed = {
        well_key: {"error_type": f.error_type, "message": _sanitize_message(f.message, f.source_path)}
        for well_key, f in failures.items()
    }

    return {
        "increment": "4.1",
        "corrective_patch_of": "4",
        "tier_classification": "Tier C - Screening-Level / Uncalibrated Educational",
        "scope": (
            "Checkshot ingestion, per-file contracts, raw QC/provenance, checkshot-to-deviation "
            "depth-reference comparison, duplicate-tie conditioning, OWT/TWT handling, "
            "average/interval velocity diagnostics, Poseidon 2 sonic-checkshot drift diagnostic, "
            "and forward/inverse time-depth interpolation within validated checkshot coverage "
            "only. Increment 4.1 is a narrowly scoped corrective patch: it replaces the order-"
            "dependent 'keep first, drop later' resolution of a repeated TVDSS/OWT value in the "
            "Depth-conditioned table with an explicit, order-invariant, median-based, fully "
            "registered axis-tie conditioning policy, and adds numerical-validation hardening to "
            "the trapezoidal integrator and the sonic-checkshot drift diagnostic. No formation-"
            "top correction, lithology interpretation, density modelling, pore-pressure "
            "prediction, elastic properties, rock strength, stress modelling, or wellbore-"
            "stability analysis is performed in this increment."
        ),
        "n_wells_checkshot_available": len(results),
        "n_wells_checkshot_not_available": len(availability),
        "n_wells_failed": len(failures),
        "wells": wells,
        "failed_wells": failed,
        "sonic_checkshot_drift": (
            {
                well_key: {
                    "md_interval_m": [d.md_interval_start_m, d.md_interval_end_m],
                    "sonic_minus_checkshot_ms": d.sonic_minus_checkshot_ms,
                    "sonic_minus_checkshot_percent": d.sonic_minus_checkshot_percent,
                }
                for well_key, d in drift_results.items()
            }
        ),
        "time_depth_mapping": (
            {
                well_key: {
                    "mapped_fraction": m.mapped_fraction,
                    "n_extrapolated": m.n_extrapolated,
                    "checkshot_depth_range_m": [m.checkshot_depth_min_m, m.checkshot_depth_max_m],
                }
                for well_key, m in mappings.items()
            }
        ),
    }


#### Step 7 — Write the per-file checkshot contracts (`config/checkshot_contracts.yml`)

**Technical objective:** the human-authored, human-reviewable per-file contract for each of the three approved checkshot files, with every `expected_*` value independently recomputed from the actual files (see `INCREMENT_04_MANIFEST.md`).

In [ ]:
%%writefile config/checkshot_contracts.yml
# config/checkshot_contracts.yml
#
# Increment 4 human-authored, human-reviewable per-file checkshot (velocity
# survey) contract for each of the three APPROVED checkshot files
# (Poseidon2-Checkshot.txt, Boreas1-Checkshot.txt, Proteus1-Checkshot.txt -
# note the exact, literal hyphenated filenames: this project's approved
# Drive filenames, confirmed against the user-supplied SHA-256 values,
# regardless of any different local/sandbox spelling a given execution
# environment's uploaded copy may carry - see INCREMENT_04_MANIFEST.md,
# "Filename-provenance discipline", which mirrors the identical Increment
# 3.1 space-vs-underscore lesson).
#
# Every expected_* numeric field below was independently recomputed from
# the actual three files (never assumed, never copied from prose) - see
# INCREMENT_04_MANIFEST.md Section "Independently recomputed checkshot
# statistics" for the exact recomputation. A mismatch in any expected_*
# field is a blocking ERROR at load time (p2mem.io.checkshot.
# resolve_checkshot_contract) - ingestion never silently proceeds with
# "close enough" values.
#
# NOT APPROVED / MUST NOT BE USED (recorded here for traceability only -
# no contract entry exists for these, and none is created for them):
# Poseidon1-Checkshot.txt, Kronos1-Checkshot.txt, Pharos1-Checkshot.txt.
#
# Poseidon North 1 has NO approved checkshot file at all. This is recorded
# as a factual data gap (checkshot_availability: NOT_AVAILABLE) elsewhere
# (see p2mem.checkshot_models.CheckshotAvailabilityRecord and the
# Increment 4 integration script/notebook) - never as a missing/failed
# contract entry here, and never filled by substituting another well's
# file.
#
# Well-identity evidence and model-use status (Increment 4 vocabulary -
# see p2mem.checkshot_models module docstring):
#   - Poseidon2-Checkshot.txt / Boreas1-Checkshot.txt: each filename
#     unambiguously matches the sole candidate well of that name in this
#     project (there is only one "Poseidon 2" and one "Boreas 1"); status
#     "verified". Poseidon 2 is the ONLY checkshot approved to define a
#     primary time-depth relationship (model_use_status: primary_model);
#     Boreas 1 is supporting QC data only (model_use_status: qc_only) and
#     must never be transferred into Poseidon 2 as a substitute time-depth
#     model.
#   - Proteus1-Checkshot.txt: the project's approved well is "Proteus
#     1ST2" (a sidetrack), but this file's own name and content carry no
#     "ST2"/sidetrack identifier and no embedded well name at all -
#     association with Proteus 1ST2 rests on filename/project-context/
#     depth-tie plausibility only, not independent proof; status
#     "inferred_unverified", model_use_status: qc_only. Never described as
#     verified anywhere downstream.
#
# Depth-basis interpretation (DEPTH_BASIS_NOT_EXPLICITLY_DECLARED):
# every file's first column is labelled only "Depth" (never "MD"). It is
# treated as a CANDIDATE measured-depth axis and evaluated empirically
# against each well's own locked Increment 3/3.1.1 `petrel_source_trace`
# survey MD->TVD relationship (see checkshot_depth_tie_qc.csv); it is
# never silently renamed to MD_m or assumed correct without that check.
#
# Duplicate-tie / interpolation / extrapolation policy (identical across
# all three files - see p2mem.time_depth module docstring for full
# rationale): raw duplicate rows are always preserved exactly; the
# conditioned representative for a tied Depth group is that group's
# per-column MEDIAN (disclosed as equal to the arithmetic mean for every
# group size 2, the only group size observed in this project); every tie
# is individually registered in checkshot_duplicate_tie_register.csv.
# Interpolation is piecewise-linear only (no spline). Extrapolation beyond
# a file's own validated Depth coverage is disallowed everywhere (NaN,
# never a fabricated value).
#
# Numeric-range tolerance (0.0001, identical across all three files): the
# expected min/max values below were read directly from the source file's
# own numeric text (4 decimal places for OWT, 1 for Depth/TVDSS); this
# tolerance absorbs ordinary floating-point round-trip noise without
# masking a genuinely different file.
files:
  "Poseidon2-Checkshot.txt":
    expected_sha256: "69343c84a16180e6146241f3b02acc44b72b33faeff2a66333d518766cc8759d"
    project_well_key: "Poseidon_2"
    well_identity_evidence_status: "verified"
    well_identity_evidence_notes: >
      Filename unambiguously matches the sole "Poseidon 2" well in this
      project; SHA-256 independently confirmed against the user-supplied
      value. No embedded well identifier exists in the file's own content
      (identical minimal header format across all three approved files),
      but no naming/sidetrack ambiguity exists for this well either.
    model_use_status: "primary_model"
    expected_survey_statement: "VELOCITY SURVEY:(Schlumberger) OWT, vertically corrected, relative to SRD (MSL)"
    expected_column_header_line: "Depth\tTVDSS\tOWT(sec)"
    expected_column_order: [Depth, TVDSS, "OWT(sec)"]
    expected_time_type: "OWT"
    expected_time_unit: "s"
    expected_vertical_correction_fragment: "vertically corrected"
    expected_srd_reference_fragment: "relative to SRD (MSL)"
    expected_row_count: 220
    expected_depth_min_m: 1267.8
    expected_depth_max_m: 4700.0
    expected_tvdss_min_m: 1246.0
    expected_tvdss_max_m: 4676.8
    expected_owt_min_s: 0.6802
    expected_owt_max_s: 1.5469
    numeric_range_tolerance: 0.0001
    depth_basis_interpretation_status: "candidate_md_evaluated_against_locked_survey"
    duplicate_tie_policy: "raw_rows_preserved_median_representative_all_ties_registered"
    interpolation_policy: "piecewise_linear_no_extrapolation"
    extrapolation_policy: "disallowed_nan_outside_coverage"
    notes: >
      Only checkshot approved to define Poseidon 2's primary time-depth
      relationship. Raw QC (independently recomputed, not assumed): 5
      repeated Depth values (10 rows, all group size 2), 6 non-increasing
      TVDSS steps, 4 non-increasing OWT steps, one 257.0 m acquisition gap
      between Depth=1313.1 m and Depth=1570.1 m. Checkshot Depth coverage
      (1267.8-4700.0 m) is a PARTIAL subset of this well's LAS MD interval
      - LAS samples outside this range are never extrapolated (see
      time_depth_mapping_summary.csv). Depth-vs-locked-survey comparison
      recomputed max abs residual reported in checkshot_depth_tie_qc.csv
      and INCREMENT_04_MANIFEST.md (not hardcoded here).

  "Boreas1-Checkshot.txt":
    expected_sha256: "dd7c03116277e3cbbe5bcbb31567127f352bc5d12b04d74a9c4a428a765bc44f"
    project_well_key: "Boreas_1"
    well_identity_evidence_status: "verified"
    well_identity_evidence_notes: >
      Filename unambiguously matches the sole "Boreas 1" well in this
      project; SHA-256 independently confirmed against the user-supplied
      value. No embedded well identifier exists in the file's own content
      (identical minimal header format across all three approved files),
      but no naming/sidetrack ambiguity exists for this well either.
    model_use_status: "qc_only"
    expected_survey_statement: "VELOCITY SURVEY:(Schlumberger) OWT, vertically corrected, relative to SRD (MSL)"
    expected_column_header_line: "Depth\tTVDSS\tOWT(sec)"
    expected_column_order: [Depth, TVDSS, "OWT(sec)"]
    expected_time_type: "OWT"
    expected_time_unit: "s"
    expected_vertical_correction_fragment: "vertically corrected"
    expected_srd_reference_fragment: "relative to SRD (MSL)"
    expected_row_count: 212
    expected_depth_min_m: 507.1
    expected_depth_max_m: 5114.0
    expected_tvdss_min_m: 486.0
    expected_tvdss_max_m: 5089.8
    expected_owt_min_s: 0.3201
    expected_owt_max_s: 1.6466
    numeric_range_tolerance: 0.0001
    depth_basis_interpretation_status: "candidate_md_evaluated_against_locked_survey"
    duplicate_tie_policy: "raw_rows_preserved_median_representative_all_ties_registered"
    interpolation_policy: "piecewise_linear_no_extrapolation"
    extrapolation_policy: "disallowed_nan_outside_coverage"
    notes: >
      NEWLY ADMITTED in Increment 4 as SUPPORTING QC DATA ONLY
      (model_use_status: qc_only) - never transferred into Poseidon 2 as a
      substitute time-depth model. Raw QC (independently recomputed, not
      assumed): 3 repeated Depth values (6 rows, all group size 2), 4
      non-increasing TVDSS steps (one of the four is a same-TVDSS,
      distinct-Depth pair - see checkshot_duplicate_tie_register.csv and
      INCREMENT_04_MANIFEST.md), OWT strictly increasing (0 non-increasing
      steps). Depth-vs-locked-survey comparison shows a near-constant
      offset pattern (recomputed value in checkshot_depth_tie_qc.csv) -
      reported as an OBSERVED datum-like offset pattern, not a proven
      datum error; both the checkshot-supplied TVDSS and the locked survey
      TVDSS are preserved and reported side by side, neither silently
      corrected to match the other.

  "Proteus1-Checkshot.txt":
    expected_sha256: "dd2eb37d38a9caea7630db80629a11ee9a8b07bd395c9be637f115d4e607e6f7"
    project_well_key: "Proteus_1ST2"
    well_identity_evidence_status: "inferred_unverified"
    well_identity_evidence_notes: >
      This project's approved well is "Proteus 1ST2" (a sidetrack), but
      this file's own filename and content carry no "ST2"/sidetrack
      identifier and no embedded well name of any kind. Association with
      Proteus 1ST2 is inferred from filename, project/geographic context,
      depth coverage, and numerical depth-tie plausibility against the
      locked Proteus 1ST2 deviation survey ONLY - never independently
      proven by file content, and never described as verified anywhere in
      this project's outputs or notebook.
    model_use_status: "qc_only"
    expected_survey_statement: "VELOCITY SURVEY:(Schlumberger) OWT, vertically corrected, relative to SRD (MSL)"
    expected_column_header_line: "Depth\tTVDSS\tOWT(sec)"
    expected_column_order: [Depth, TVDSS, "OWT(sec)"]
    expected_time_type: "OWT"
    expected_time_unit: "s"
    expected_vertical_correction_fragment: "vertically corrected"
    expected_srd_reference_fragment: "relative to SRD (MSL)"
    expected_row_count: 232
    expected_depth_min_m: 479.6
    expected_depth_max_m: 5238.9
    expected_tvdss_min_m: 457.5
    expected_tvdss_max_m: 5212.3
    expected_owt_min_s: 0.3005
    expected_owt_max_s: 1.6758
    numeric_range_tolerance: 0.0001
    depth_basis_interpretation_status: "candidate_md_evaluated_against_locked_survey"
    duplicate_tie_policy: "raw_rows_preserved_median_representative_all_ties_registered"
    interpolation_policy: "piecewise_linear_no_extrapolation"
    extrapolation_policy: "disallowed_nan_outside_coverage"
    notes: >
      NEWLY ADMITTED in Increment 4 as SUPPORTING QC DATA ONLY
      (model_use_status: qc_only), with UNVERIFIED well-identity evidence
      (see well_identity_evidence_notes) - a visible, non-blocking warning
      (WELL_IDENTITY_INFERRED_UNVERIFIED) is raised on every load. Raw QC
      (independently recomputed, not assumed): strictly increasing in
      Depth, TVDSS, and OWT throughout (0 repeated or non-increasing steps
      in any column) - no duplicate-tie register entries for this file.
      Depth-vs-locked-survey comparison shows a near-constant offset
      pattern (recomputed value in checkshot_depth_tie_qc.csv) - reported
      as an OBSERVED datum-like offset pattern, not a proven datum error;
      both the checkshot-supplied TVDSS and the locked survey TVDSS are
      preserved and reported side by side, neither silently corrected to
      match the other.


#### Step 7a — Write the one CRLF-sensitive fixture as explicit bytes (Increment 4.1.2)

**Technical objective:** write `tests/fixtures/checkshot_valid.txt` — the only packaged fixture anywhere in this project confirmed (by a project-wide byte scan) to use CRLF line endings — via an ordinary Python code cell that reconstructs its exact bytes from a `repr()`-escaped bytes-literal, instead of `%%writefile`.

**Why this is a separate step, not part of Step 7b's `%%writefile` loop:** a real Google Colab fresh-runtime execution of the Increment 4.1.1 notebook reported `tests/fixtures/checkshot_valid.txt` reconstructing with LF line endings instead of its packaged CRLF, causing `tests/test_checkshot.py::test_valid_file_parses_header_and_rows` to fail (`assert 'LF' == 'CRLF'`) — while every other `%%writefile` target in this notebook is LF-only and therefore unaffected. The defect is in how a real Colab kernel's cell-source handling treats embedded line-ending bytes, not in this notebook's own JSON encoding (confirmed byte-for-byte correct on the Linux/Python side — see `INCREMENT_04_1_2_MANIFEST.md` Section 1) — so it cannot be fixed by re-embedding the same bytes through `%%writefile` again. This cell instead stores the file's CRLF bytes only as ASCII backslash-escape sequences inside a Python bytes-literal (`repr()` of the raw bytes), which even Colab's own line-ending normalization has no actual line-break byte to act on; the real ``/`
` bytes exist only from the moment the Python interpreter parses that literal at execution time, and are written with `Path.write_bytes()` (binary mode, no text-mode translation on any platform). This cell, and the fixture it writes, are therefore EXCLUDED from Step 7b's `%%writefile` cell count and from the ordinary `%%writefile`-vs-packaged-file parity audit — they are verified separately, by executing this cell and comparing its actual output bytes to the packaged file (see `dev_scratch_inc4_1/verify_crlf_fixture_and_gate_04.py`, not packaged).

**Failure behavior:** raises `AssertionError` if the written bytes do not exactly match what was just read back, or if any bare (non-CRLF) line feed is found.

In [ ]:
from pathlib import Path

# Increment 4.1.2: tests/fixtures/checkshot_valid.txt is written here as explicit bytes, not via
# %%writefile, because its CRLF line endings were found (by a real Google
# Colab fresh-runtime run) to reconstruct as LF when embedded through
# %%writefile - see build_notebook_04.py's crlf_exact_fixture_cell() and
# INCREMENT_04_1_2_MANIFEST.md Section 1. The bytes below are an exact
# repr() of the packaged file's own raw bytes: every CRLF is the literal
# two-character escape sequence \r\n (never a raw control byte in this
# cell's own source), so no line-ending normalizer - Colab's included -
# has any actual line-break byte to canonicalize. write_bytes() is binary
# mode: no text-mode newline translation occurs on any platform.
_checkshot_valid_bytes = b'VELOCITY SURVEY:(Schlumberger) OWT, vertically corrected, relative to SRD (MSL)\r\nDepth\tTVDSS\tOWT(sec)\r\n100.0\t95.0\t0.1000\r\n200.0\t190.0\t0.1900\r\n200.0\t190.5\t0.1910\r\n300.0\t285.0\t0.2800\r\n400.0\t380.0\t0.3700\r\n'
_checkshot_valid_path = Path('tests/fixtures/checkshot_valid.txt')
_checkshot_valid_path.write_bytes(_checkshot_valid_bytes)
assert _checkshot_valid_path.read_bytes() == _checkshot_valid_bytes, (
    "byte mismatch immediately after writing - write_bytes() must be exact"
)
assert _checkshot_valid_bytes.count(b"\r\n") == _checkshot_valid_bytes.count(b"\n"), (
    "every line feed must be preceded by a carriage return (pure CRLF, no bare LF)"
)
print(
    f"Wrote {_checkshot_valid_path}: {len(_checkshot_valid_bytes)} bytes, "
    f"CRLF line endings (explicit \\r\\n escapes, binary write - immune to "
    f"text-mode/Colab line-ending normalization)"
)


#### Step 7b — Write the remaining synthetic test fixtures and the new test suites

In [ ]:
%%writefile tests/fixtures/checkshot_strictly_increasing.txt
VELOCITY SURVEY:(Schlumberger) OWT, vertically corrected, relative to SRD (MSL)
Depth	TVDSS	OWT(sec)
100.0	95.0	0.1000
200.0	190.0	0.1900
300.0	285.0	0.2800


In [ ]:
%%writefile tests/fixtures/checkshot_axis_tie.txt
VELOCITY SURVEY:(Schlumberger) OWT, vertically corrected, relative to SRD (MSL)
Depth	TVDSS	OWT(sec)
100.0	95.0	0.1000
200.0	190.0	0.1900
300.0	190.0	0.2000
400.0	285.0	0.2800


In [ ]:
%%writefile tests/fixtures/checkshot_missing_header.txt
VELOCITY SURVEY:(Schlumberger) OWT, vertically corrected, relative to SRD (MSL)


In [ ]:
%%writefile tests/fixtures/checkshot_wrong_header_columns.txt
VELOCITY SURVEY:(Schlumberger) OWT, vertically corrected, relative to SRD (MSL)
Depth	OWT(sec)
100.0	0.1000


In [ ]:
%%writefile tests/fixtures/checkshot_row_width_mismatch.txt
VELOCITY SURVEY:(Schlumberger) OWT, vertically corrected, relative to SRD (MSL)
Depth	TVDSS	OWT(sec)
100.0	95.0	0.1000
200.0	190.0


In [ ]:
%%writefile tests/fixtures/checkshot_malformed_numeric.txt
VELOCITY SURVEY:(Schlumberger) OWT, vertically corrected, relative to SRD (MSL)
Depth	TVDSS	OWT(sec)
100.0	95.0	XYZ


In [ ]:
%%writefile tests/fixtures/checkshot_nonfinite.txt
VELOCITY SURVEY:(Schlumberger) OWT, vertically corrected, relative to SRD (MSL)
Depth	TVDSS	OWT(sec)
100.0	95.0	nan


In [ ]:
%%writefile tests/fixtures/checkshot_blank_line_mid.txt
VELOCITY SURVEY:(Schlumberger) OWT, vertically corrected, relative to SRD (MSL)
Depth	TVDSS	OWT(sec)
100.0	95.0	0.1000

200.0	190.0	0.1900


In [ ]:
%%writefile tests/fixtures/checkshot_no_data_rows.txt
VELOCITY SURVEY:(Schlumberger) OWT, vertically corrected, relative to SRD (MSL)
Depth	TVDSS	OWT(sec)


In [ ]:
%%writefile tests/fixtures/checkshot_hidden_reversal.txt
VELOCITY SURVEY:(Schlumberger) OWT, vertically corrected, relative to SRD (MSL)
Depth	TVDSS	OWT(sec)
100.0	95.0	0.1000
200.0	190.0	0.1900
300.0	285.0	0.2000
400.0	190.0	0.2800


In [ ]:
%%writefile tests/test_checkshot.py
"""
tests/test_checkshot.py - Validation suite for p2mem.io.checkshot
(Increment 4: checkshot file ingestion and per-file contract resolution;
Increment 4.1.1: a batch-isolation regression test proving a TimeDepthError
raised during numerical conditioning for one well - via the synthetic
`checkshot_hidden_reversal.txt` fixture - is caught and isolated exactly
like every other expected per-well failure, never propagated out of
load_checkshot_surveys and never caught with a blanket except Exception).

PORTABLE unit tests only: small synthetic fixtures under tests/fixtures/
(checkshot_*.txt). Real three-file integration (which requires the
private/raw project checkshot files) is a separate notebook/script run -
see dev_scratch_inc4/analyze_checkshot.py (not packaged) and
INCREMENT_04_MANIFEST.md for the actual recomputed results.
"""

import hashlib
from pathlib import Path

import pytest
import yaml

from p2mem.checkshot_models import CheckshotFileContract
from p2mem.io.checkshot import (
    DEPTH_BASIS_NOT_EXPLICITLY_DECLARED_CODE,
    WELL_IDENTITY_INFERRED_UNVERIFIED_CODE,
    CheckshotContractDefinitionError,
    CheckshotContractError,
    CheckshotFileNotFoundError,
    CheckshotParsingError,
    load_checkshot_contract_config,
    load_checkshot_file,
    load_checkshot_surveys,
    parse_checkshot_header,
    read_checkshot_rows,
    resolve_checkshot_contract,
)

FIXTURES = Path(__file__).parent / "fixtures"


def _sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def _base_contract(filename: str, **overrides) -> CheckshotFileContract:
    fixture_path = FIXTURES / filename
    default_sha256 = _sha256(fixture_path) if fixture_path.exists() else "0" * 64
    defaults = dict(
        source_filename=filename,
        expected_sha256=default_sha256,
        project_well_key="Test_Well_1",
        well_identity_evidence_status="verified",
        well_identity_evidence_notes="Test fixture; identity not a real project well.",
        model_use_status="primary_model",
        expected_survey_statement=(
            "VELOCITY SURVEY:(Schlumberger) OWT, vertically corrected, relative to SRD (MSL)"
        ),
        expected_column_header_line="Depth\tTVDSS\tOWT(sec)",
        expected_column_order=("Depth", "TVDSS", "OWT(sec)"),
        expected_column_count=3,
        expected_time_type="OWT",
        expected_time_unit="s",
        expected_vertical_correction_fragment="vertically corrected",
        expected_srd_reference_fragment="relative to SRD (MSL)",
        expected_row_count=5,
        expected_depth_min_m=100.0,
        expected_depth_max_m=400.0,
        expected_tvdss_min_m=95.0,
        expected_tvdss_max_m=380.0,
        expected_owt_min_s=0.1000,
        expected_owt_max_s=0.3700,
        numeric_range_tolerance=0.0001,
        depth_basis_interpretation_status="candidate_md_evaluated_against_locked_survey",
        duplicate_tie_policy="raw_rows_preserved_median_representative_all_ties_registered",
        interpolation_policy="piecewise_linear_no_extrapolation",
        extrapolation_policy="disallowed_nan_outside_coverage",
        notes="Test fixture contract.",
    )
    defaults.update(overrides)
    return CheckshotFileContract(**defaults)


# ---------------------------------------------------------------------------
# Structural parsing
# ---------------------------------------------------------------------------
def test_valid_file_parses_header_and_rows():
    header = parse_checkshot_header(str(FIXTURES / "checkshot_valid.txt"))
    assert header.survey_statement.startswith("VELOCITY SURVEY")
    assert header.column_names == ("Depth", "TVDSS", "OWT(sec)")
    assert header.line_ending_convention == "CRLF"
    stations = read_checkshot_rows(str(FIXTURES / "checkshot_valid.txt"))
    assert stations.Depth_source_m.size == 5
    assert stations.Depth_source_m.tolist() == [100.0, 200.0, 200.0, 300.0, 400.0]


def test_strictly_increasing_file_has_no_duplicate_depth():
    stations = read_checkshot_rows(str(FIXTURES / "checkshot_strictly_increasing.txt"))
    assert stations.Depth_source_m.tolist() == [100.0, 200.0, 300.0]


def test_file_not_found_raises_typed_error():
    with pytest.raises(CheckshotFileNotFoundError):
        parse_checkshot_header(str(FIXTURES / "does_not_exist.txt"))


def test_missing_header_line_rejected():
    with pytest.raises(CheckshotParsingError):
        parse_checkshot_header(str(FIXTURES / "checkshot_missing_header.txt"))


def test_wrong_header_column_count_rejected():
    with pytest.raises(CheckshotParsingError):
        parse_checkshot_header(str(FIXTURES / "checkshot_wrong_header_columns.txt"))


def test_row_width_mismatch_rejected():
    with pytest.raises(CheckshotParsingError):
        read_checkshot_rows(str(FIXTURES / "checkshot_row_width_mismatch.txt"))


def test_malformed_numeric_token_rejected():
    with pytest.raises(CheckshotParsingError):
        read_checkshot_rows(str(FIXTURES / "checkshot_malformed_numeric.txt"))


def test_nonfinite_literal_token_rejected():
    with pytest.raises(CheckshotParsingError):
        read_checkshot_rows(str(FIXTURES / "checkshot_nonfinite.txt"))


def test_blank_line_before_last_row_rejected():
    with pytest.raises(CheckshotParsingError):
        read_checkshot_rows(str(FIXTURES / "checkshot_blank_line_mid.txt"))


def test_no_data_rows_rejected():
    with pytest.raises(CheckshotParsingError):
        read_checkshot_rows(str(FIXTURES / "checkshot_no_data_rows.txt"))


# ---------------------------------------------------------------------------
# Contract resolution
# ---------------------------------------------------------------------------
def test_valid_file_and_contract_resolves_cleanly():
    path = str(FIXTURES / "checkshot_valid.txt")
    header = parse_checkshot_header(path)
    stations = read_checkshot_rows(path)
    contract = _base_contract("checkshot_valid.txt")
    issues = resolve_checkshot_contract(header, stations, contract)
    assert all(i.severity == "WARNING" for i in issues)
    codes = {i.code for i in issues}
    assert DEPTH_BASIS_NOT_EXPLICITLY_DECLARED_CODE in codes


def test_wrong_filename_is_blocking():
    path = str(FIXTURES / "checkshot_valid.txt")
    header = parse_checkshot_header(path)
    stations = read_checkshot_rows(path)
    contract = _base_contract("checkshot_valid.txt", source_filename="Some-Other-Name.txt")
    issues = resolve_checkshot_contract(header, stations, contract)
    assert any(i.code == "FILENAME_MISMATCH" and i.severity == "ERROR" for i in issues)


def test_wrong_sha256_is_blocking():
    path = str(FIXTURES / "checkshot_valid.txt")
    header = parse_checkshot_header(path)
    stations = read_checkshot_rows(path)
    contract = _base_contract("checkshot_valid.txt", expected_sha256="0" * 64)
    issues = resolve_checkshot_contract(header, stations, contract)
    assert any(i.code == "SHA256_MISMATCH" and i.severity == "ERROR" for i in issues)


def test_wrong_survey_statement_is_blocking():
    path = str(FIXTURES / "checkshot_valid.txt")
    header = parse_checkshot_header(path)
    stations = read_checkshot_rows(path)
    contract = _base_contract("checkshot_valid.txt", expected_survey_statement="WRONG STATEMENT")
    issues = resolve_checkshot_contract(header, stations, contract)
    codes = {(i.code, i.severity) for i in issues}
    assert ("SURVEY_STATEMENT_MISMATCH", "ERROR") in codes


def test_fragment_missing_from_actual_header_is_blocking():
    # Fragment checks are independent of the exact-statement check: they
    # verify the CONTRACT's declared fragment is present in the ACTUAL
    # file's own header text (not the contract's expected text).
    path = str(FIXTURES / "checkshot_valid.txt")
    header = parse_checkshot_header(path)
    stations = read_checkshot_rows(path)
    contract = _base_contract(
        "checkshot_valid.txt", expected_vertical_correction_fragment="this fragment is not present"
    )
    issues = resolve_checkshot_contract(header, stations, contract)
    assert any(i.code == "VERTICAL_CORRECTION_STATEMENT_MISSING" and i.severity == "ERROR" for i in issues)


def test_wrong_column_header_line_is_blocking():
    path = str(FIXTURES / "checkshot_valid.txt")
    header = parse_checkshot_header(path)
    stations = read_checkshot_rows(path)
    contract = _base_contract("checkshot_valid.txt", expected_column_header_line="A\tB\tC")
    issues = resolve_checkshot_contract(header, stations, contract)
    assert any(i.code == "COLUMN_HEADER_LINE_MISMATCH" and i.severity == "ERROR" for i in issues)


def test_wrong_column_order_is_blocking():
    path = str(FIXTURES / "checkshot_valid.txt")
    header = parse_checkshot_header(path)
    stations = read_checkshot_rows(path)
    contract = _base_contract("checkshot_valid.txt", expected_column_order=("TVDSS", "Depth", "OWT(sec)"))
    issues = resolve_checkshot_contract(header, stations, contract)
    assert any(i.code == "COLUMN_ORDER_MISMATCH" and i.severity == "ERROR" for i in issues)


def test_wrong_row_count_is_blocking():
    path = str(FIXTURES / "checkshot_valid.txt")
    header = parse_checkshot_header(path)
    stations = read_checkshot_rows(path)
    contract = _base_contract("checkshot_valid.txt", expected_row_count=999)
    issues = resolve_checkshot_contract(header, stations, contract)
    assert any(i.code == "ROW_COUNT_MISMATCH" and i.severity == "ERROR" for i in issues)


def test_numeric_range_mismatch_is_blocking():
    path = str(FIXTURES / "checkshot_valid.txt")
    header = parse_checkshot_header(path)
    stations = read_checkshot_rows(path)
    contract = _base_contract("checkshot_valid.txt", expected_depth_max_m=999.0)
    issues = resolve_checkshot_contract(header, stations, contract)
    assert any(i.code == "NUMERIC_RANGE_MISMATCH" and i.severity == "ERROR" for i in issues)


def test_numeric_range_within_tolerance_is_not_flagged():
    path = str(FIXTURES / "checkshot_valid.txt")
    header = parse_checkshot_header(path)
    stations = read_checkshot_rows(path)
    contract = _base_contract("checkshot_valid.txt", expected_depth_max_m=400.00005)
    issues = resolve_checkshot_contract(header, stations, contract)
    assert not any(i.code == "NUMERIC_RANGE_MISMATCH" for i in issues)


def test_depth_basis_warning_always_present_on_success():
    path = str(FIXTURES / "checkshot_valid.txt")
    header = parse_checkshot_header(path)
    stations = read_checkshot_rows(path)
    contract = _base_contract("checkshot_valid.txt")
    issues = resolve_checkshot_contract(header, stations, contract)
    codes = {i.code for i in issues}
    assert DEPTH_BASIS_NOT_EXPLICITLY_DECLARED_CODE in codes


def test_well_identity_inferred_unverified_warning_present_when_declared():
    path = str(FIXTURES / "checkshot_valid.txt")
    header = parse_checkshot_header(path)
    stations = read_checkshot_rows(path)
    contract = _base_contract(
        "checkshot_valid.txt",
        well_identity_evidence_status="inferred_unverified",
        model_use_status="qc_only",
    )
    issues = resolve_checkshot_contract(header, stations, contract)
    codes = {i.code for i in issues}
    assert WELL_IDENTITY_INFERRED_UNVERIFIED_CODE in codes


def test_well_identity_inferred_warning_absent_when_verified():
    path = str(FIXTURES / "checkshot_valid.txt")
    header = parse_checkshot_header(path)
    stations = read_checkshot_rows(path)
    contract = _base_contract("checkshot_valid.txt", well_identity_evidence_status="verified")
    issues = resolve_checkshot_contract(header, stations, contract)
    codes = {i.code for i in issues}
    assert WELL_IDENTITY_INFERRED_UNVERIFIED_CODE not in codes


# ---------------------------------------------------------------------------
# load_checkshot_file / load_checkshot_surveys
# ---------------------------------------------------------------------------
def test_load_checkshot_file_returns_typed_result_with_ties_and_conditioning():
    contract = _base_contract("checkshot_valid.txt")
    result = load_checkshot_file(str(FIXTURES / "checkshot_valid.txt"), contract)
    assert result.contract_status == "PASSED"
    assert result.raw.Depth_source_m.size == 5
    assert result.conditioned.n_conditioned_rows == 4  # one duplicate Depth=200.0 group collapsed
    assert result.conditioned.n_tie_groups == 1
    assert len(result.duplicate_ties) == 1
    tie = result.duplicate_ties[0]
    assert tie.tie_value_m == 200.0
    assert tie.group_size == 2
    assert tie.selected_representative_tvdss_m == pytest.approx((190.0 + 190.5) / 2.0)
    assert result.depth_comparison is None  # no survey trajectory supplied


def test_load_checkshot_file_contract_error_carries_all_issues():
    contract = _base_contract("checkshot_valid.txt", expected_sha256="0" * 64)
    with pytest.raises(CheckshotContractError) as exc_info:
        load_checkshot_file(str(FIXTURES / "checkshot_valid.txt"), contract)
    assert any(i.code == "SHA256_MISMATCH" for i in exc_info.value.issues)


def test_load_checkshot_file_with_survey_trajectory_computes_depth_comparison():
    import numpy as np

    contract = _base_contract(
        "checkshot_strictly_increasing.txt",
        expected_row_count=3,
        expected_depth_min_m=100.0, expected_depth_max_m=300.0,
        expected_tvdss_min_m=95.0, expected_tvdss_max_m=285.0,
        expected_owt_min_s=0.10, expected_owt_max_s=0.28,
    )
    survey_md = np.array([0.0, 100.0, 200.0, 300.0, 400.0])
    survey_tvd = np.array([0.0, 95.0, 190.0, 285.0, 380.0])
    result = load_checkshot_file(
        str(FIXTURES / "checkshot_strictly_increasing.txt"),
        contract,
        survey_md_m=survey_md,
        survey_tvd_m=survey_tvd,
        datum_elevation_m=0.0,
    )
    assert result.depth_comparison is not None
    assert result.depth_comparison.n_compared == 3
    assert result.depth_comparison.max_abs_residual_m == pytest.approx(0.0, abs=1e-9)


def test_load_checkshot_file_builds_axis_conditioned_tables_when_no_axis_ties():
    # checkshot_valid.txt has a Depth tie but its Depth-conditioned
    # TVDSS/OWT arrays are both strictly increasing already -> zero
    # axis-tie entries, but the tables must still be built (Increment 4.1).
    contract = _base_contract("checkshot_valid.txt")
    result = load_checkshot_file(str(FIXTURES / "checkshot_valid.txt"), contract)
    assert result.axis_tie_entries == ()
    assert set(result.axis_tables.keys()) == {"tvdss_to_owt", "owt_to_tvdss"}
    assert result.axis_tables["tvdss_to_owt"].n_axis_tie_groups == 0
    assert result.axis_tables["owt_to_tvdss"].n_axis_tie_groups == 0


def test_load_checkshot_file_registers_genuine_axis_tie_and_conditions_median(tmp_path=None):
    # checkshot_axis_tie.txt has NO Depth ties, but TVDSS=190.0 repeats at
    # Depth=200 and Depth=300 with DIFFERENT OWT values (0.19 vs 0.20) -
    # this is the Increment 4.1 order-invariant axis-tie-conditioning path
    # exercised end-to-end through load_checkshot_file.
    contract = _base_contract(
        "checkshot_axis_tie.txt",
        expected_row_count=4,
        expected_depth_min_m=100.0, expected_depth_max_m=400.0,
        expected_tvdss_min_m=95.0, expected_tvdss_max_m=285.0,
        expected_owt_min_s=0.10, expected_owt_max_s=0.28,
    )
    result = load_checkshot_file(str(FIXTURES / "checkshot_axis_tie.txt"), contract)
    assert result.conditioned.n_tie_groups == 0  # no Depth ties
    assert len(result.axis_tie_entries) == 1
    tie = result.axis_tie_entries[0]
    assert tie.interpolation_direction == "tvdss_to_owt"
    assert tie.tie_axis_value == pytest.approx(190.0)
    assert tie.tie_kind == "genuinely_non_unique"
    assert tie.selected_representative_dependent_value == pytest.approx(0.195)  # median of 0.19/0.20
    table = result.axis_tables["tvdss_to_owt"]
    assert table.n_axis_tie_groups == 1
    assert table.axis_values.size == 3  # 4 input points collapse to 3 after the tie


def test_batch_isolates_one_failed_well_from_successful_wells():
    contracts = {
        "checkshot_valid.txt": _base_contract("checkshot_valid.txt"),
        "checkshot_strictly_increasing.txt": _base_contract(
            "checkshot_strictly_increasing.txt", expected_sha256="0" * 64, expected_row_count=3
        ),
    }
    file_paths = {
        "Well_A": str(FIXTURES / "checkshot_valid.txt"),
        "Well_B": str(FIXTURES / "checkshot_strictly_increasing.txt"),
    }
    results, failures = load_checkshot_surveys(file_paths, contracts)
    assert "Well_A" in results
    assert "Well_B" in failures
    assert failures["Well_B"].error_type == "contract_failure"


def test_batch_isolates_numerical_conditioning_failure_from_successful_wells():
    # Increment 4.1.1 (Blocking Defect 4): a TimeDepthError raised during
    # numerical conditioning (here, the hidden-reversal-via-global-
    # grouping defect fixed in build_axis_conditioned_lookup_table) for
    # one well must be caught and isolated exactly like a file/parsing/
    # contract failure - it must NOT propagate out of load_checkshot_
    # surveys and stop the other, valid well from loading. Never caught
    # via a blanket `except Exception`.
    contracts = {
        "checkshot_valid.txt": _base_contract("checkshot_valid.txt"),
        "checkshot_hidden_reversal.txt": _base_contract(
            "checkshot_hidden_reversal.txt",
            expected_row_count=4,
            expected_depth_min_m=100.0, expected_depth_max_m=400.0,
            expected_tvdss_min_m=95.0, expected_tvdss_max_m=285.0,
            expected_owt_min_s=0.10, expected_owt_max_s=0.28,
        ),
    }
    file_paths = {
        "Well_A": str(FIXTURES / "checkshot_valid.txt"),
        "Well_B": str(FIXTURES / "checkshot_hidden_reversal.txt"),
    }
    results, failures = load_checkshot_surveys(file_paths, contracts)

    assert "Well_A" in results  # valid well still loads
    assert "Well_B" in failures  # failing well is isolated, not propagated
    assert failures["Well_B"].error_type == "numerical_conditioning_failure"
    message = failures["Well_B"].message
    assert "genuine reversal" in message
    # No absolute path leakage in the failure message.
    assert str(FIXTURES) not in message
    assert "/" not in message  # TimeDepthError messages carry well_key/numeric values only


def test_batch_missing_contract_raises_definition_error():
    contracts = {"checkshot_valid.txt": _base_contract("checkshot_valid.txt")}
    file_paths = {"Well_X": str(FIXTURES / "checkshot_strictly_increasing.txt")}
    with pytest.raises(CheckshotContractDefinitionError):
        load_checkshot_surveys(file_paths, contracts)


# ---------------------------------------------------------------------------
# Contract YAML loading
# ---------------------------------------------------------------------------
def _valid_yaml_entry(filename: str, sha256: str) -> dict:
    return {
        "expected_sha256": sha256,
        "project_well_key": "Test_Well_1",
        "well_identity_evidence_status": "verified",
        "well_identity_evidence_notes": "Test note.",
        "model_use_status": "primary_model",
        "expected_survey_statement": (
            "VELOCITY SURVEY:(Schlumberger) OWT, vertically corrected, relative to SRD (MSL)"
        ),
        "expected_column_header_line": "Depth\tTVDSS\tOWT(sec)",
        "expected_column_order": ["Depth", "TVDSS", "OWT(sec)"],
        "expected_time_type": "OWT",
        "expected_time_unit": "s",
        "expected_vertical_correction_fragment": "vertically corrected",
        "expected_srd_reference_fragment": "relative to SRD (MSL)",
        "expected_row_count": 5,
        "expected_depth_min_m": 100.0,
        "expected_depth_max_m": 400.0,
        "expected_tvdss_min_m": 95.0,
        "expected_tvdss_max_m": 380.0,
        "expected_owt_min_s": 0.1,
        "expected_owt_max_s": 0.37,
        "numeric_range_tolerance": 0.0001,
        "depth_basis_interpretation_status": "candidate_md_evaluated_against_locked_survey",
        "duplicate_tie_policy": "raw_rows_preserved_median_representative_all_ties_registered",
        "interpolation_policy": "piecewise_linear_no_extrapolation",
        "extrapolation_policy": "disallowed_nan_outside_coverage",
        "notes": "Test entry.",
    }


def test_valid_contract_yaml_loads(tmp_path):
    sha = _sha256(FIXTURES / "checkshot_valid.txt")
    doc = {"files": {"checkshot_valid.txt": _valid_yaml_entry("checkshot_valid.txt", sha)}}
    p = tmp_path / "contracts.yml"
    p.write_text(yaml.safe_dump(doc))
    contracts = load_checkshot_contract_config(str(p))
    assert "checkshot_valid.txt" in contracts
    assert contracts["checkshot_valid.txt"].expected_row_count == 5


def test_duplicate_contract_key_rejected(tmp_path):
    sha = _sha256(FIXTURES / "checkshot_valid.txt")
    raw_yaml = f"""
files:
  checkshot_valid.txt:
    expected_sha256: "{sha}"
  checkshot_valid.txt:
    expected_sha256: "{sha}"
"""
    p = tmp_path / "contracts.yml"
    p.write_text(raw_yaml)
    with pytest.raises(CheckshotContractDefinitionError):
        load_checkshot_contract_config(str(p))


def test_missing_required_field_rejected(tmp_path):
    sha = _sha256(FIXTURES / "checkshot_valid.txt")
    entry = _valid_yaml_entry("checkshot_valid.txt", sha)
    del entry["expected_row_count"]
    doc = {"files": {"checkshot_valid.txt": entry}}
    p = tmp_path / "contracts.yml"
    p.write_text(yaml.safe_dump(doc))
    with pytest.raises(CheckshotContractDefinitionError):
        load_checkshot_contract_config(str(p))


def test_invalid_identity_status_rejected(tmp_path):
    sha = _sha256(FIXTURES / "checkshot_valid.txt")
    entry = _valid_yaml_entry("checkshot_valid.txt", sha)
    entry["well_identity_evidence_status"] = "not_a_real_status"
    doc = {"files": {"checkshot_valid.txt": entry}}
    p = tmp_path / "contracts.yml"
    p.write_text(yaml.safe_dump(doc))
    with pytest.raises(CheckshotContractDefinitionError):
        load_checkshot_contract_config(str(p))


def test_invalid_model_use_status_rejected(tmp_path):
    sha = _sha256(FIXTURES / "checkshot_valid.txt")
    entry = _valid_yaml_entry("checkshot_valid.txt", sha)
    entry["model_use_status"] = "not_a_real_status"
    doc = {"files": {"checkshot_valid.txt": entry}}
    p = tmp_path / "contracts.yml"
    p.write_text(yaml.safe_dump(doc))
    with pytest.raises(CheckshotContractDefinitionError):
        load_checkshot_contract_config(str(p))


def test_wrong_column_order_length_rejected(tmp_path):
    sha = _sha256(FIXTURES / "checkshot_valid.txt")
    entry = _valid_yaml_entry("checkshot_valid.txt", sha)
    entry["expected_column_order"] = ["Depth", "TVDSS"]
    doc = {"files": {"checkshot_valid.txt": entry}}
    p = tmp_path / "contracts.yml"
    p.write_text(yaml.safe_dump(doc))
    with pytest.raises(CheckshotContractDefinitionError):
        load_checkshot_contract_config(str(p))


def test_min_greater_than_max_rejected(tmp_path):
    sha = _sha256(FIXTURES / "checkshot_valid.txt")
    entry = _valid_yaml_entry("checkshot_valid.txt", sha)
    entry["expected_depth_min_m"] = 500.0
    entry["expected_depth_max_m"] = 100.0
    doc = {"files": {"checkshot_valid.txt": entry}}
    p = tmp_path / "contracts.yml"
    p.write_text(yaml.safe_dump(doc))
    with pytest.raises(CheckshotContractDefinitionError):
        load_checkshot_contract_config(str(p))


def test_negative_tolerance_rejected(tmp_path):
    sha = _sha256(FIXTURES / "checkshot_valid.txt")
    entry = _valid_yaml_entry("checkshot_valid.txt", sha)
    entry["numeric_range_tolerance"] = -0.1
    doc = {"files": {"checkshot_valid.txt": entry}}
    p = tmp_path / "contracts.yml"
    p.write_text(yaml.safe_dump(doc))
    with pytest.raises(CheckshotContractDefinitionError):
        load_checkshot_contract_config(str(p))


def test_real_checkshot_contracts_yaml_loads_and_matches_real_files():
    """Sanity check against the actual packaged Increment 4 contract file
    and the real approved (hyphenated) filenames - values themselves are
    exercised end-to-end only in the dev-only real integration run."""
    contracts = load_checkshot_contract_config(
        str(Path(__file__).parent.parent / "config" / "checkshot_contracts.yml")
    )
    assert set(contracts) == {
        "Poseidon2-Checkshot.txt",
        "Boreas1-Checkshot.txt",
        "Proteus1-Checkshot.txt",
    }
    assert contracts["Poseidon2-Checkshot.txt"].model_use_status == "primary_model"
    assert contracts["Boreas1-Checkshot.txt"].model_use_status == "qc_only"
    assert contracts["Proteus1-Checkshot.txt"].well_identity_evidence_status == "inferred_unverified"


In [ ]:
%%writefile tests/test_time_depth.py
"""
tests/test_time_depth.py - Validation suite for p2mem.time_depth
(Increment 4: duplicate-tie conditioning, velocity diagnostics,
forward/inverse time-depth interpolation, checkshot-vs-survey depth
comparison, and the sonic-checkshot drift diagnostic; Increment 4.1:
order-invariant axis-tie conditioning for TVDSS<->OWT/TWT inversion, and
numerical-validation hardening of trapezoidal_integrate and
compute_sonic_checkshot_drift; Increment 4.1.1: a hidden-reversal-via-
global-grouping regression suite for build_axis_conditioned_lookup_table,
full-canonical-MD (not just selected-run) monotonicity tests for
compute_sonic_checkshot_drift, a typed-error test for zero in-coverage
checkshot rows in compare_checkshot_to_survey, and a dtype-rejection
audit for seconds_to_milliseconds/milliseconds_to_seconds).

PORTABLE unit tests only: small synthetic arrays, never the real project
files. The exact real-data axis-tie-conditioning results (the three
specific Increment 4 first-point regressions this patch fixes) are
independently reproduced and asserted against the real checkshot files in
`dev_scratch_inc4_1/run_integration_04.py` (dev-only, not packaged) - not
here, consistent with this file's synthetic-data-only scope. The synthetic
tests below reproduce the identical *shape* of that defect (an
order-dependent "keep first" tie-break vs. an order-invariant median) so
the mechanism is unit-tested independently of any real file.
"""

import math

import numpy as np
import pytest

from p2mem.checkshot_models import AxisConditionedLookupTable, ConditionedCheckshotData
from p2mem.time_depth import (
    TimeDepthError,
    build_axis_conditioned_lookup_table,
    build_axis_conditioned_tables_for_well,
    compare_checkshot_to_survey,
    compute_sonic_checkshot_drift,
    compute_velocity_diagnostics,
    depth_to_owt,
    depth_to_tvdss,
    depth_to_twt,
    detect_and_condition_depth_ties,
    find_longest_finite_positive_run,
    map_las_md_to_checkshot_time,
    milliseconds_to_seconds,
    owt_to_tvdss,
    seconds_to_milliseconds,
    trapezoidal_integrate,
    tvdss_to_owt,
    tvdss_to_twt,
    twt_to_tvdss,
)
from p2mem.units import owt_to_twt as locked_owt_to_twt


def _conditioned(depth, tvdss, owt) -> ConditionedCheckshotData:
    depth = np.asarray(depth, dtype=np.float64)
    tvdss = np.asarray(tvdss, dtype=np.float64)
    owt = np.asarray(owt, dtype=np.float64)
    return ConditionedCheckshotData(
        Depth_conditioned_m=depth,
        TVDSS_conditioned_m=tvdss,
        OWT_conditioned_s=owt,
        n_raw_rows=int(depth.size),
        n_conditioned_rows=int(depth.size),
        n_tie_groups=0,
        conditioning_method="test",
    )


# ---------------------------------------------------------------------------
# Unit helpers / integration
# ---------------------------------------------------------------------------
def test_seconds_milliseconds_round_trip():
    s = np.array([0.5, 1.0, 2.25])
    ms = seconds_to_milliseconds(s)
    assert np.allclose(ms, [500.0, 1000.0, 2250.0])
    assert np.allclose(milliseconds_to_seconds(ms), s)


def test_trapezoidal_integrate_constant_function():
    x = np.array([0.0, 1.0, 2.0, 3.0])
    y = np.array([2.0, 2.0, 2.0, 2.0])
    assert trapezoidal_integrate(y, x) == pytest.approx(6.0)


def test_trapezoidal_integrate_linear_function_matches_analytic():
    x = np.linspace(0.0, 10.0, 101)
    y = 3.0 * x + 1.0
    # integral of (3x+1) from 0 to 10 = 1.5*100 + 10 = 160
    assert trapezoidal_integrate(y, x) == pytest.approx(160.0, rel=1e-9)


def test_trapezoidal_integrate_rejects_non_1d_input():
    with pytest.raises(TimeDepthError):
        trapezoidal_integrate(np.array([[1.0, 2.0]]), np.array([[0.0, 1.0]]))


def test_trapezoidal_integrate_rejects_mismatched_lengths():
    with pytest.raises(TimeDepthError):
        trapezoidal_integrate(np.array([1.0, 2.0, 3.0]), np.array([0.0, 1.0]))


def test_trapezoidal_integrate_rejects_fewer_than_two_samples():
    with pytest.raises(TimeDepthError):
        trapezoidal_integrate(np.array([1.0]), np.array([0.0]))


def test_trapezoidal_integrate_rejects_non_finite_values():
    with pytest.raises(TimeDepthError):
        trapezoidal_integrate(np.array([1.0, np.nan]), np.array([0.0, 1.0]))
    with pytest.raises(TimeDepthError):
        trapezoidal_integrate(np.array([1.0, 2.0]), np.array([0.0, np.inf]))


def test_trapezoidal_integrate_rejects_non_increasing_x_never_sorts():
    # Regression (Increment 4.1): the pre-4.1 implementation performed no
    # validation and could silently integrate a decreasing/duplicate x into
    # a physically invalid (e.g. negative) result. It must now reject
    # rather than sort or otherwise repair x.
    with pytest.raises(TimeDepthError):
        trapezoidal_integrate(np.array([1.0, 2.0, 3.0]), np.array([0.0, 1.0, 1.0]))  # duplicate
    with pytest.raises(TimeDepthError):
        trapezoidal_integrate(np.array([1.0, 2.0, 3.0]), np.array([0.0, 2.0, 1.0]))  # decreasing


# ---------------------------------------------------------------------------
# Duplicate-tie detection and conditioning
# ---------------------------------------------------------------------------
def test_no_ties_passes_through_unchanged():
    ties, cond = detect_and_condition_depth_ties(
        "W", [100.0, 200.0, 300.0], [95.0, 190.0, 285.0], [0.1, 0.19, 0.28]
    )
    assert ties == ()
    assert cond.n_conditioned_rows == 3
    assert cond.n_tie_groups == 0
    assert np.allclose(cond.Depth_conditioned_m, [100.0, 200.0, 300.0])


def test_duplicate_pair_collapses_to_median_which_equals_mean_for_pairs():
    ties, cond = detect_and_condition_depth_ties(
        "W", [100.0, 200.0, 200.0, 300.0], [95.0, 190.0, 190.5, 285.0], [0.10, 0.19, 0.191, 0.28]
    )
    assert cond.n_conditioned_rows == 3
    assert cond.n_tie_groups == 1
    assert len(ties) == 1
    tie = ties[0]
    assert tie.group_size == 2
    assert tie.source_row_indices == (1, 2)
    assert tie.selected_representative_tvdss_m == pytest.approx((190.0 + 190.5) / 2.0)
    assert tie.selected_representative_owt_s == pytest.approx((0.19 + 0.191) / 2.0)
    idx200 = np.where(cond.Depth_conditioned_m == 200.0)[0][0]
    assert cond.TVDSS_conditioned_m[idx200] == pytest.approx((190.0 + 190.5) / 2.0)


def test_odd_sized_group_median_is_middle_value():
    ties, cond = detect_and_condition_depth_ties(
        "W", [100.0, 100.0, 100.0], [10.0, 20.0, 30.0], [0.1, 0.2, 0.3]
    )
    assert len(ties) == 1
    assert ties[0].group_size == 3
    assert ties[0].selected_representative_tvdss_m == pytest.approx(20.0)  # median of 10,20,30


def test_raw_rows_never_overwritten_all_preserved_in_register():
    ties, _ = detect_and_condition_depth_ties(
        "W", [100.0, 200.0, 200.0], [95.0, 190.0, 190.5], [0.10, 0.19, 0.191]
    )
    assert ties[0].original_tvdss_m == (190.0, 190.5)
    assert ties[0].original_owt_s == (0.19, 0.191)


def test_mismatched_array_lengths_raises():
    with pytest.raises(TimeDepthError):
        detect_and_condition_depth_ties("W", [1.0, 2.0], [1.0], [1.0, 2.0])


def test_empty_arrays_raises():
    with pytest.raises(TimeDepthError):
        detect_and_condition_depth_ties("W", [], [], [])


def test_out_of_order_distinct_depths_raises():
    # distinct Depth values visited out of sorted order (1, 3, 2) -- not an
    # ordinary repeated tie-in, should be rejected rather than silently conditioned.
    with pytest.raises(TimeDepthError):
        detect_and_condition_depth_ties("W", [1.0, 3.0, 2.0], [1.0, 3.0, 2.0], [0.1, 0.3, 0.2])


# ---------------------------------------------------------------------------
# Velocity diagnostics
# ---------------------------------------------------------------------------
def test_vavg_and_vint_basic_case():
    cond = _conditioned([0.0, 100.0, 200.0], [0.0, 100.0, 200.0], [0.05, 0.10, 0.15])
    result = compute_velocity_diagnostics("W", cond)
    assert np.allclose(result.Vavg_m_s[1:], [1000.0, 1333.333333], rtol=1e-6)
    assert np.allclose(result.Vint_m_s, [2000.0, 2000.0])
    assert result.n_vint_invalid == 0


def test_vint_nan_on_zero_delta_owt_never_inf():
    cond = _conditioned([0.0, 100.0], [0.0, 100.0], [0.05, 0.05])
    result = compute_velocity_diagnostics("W", cond)
    assert math.isnan(result.Vint_m_s[0])
    assert not np.isinf(result.Vint_m_s[0])
    assert result.n_vint_invalid == 1
    assert result.vint_invalid_reason_counts["non_positive_delta_owt"] == 1


def test_vint_nan_on_negative_delta_tvdss_never_negative_velocity():
    cond = _conditioned([0.0, 100.0], [100.0, 50.0], [0.05, 0.10])
    result = compute_velocity_diagnostics("W", cond)
    assert math.isnan(result.Vint_m_s[0])
    assert result.vint_invalid_reason_counts["non_positive_delta_tvdss"] == 1


def test_vavg_nan_when_owt_non_positive():
    cond = _conditioned([0.0, 100.0], [0.0, 100.0], [0.0, 0.10])
    result = compute_velocity_diagnostics("W", cond)
    assert math.isnan(result.Vavg_m_s[0])


# ---------------------------------------------------------------------------
# Checkshot-vs-survey depth comparison
# ---------------------------------------------------------------------------
def test_compare_checkshot_to_survey_zero_residual_when_consistent():
    survey_md = np.array([0.0, 100.0, 200.0, 300.0])
    survey_tvd = np.array([0.0, 95.0, 190.0, 285.0])
    depth_source = np.array([100.0, 200.0])
    tvdss_source = np.array([95.0, 190.0])
    result = compare_checkshot_to_survey(
        "W", depth_source, tvdss_source, survey_md, survey_tvd, datum_elevation_m=0.0,
        depth_basis_interpretation_status="candidate_md_evaluated_against_locked_survey",
    )
    assert result.max_abs_residual_m == pytest.approx(0.0, abs=1e-9)
    assert result.n_compared == 2
    assert result.n_outside_survey_md_coverage == 0


def test_compare_checkshot_to_survey_constant_offset_detected():
    survey_md = np.array([0.0, 100.0, 200.0, 300.0])
    survey_tvd = np.array([0.0, 95.0, 190.0, 285.0])
    depth_source = np.array([100.0, 200.0, 300.0])
    tvdss_source = survey_tvd[1:] - 0.5  # checkshot reads 0.5 m shallower everywhere
    result = compare_checkshot_to_survey(
        "W", depth_source, tvdss_source, survey_md, survey_tvd, datum_elevation_m=0.0,
        depth_basis_interpretation_status="candidate_md_evaluated_against_locked_survey",
    )
    assert result.mean_residual_m == pytest.approx(0.5, abs=1e-9)
    assert "near-constant" in result.residual_trend_description


def test_compare_checkshot_to_survey_excludes_out_of_coverage_rows():
    survey_md = np.array([0.0, 100.0, 200.0])
    survey_tvd = np.array([0.0, 95.0, 190.0])
    depth_source = np.array([50.0, 500.0])  # 500 is outside survey MD coverage
    tvdss_source = np.array([47.5, 999.0])
    result = compare_checkshot_to_survey(
        "W", depth_source, tvdss_source, survey_md, survey_tvd, datum_elevation_m=0.0,
        depth_basis_interpretation_status="candidate_md_evaluated_against_locked_survey",
    )
    assert result.n_compared == 1
    assert result.n_outside_survey_md_coverage == 1


def test_compare_checkshot_to_survey_rejects_non_monotonic_survey():
    survey_md = np.array([0.0, 100.0, 50.0])
    survey_tvd = np.array([0.0, 95.0, 190.0])
    with pytest.raises(TimeDepthError):
        compare_checkshot_to_survey(
            "W", np.array([10.0]), np.array([10.0]), survey_md, survey_tvd, datum_elevation_m=0.0,
            depth_basis_interpretation_status="candidate_md_evaluated_against_locked_survey",
        )


# ---------------------------------------------------------------------------
# Depth-domain forward interpolation
# ---------------------------------------------------------------------------
def test_depth_to_tvdss_inside_and_outside_coverage():
    cond = _conditioned([100.0, 200.0, 300.0], [95.0, 190.0, 285.0], [0.10, 0.19, 0.28])
    values, mask = depth_to_tvdss([150.0, 50.0, 400.0], cond)
    assert values[0] == pytest.approx(142.5)
    assert math.isnan(values[1])
    assert math.isnan(values[2])
    assert mask.tolist() == [True, False, False]


def test_depth_to_owt_exact_at_station():
    cond = _conditioned([100.0, 200.0, 300.0], [95.0, 190.0, 285.0], [0.10, 0.19, 0.28])
    values, mask = depth_to_owt([200.0], cond)
    assert values[0] == pytest.approx(0.19)


def test_depth_to_twt_uses_locked_owt_to_twt():
    cond = _conditioned([100.0, 200.0], [95.0, 190.0], [0.10, 0.20])
    twt, mask = depth_to_twt([100.0, 200.0], cond)
    owt, _ = depth_to_owt([100.0, 200.0], cond)
    assert np.allclose(twt, locked_owt_to_twt(owt))
    assert np.allclose(twt, [0.20, 0.40])


# ---------------------------------------------------------------------------
# Time-domain forward/inverse interpolation via AxisConditionedLookupTable
# (Increment 4.1: tvdss_to_owt/owt_to_tvdss/tvdss_to_twt/twt_to_tvdss now
# take a pre-built AxisConditionedLookupTable, not a bare
# ConditionedCheckshotData, and return a 2-tuple (values, mask) - the old
# 3-tuple with n_dropped no longer exists, because points are never
# dropped: every tied point is registered and folded into a median
# representative instead.)
# ---------------------------------------------------------------------------
def test_owt_to_tvdss_inverts_cleanly_when_strictly_monotonic():
    cond = _conditioned([100.0, 200.0, 300.0], [95.0, 190.0, 285.0], [0.10, 0.19, 0.28])
    _, tables = build_axis_conditioned_tables_for_well("W", cond)
    values, mask = owt_to_tvdss([0.19], tables["owt_to_tvdss"])
    assert values[0] == pytest.approx(190.0)
    assert bool(mask[0]) is True


def test_tvdss_to_owt_and_owt_to_tvdss_are_consistent_round_trip():
    cond = _conditioned([100.0, 200.0, 300.0], [95.0, 190.0, 285.0], [0.10, 0.19, 0.28])
    _, tables = build_axis_conditioned_tables_for_well("W", cond)
    owt, mask1 = tvdss_to_owt([142.5], tables["tvdss_to_owt"])
    tvdss_back, mask2 = owt_to_tvdss(owt, tables["owt_to_tvdss"])
    assert tvdss_back[0] == pytest.approx(142.5, abs=1e-6)
    assert bool(mask1[0]) and bool(mask2[0])


def test_tvdss_to_twt_and_twt_to_tvdss_round_trip():
    cond = _conditioned([100.0, 200.0, 300.0], [95.0, 190.0, 285.0], [0.10, 0.19, 0.28])
    _, tables = build_axis_conditioned_tables_for_well("W", cond)
    twt, _ = tvdss_to_twt([190.0], tables["tvdss_to_owt"])
    assert twt[0] == pytest.approx(0.38)
    tvdss_back, _ = twt_to_tvdss([0.38], tables["owt_to_tvdss"])
    assert tvdss_back[0] == pytest.approx(190.0)


def test_tvdss_to_owt_and_owt_to_tvdss_reject_wrong_direction_table():
    # Guards against passing the OWT->TVDSS table where a TVDSS->OWT table
    # (or vice versa) is required.
    cond = _conditioned([100.0, 200.0, 300.0], [95.0, 190.0, 285.0], [0.10, 0.19, 0.28])
    _, tables = build_axis_conditioned_tables_for_well("W", cond)
    with pytest.raises(TimeDepthError):
        tvdss_to_owt([150.0], tables["owt_to_tvdss"])
    with pytest.raises(TimeDepthError):
        owt_to_tvdss([0.15], tables["tvdss_to_owt"])


def test_tvdss_to_owt_never_extrapolates_outside_table_coverage():
    cond = _conditioned([100.0, 200.0, 300.0], [95.0, 190.0, 285.0], [0.10, 0.19, 0.28])
    _, tables = build_axis_conditioned_tables_for_well("W", cond)
    values, mask = tvdss_to_owt([50.0, 142.5, 400.0], tables["tvdss_to_owt"])
    assert math.isnan(values[0])
    assert not math.isnan(values[1])
    assert math.isnan(values[2])
    assert mask.tolist() == [False, True, False]


# ---------------------------------------------------------------------------
# Order-invariant axis-tie conditioning (Increment 4.1 core fix):
# build_axis_conditioned_lookup_table / build_axis_conditioned_tables_for_well
# ---------------------------------------------------------------------------
def test_tvdss_axis_tie_resolves_to_median_and_is_order_invariant():
    # A TVDSS tie (depths 200 & 300 both read TVDSS=100.0) with DIFFERENT
    # OWT dependent values. Two versions below carry the identical set of
    # tied values but with the assignment across the two physical rows
    # swapped - this reproduces the Increment 4 "keep first, drop later"
    # defect shape: an order-dependent tie-break would give a DIFFERENT
    # answer for version A vs version B (whichever value happened to sit
    # in the row visited "first"); the Increment 4.1 median representative
    # must give the IDENTICAL answer for both.
    depth = np.array([100.0, 200.0, 300.0])
    tvdss = np.array([90.0, 100.0, 100.0])
    owt_a = np.array([0.10, 0.20, 0.21])
    owt_b = np.array([0.10, 0.21, 0.20])  # same tied values, swapped assignment

    entries_a, table_a = build_axis_conditioned_lookup_table(
        "W", "tvdss_to_owt", "TVDSS_conditioned_m", "OWT_conditioned_s", depth, tvdss, owt_a
    )
    entries_b, table_b = build_axis_conditioned_lookup_table(
        "W", "tvdss_to_owt", "TVDSS_conditioned_m", "OWT_conditioned_s", depth, tvdss, owt_b
    )

    assert np.allclose(table_a.axis_values, [90.0, 100.0])
    assert np.allclose(table_a.dependent_values, table_b.dependent_values)
    assert table_a.dependent_values[1] == pytest.approx(0.205)  # median (== mean) of {0.20, 0.21}
    assert len(entries_a) == 1 and len(entries_b) == 1
    assert entries_a[0].tie_kind == "genuinely_non_unique"
    assert entries_a[0].group_size == 2
    assert entries_a[0].dependent_value_spread == pytest.approx(0.01)
    assert entries_a[0].selected_representative_dependent_value == pytest.approx(0.205)
    assert table_a.n_axis_tie_groups == 1
    assert table_a.n_genuinely_nonunique_groups == 1
    assert table_a.n_identical_pairs == 0
    assert table_a.n_collapsed_points == 1
    assert table_a.n_input_points == 3
    assert table_a.n_output_points == 2


def test_owt_axis_tie_resolves_to_median_and_is_order_invariant():
    # Same defect shape as above, but the tie is now on the OWT axis (the
    # independent axis being inverted FROM for owt_to_tvdss) with different
    # TVDSS dependent values.
    depth = np.array([100.0, 200.0, 300.0])
    owt = np.array([0.10, 0.20, 0.20])
    tvdss_a = np.array([90.0, 190.0, 191.0])
    tvdss_b = np.array([90.0, 191.0, 190.0])  # same tied values, swapped assignment

    entries_a, table_a = build_axis_conditioned_lookup_table(
        "W", "owt_to_tvdss", "OWT_conditioned_s", "TVDSS_conditioned_m", depth, owt, tvdss_a
    )
    entries_b, table_b = build_axis_conditioned_lookup_table(
        "W", "owt_to_tvdss", "OWT_conditioned_s", "TVDSS_conditioned_m", depth, owt, tvdss_b
    )

    assert np.allclose(table_a.dependent_values, table_b.dependent_values)
    assert table_a.dependent_values[1] == pytest.approx(190.5)
    assert entries_a[0].tie_kind == "genuinely_non_unique"
    assert entries_a[0].group_size == 2


def test_identical_axis_and_dependent_pair_collapses_without_changing_value():
    # Both the independent axis AND the dependent value are identical
    # across the tied rows: this must still be counted/registered
    # (tie_kind="identical_pair"), never silently merged away unrecorded,
    # even though the representative value is numerically unchanged.
    depth = np.array([100.0, 200.0, 300.0])
    tvdss = np.array([90.0, 100.0, 100.0])
    owt = np.array([0.10, 0.20, 0.20])
    entries, table = build_axis_conditioned_lookup_table(
        "W", "tvdss_to_owt", "TVDSS_conditioned_m", "OWT_conditioned_s", depth, tvdss, owt
    )
    assert len(entries) == 1
    assert entries[0].tie_kind == "identical_pair"
    assert entries[0].dependent_value_spread == 0.0
    assert entries[0].selected_representative_dependent_value == pytest.approx(0.20)
    assert table.n_identical_pairs == 1
    assert table.n_genuinely_nonunique_groups == 0
    assert table.n_collapsed_points == 1


def test_exact_duplicate_axis_and_dependent_rows_are_still_counted():
    # A three-row group, all identical on both axes: group_size must be 3,
    # not silently deduplicated to a smaller count.
    depth = np.array([100.0, 200.0, 300.0, 400.0])
    tvdss = np.array([90.0, 100.0, 100.0, 100.0])
    owt = np.array([0.10, 0.20, 0.20, 0.20])
    entries, table = build_axis_conditioned_lookup_table(
        "W", "tvdss_to_owt", "TVDSS_conditioned_m", "OWT_conditioned_s", depth, tvdss, owt
    )
    assert len(entries) == 1
    assert entries[0].group_size == 3
    assert entries[0].tie_kind == "identical_pair"
    assert table.n_collapsed_points == 2


def test_genuine_reversal_after_grouping_raises_never_sorted_or_discarded():
    # Group values are 100.0 then 90.0 in axis order - a genuine reversal
    # (not a tie), which must raise rather than being sorted, discarded,
    # or otherwise forced monotonic.
    depth = np.array([100.0, 200.0, 300.0])
    tvdss = np.array([100.0, 100.0, 90.0])
    owt = np.array([0.20, 0.21, 0.05])
    with pytest.raises(TimeDepthError):
        build_axis_conditioned_lookup_table(
            "W", "tvdss_to_owt", "TVDSS_conditioned_m", "OWT_conditioned_s", depth, tvdss, owt
        )


def test_axis_conditioned_table_rejects_non_finite_independent_values():
    depth = np.array([100.0, 200.0])
    tvdss = np.array([90.0, np.nan])
    owt = np.array([0.1, 0.2])
    with pytest.raises(TimeDepthError):
        build_axis_conditioned_lookup_table(
            "W", "tvdss_to_owt", "TVDSS_conditioned_m", "OWT_conditioned_s", depth, tvdss, owt
        )


def test_axis_conditioned_table_rejects_non_finite_dependent_values():
    depth = np.array([100.0, 200.0])
    tvdss = np.array([90.0, 100.0])
    owt = np.array([0.1, np.inf])
    with pytest.raises(TimeDepthError):
        build_axis_conditioned_lookup_table(
            "W", "tvdss_to_owt", "TVDSS_conditioned_m", "OWT_conditioned_s", depth, tvdss, owt
        )


def test_axis_conditioned_table_rejects_mismatched_array_lengths():
    depth = np.array([100.0, 200.0, 300.0])
    tvdss = np.array([90.0, 100.0])
    owt = np.array([0.1, 0.2, 0.3])
    with pytest.raises(TimeDepthError):
        build_axis_conditioned_lookup_table(
            "W", "tvdss_to_owt", "TVDSS_conditioned_m", "OWT_conditioned_s", depth, tvdss, owt
        )


def test_build_axis_conditioned_tables_for_well_builds_both_directions():
    cond = _conditioned(
        [100.0, 200.0, 300.0, 400.0], [95.0, 190.0, 190.5, 285.0], [0.10, 0.19, 0.191, 0.28]
    )
    entries, tables = build_axis_conditioned_tables_for_well("W", cond)
    assert set(tables.keys()) == {"tvdss_to_owt", "owt_to_tvdss"}
    assert isinstance(tables["tvdss_to_owt"], AxisConditionedLookupTable)
    assert tables["tvdss_to_owt"].interpolation_direction == "tvdss_to_owt"
    assert tables["owt_to_tvdss"].interpolation_direction == "owt_to_tvdss"
    # No ties on TVDSS or OWT in this fixture -> zero axis-tie entries.
    assert entries == ()
    assert tables["tvdss_to_owt"].n_axis_tie_groups == 0
    assert tables["owt_to_tvdss"].n_axis_tie_groups == 0


# ---------------------------------------------------------------------------
# LAS MD -> checkshot time mapping (partial coverage)
# ---------------------------------------------------------------------------
def test_map_las_md_to_checkshot_time_partial_coverage_never_extrapolates():
    cond = _conditioned([100.0, 200.0, 300.0], [95.0, 190.0, 285.0], [0.10, 0.19, 0.28])
    las_md = np.array([50.0, 150.0, 250.0, 350.0])
    owt_mapped, twt_mapped, summary = map_las_md_to_checkshot_time("W", las_md, cond)
    assert math.isnan(owt_mapped[0])  # shallower than coverage
    assert not math.isnan(owt_mapped[1])
    assert not math.isnan(owt_mapped[2])
    assert math.isnan(owt_mapped[3])  # deeper than coverage
    assert summary.n_extrapolated == 0
    assert summary.n_inside_coverage == 2
    assert summary.n_shallower_than_coverage == 1
    assert summary.n_deeper_than_coverage == 1
    assert summary.mapped_fraction == pytest.approx(0.5)


def test_map_las_md_rejects_non_finite_input():
    cond = _conditioned([100.0, 200.0], [95.0, 190.0], [0.10, 0.19])
    with pytest.raises(TimeDepthError):
        map_las_md_to_checkshot_time("W", np.array([100.0, np.nan]), cond)


# ---------------------------------------------------------------------------
# Sonic-checkshot drift
# ---------------------------------------------------------------------------
def test_find_longest_finite_positive_run_picks_longest():
    md = np.arange(10.0)
    vp = np.array([np.nan, 1.0, 2.0, np.nan, np.nan, 3.0, 4.0, 5.0, np.nan, np.nan])
    start, end = find_longest_finite_positive_run(md, vp)
    assert (start, end) == (5, 7)


def test_find_longest_finite_positive_run_raises_when_none_valid():
    md = np.arange(5.0)
    vp = np.full(5, np.nan)
    with pytest.raises(TimeDepthError):
        find_longest_finite_positive_run(md, vp)


def test_sonic_checkshot_drift_known_constant_velocity_matches_analytic():
    # Constant VP => sonic transit time = (md2-md1)/VP exactly (trapezoidal
    # integration of a constant is exact).
    md = np.linspace(0.0, 1000.0, 1001)
    vp = np.full(md.shape, 2000.0)  # m/s
    cond = _conditioned([0.0, 1000.0], [0.0, 1000.0], [0.0, 1000.0 / 2000.0])
    drift = compute_sonic_checkshot_drift("Poseidon_2", md, vp, cond)
    assert drift.sonic_transit_time_s == pytest.approx(1000.0 / 2000.0, rel=1e-9)
    assert drift.checkshot_owt_increment_s == pytest.approx(1000.0 / 2000.0, rel=1e-9)
    assert drift.sonic_minus_checkshot_ms == pytest.approx(0.0, abs=1e-6)
    assert drift.checkshot_minus_sonic_ms == pytest.approx(0.0, abs=1e-6)


def test_sonic_checkshot_drift_reports_both_signs_and_percent():
    md = np.linspace(0.0, 100.0, 101)
    vp = np.full(md.shape, 1000.0)  # transit time = 0.1 s
    cond = _conditioned([0.0, 100.0], [0.0, 100.0], [0.0, 0.09])  # checkshot increment 0.09 s
    drift = compute_sonic_checkshot_drift("Poseidon_2", md, vp, cond)
    assert drift.sonic_minus_checkshot_ms == pytest.approx(-drift.checkshot_minus_sonic_ms)
    assert drift.sonic_minus_checkshot_percent == pytest.approx(
        (0.1 - 0.09) / 0.09 * 100.0, rel=1e-6
    )


def test_sonic_checkshot_drift_rejects_interval_beyond_checkshot_coverage():
    md = np.linspace(0.0, 1000.0, 1001)
    vp = np.full(md.shape, 2000.0)
    cond = _conditioned([200.0, 500.0], [190.0, 480.0], [0.19, 0.45])  # narrower than sonic run
    with pytest.raises(TimeDepthError):
        compute_sonic_checkshot_drift("Poseidon_2", md, vp, cond)


def test_sonic_checkshot_drift_rejects_decreasing_md_never_negative_transit_time():
    # Regression (Increment 4.1): the pre-4.1 implementation validated none
    # of this and could silently integrate a decreasing MD run into a
    # physically invalid NEGATIVE "transit time". It must now raise
    # TimeDepthError instead of returning any result.
    md = np.array([100.0, 200.0, 150.0, 250.0])  # decreasing at index 1->2
    vp = np.full(md.shape, 2000.0)
    cond = _conditioned([0.0, 300.0], [0.0, 300.0], [0.0, 0.15])
    with pytest.raises(TimeDepthError):
        compute_sonic_checkshot_drift("Poseidon_2", md, vp, cond)


def test_sonic_checkshot_drift_rejects_duplicate_md():
    md = np.array([100.0, 200.0, 200.0, 300.0])
    vp = np.full(md.shape, 2000.0)
    cond = _conditioned([0.0, 300.0], [0.0, 300.0], [0.0, 0.15])
    with pytest.raises(TimeDepthError):
        compute_sonic_checkshot_drift("Poseidon_2", md, vp, cond)


def test_sonic_checkshot_drift_rejects_non_finite_md():
    md = np.array([100.0, np.nan, 300.0])
    vp = np.full(md.shape, 2000.0)
    cond = _conditioned([0.0, 300.0], [0.0, 300.0], [0.0, 0.15])
    with pytest.raises(TimeDepthError):
        compute_sonic_checkshot_drift("Poseidon_2", md, vp, cond)


def test_sonic_checkshot_drift_rejects_incompatible_md_vp_shapes():
    md = np.array([100.0, 200.0, 300.0])
    vp = np.array([2000.0, 2000.0])  # length mismatch
    cond = _conditioned([0.0, 300.0], [0.0, 300.0], [0.0, 0.15])
    with pytest.raises(TimeDepthError):
        compute_sonic_checkshot_drift("Poseidon_2", md, vp, cond)


def test_sonic_checkshot_drift_rejects_non_positive_checkshot_owt_increment():
    md = np.linspace(100.0, 300.0, 21)
    vp = np.full(md.shape, 2000.0)
    # Checkshot OWT is flat (zero increment) across the identical Depth
    # interval used for comparison.
    cond = _conditioned([100.0, 300.0], [100.0, 300.0], [0.15, 0.15])
    with pytest.raises(TimeDepthError):
        compute_sonic_checkshot_drift("Poseidon_2", md, vp, cond)


# ---------------------------------------------------------------------------
# Increment 4.1.1 - Blocking Defect 1: hidden reversal via global exact-
# value grouping in build_axis_conditioned_lookup_table
# ---------------------------------------------------------------------------
def test_hidden_reversal_returning_to_seen_value_raises():
    # [100.0, 200.0, 100.0]: the pre-4.1.1 implementation grouped ALL
    # occurrences of 100.0 together globally BEFORE checking for a
    # reversal, producing an apparently-valid [100.0, 200.0] and silently
    # hiding the genuine 200 -> 100 reversal. This must now raise.
    depth = np.array([100.0, 200.0, 300.0])
    independent = np.array([100.0, 200.0, 100.0])
    dependent = np.array([1.0, 2.0, 3.0])
    with pytest.raises(TimeDepthError):
        build_axis_conditioned_lookup_table(
            "W", "tvdss_to_owt", "TVDSS_conditioned_m", "OWT_conditioned_s",
            depth, independent, dependent,
        )


def test_hidden_reversal_with_adjacent_tie_before_it_raises():
    # [100.0, 200.0, 150.0, 150.0, 300.0]: a genuine reversal (200 -> 150)
    # immediately followed by a valid adjacent tie (150.0, 150.0) and then
    # a further increase. The reversal must still be caught even though a
    # legitimate adjacent tie appears later in the sequence.
    depth = np.array([100.0, 200.0, 300.0, 400.0, 500.0])
    independent = np.array([100.0, 200.0, 150.0, 150.0, 300.0])
    dependent = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
    with pytest.raises(TimeDepthError):
        build_axis_conditioned_lookup_table(
            "W", "tvdss_to_owt", "TVDSS_conditioned_m", "OWT_conditioned_s",
            depth, independent, dependent,
        )


def test_valid_leading_adjacent_tie_remains_valid_and_order_invariant():
    # [100.0, 100.0, 200.0]: a genuine adjacent tie at the START of the
    # sequence, not a reversal - must remain valid (Increment 4.1.1 must
    # not regress this legitimate case) and order-invariant across which
    # of the two tied rows carries which dependent value.
    depth = np.array([100.0, 200.0, 300.0])
    independent = np.array([100.0, 100.0, 200.0])
    dependent_a = np.array([1.0, 2.0, 3.0])
    dependent_b = np.array([2.0, 1.0, 3.0])  # same tied values, swapped assignment

    entries_a, table_a = build_axis_conditioned_lookup_table(
        "W", "tvdss_to_owt", "TVDSS_conditioned_m", "OWT_conditioned_s",
        depth, independent, dependent_a,
    )
    entries_b, table_b = build_axis_conditioned_lookup_table(
        "W", "tvdss_to_owt", "TVDSS_conditioned_m", "OWT_conditioned_s",
        depth, independent, dependent_b,
    )
    assert np.allclose(table_a.axis_values, [100.0, 200.0])
    assert np.allclose(table_a.dependent_values, table_b.dependent_values)
    assert table_a.dependent_values[0] == pytest.approx(1.5)  # median (== mean) of {1.0, 2.0}
    assert len(entries_a) == 1
    assert entries_a[0].group_size == 2


def test_valid_trailing_adjacent_tie_remains_valid_and_order_invariant():
    # [100.0, 200.0, 200.0, 300.0]: a genuine adjacent tie in the MIDDLE of
    # the sequence, not a reversal - must remain valid and order-invariant.
    depth = np.array([100.0, 200.0, 300.0, 400.0])
    independent = np.array([100.0, 200.0, 200.0, 300.0])
    dependent_a = np.array([1.0, 2.0, 3.0, 4.0])
    dependent_b = np.array([1.0, 3.0, 2.0, 4.0])  # same tied values, swapped assignment

    entries_a, table_a = build_axis_conditioned_lookup_table(
        "W", "tvdss_to_owt", "TVDSS_conditioned_m", "OWT_conditioned_s",
        depth, independent, dependent_a,
    )
    entries_b, table_b = build_axis_conditioned_lookup_table(
        "W", "tvdss_to_owt", "TVDSS_conditioned_m", "OWT_conditioned_s",
        depth, independent, dependent_b,
    )
    assert np.allclose(table_a.axis_values, [100.0, 200.0, 300.0])
    assert np.allclose(table_a.dependent_values, table_b.dependent_values)
    assert table_a.dependent_values[1] == pytest.approx(2.5)  # median (== mean) of {2.0, 3.0}
    assert len(entries_a) == 1
    assert entries_a[0].group_size == 2


def test_axis_conditioned_table_rejects_non_finite_or_non_increasing_depth_conditioned():
    # Increment 4.1.1 hardening: depth_conditioned_m itself is now
    # validated finite and strictly increasing on entry.
    independent = np.array([100.0, 200.0])
    dependent = np.array([1.0, 2.0])
    with pytest.raises(TimeDepthError):
        build_axis_conditioned_lookup_table(
            "W", "tvdss_to_owt", "TVDSS_conditioned_m", "OWT_conditioned_s",
            np.array([100.0, np.nan]), independent, dependent,
        )
    with pytest.raises(TimeDepthError):
        build_axis_conditioned_lookup_table(
            "W", "tvdss_to_owt", "TVDSS_conditioned_m", "OWT_conditioned_s",
            np.array([200.0, 100.0]), independent, dependent,
        )


# ---------------------------------------------------------------------------
# Increment 4.1.1 - Blocking Defect 2: full-canonical-MD validation must
# not be limited to the selected finite-positive-VP run
# ---------------------------------------------------------------------------
def test_sonic_checkshot_drift_rejects_decreasing_md_outside_selected_run():
    # Full md_m = [0.0, 1.0, 0.0]: the final station is a decreasing MD
    # step, but its VP_m_s is NaN so it would be EXCLUDED from the
    # selected finite-positive-VP run (indices 0-1 only). The pre-4.1.1
    # implementation validated monotonicity only within that selected
    # run and would not have caught this. The COMPLETE canonical md_m
    # array must now be validated before run-selection.
    md = np.array([0.0, 1.0, 0.0])
    vp = np.array([2000.0, 2000.0, np.nan])
    cond = _conditioned([0.0, 1.0], [0.0, 1.0], [0.0, 0.0005])
    with pytest.raises(TimeDepthError):
        compute_sonic_checkshot_drift("Poseidon_2", md, vp, cond)


def test_sonic_checkshot_drift_rejects_duplicate_md_outside_selected_run():
    # Full md_m = [0.0, 1.0, 1.0]: the final station duplicates the prior
    # MD value, but its VP_m_s is NaN so it would be excluded from the
    # selected run (indices 0-1 only). Must still raise.
    md = np.array([0.0, 1.0, 1.0])
    vp = np.array([2000.0, 2000.0, np.nan])
    cond = _conditioned([0.0, 1.0], [0.0, 1.0], [0.0, 0.0005])
    with pytest.raises(TimeDepthError):
        compute_sonic_checkshot_drift("Poseidon_2", md, vp, cond)


def test_find_longest_finite_positive_run_rejects_non_1d_input():
    with pytest.raises(TimeDepthError):
        find_longest_finite_positive_run(np.array([[1.0, 2.0]]), np.array([[1.0, 2.0]]))
    with pytest.raises(TimeDepthError):
        find_longest_finite_positive_run(np.arange(4.0), np.array([[1.0, 2.0], [3.0, 4.0]]))


def test_find_longest_finite_positive_run_rejects_mismatched_shapes():
    with pytest.raises(TimeDepthError):
        find_longest_finite_positive_run(np.arange(4.0), np.arange(3.0))


# ---------------------------------------------------------------------------
# Increment 4.1.1 - Blocking Defect 3: zero in-coverage checkshot rows
# must raise a typed TimeDepthError, never an untyped NumPy ValueError
# ---------------------------------------------------------------------------
def test_compare_checkshot_to_survey_raises_typed_error_when_zero_rows_in_coverage():
    survey_md = np.array([0.0, 100.0, 200.0])
    survey_tvd = np.array([0.0, 95.0, 190.0])
    depth_source = np.array([500.0, 600.0])  # both entirely outside survey MD coverage
    tvdss_source = np.array([499.0, 599.0])
    with pytest.raises(TimeDepthError) as excinfo:
        compare_checkshot_to_survey(
            "Boreas_1", depth_source, tvdss_source, survey_md, survey_tvd, datum_elevation_m=0.0,
            depth_basis_interpretation_status="candidate_md_evaluated_against_locked_survey",
        )
    message = str(excinfo.value)
    assert "Boreas_1" in message
    assert "500.0000" in message and "600.0000" in message  # checkshot Depth range
    assert "0.0000" in message and "200.0000" in message  # survey MD coverage
    assert "no comparison was" in message
    assert "no extrapolation was attempted" in message


# ---------------------------------------------------------------------------
# Increment 4.1.1 - unit-helper input-safety audit: seconds_to_milliseconds
# / milliseconds_to_seconds must not silently coerce boolean, string, or
# complex input, mirroring the p2mem.units Increment-1 policy
# ---------------------------------------------------------------------------
def test_seconds_to_milliseconds_rejects_boolean_input():
    with pytest.raises(TypeError):
        seconds_to_milliseconds(True)
    with pytest.raises(TypeError):
        seconds_to_milliseconds(np.array([True, False]))


def test_seconds_to_milliseconds_rejects_string_input():
    with pytest.raises(TypeError):
        seconds_to_milliseconds("3.5")
    with pytest.raises(TypeError):
        seconds_to_milliseconds(np.array(["1.0", "2.0"]))


def test_seconds_to_milliseconds_rejects_complex_input():
    with pytest.raises(TypeError):
        seconds_to_milliseconds(1.0 + 2.0j)
    with pytest.raises(TypeError):
        seconds_to_milliseconds(np.array([1.0 + 2.0j]))


def test_milliseconds_to_seconds_rejects_boolean_string_complex_input():
    with pytest.raises(TypeError):
        milliseconds_to_seconds(False)
    with pytest.raises(TypeError):
        milliseconds_to_seconds("1500")
    with pytest.raises(TypeError):
        milliseconds_to_seconds(np.array(["1500.0"]))
    with pytest.raises(TypeError):
        milliseconds_to_seconds(3.0 - 1.0j)


def test_seconds_milliseconds_still_accept_valid_scalars_arrays_and_nan():
    # Increment 4.1.1 must not regress any previously valid input: Python
    # numeric scalars, NumPy numeric arrays, and NaN propagation.
    assert seconds_to_milliseconds(1.5) == pytest.approx(1500.0)
    assert milliseconds_to_seconds(1500) == pytest.approx(1.5)
    arr = np.array([0.5, 1.0, np.nan])
    ms = seconds_to_milliseconds(arr)
    assert np.allclose(ms[:2], [500.0, 1000.0])
    assert math.isnan(ms[2])
    int_arr = np.array([1, 2, 3], dtype=np.int64)
    assert np.allclose(seconds_to_milliseconds(int_arr), [1000.0, 2000.0, 3000.0])


In [ ]:
%%writefile tests/test_checkshot_inventory.py
"""
tests/test_checkshot_inventory.py - Validation suite for
p2mem.io.checkshot_inventory (Increment 4: deterministic, metadata-only
inventory-table builders).

PORTABLE unit tests only: hand-built typed result objects, never the real
project files. Verifies (a) correct row content, (b) deterministic/sorted
ordering, (c) environment-independent output (no absolute path leakage in
a failure row's sanitized message), and (d) Tier C classification and
well-status disclosure in the JSON manifest.
"""

import numpy as np
import pytest

from p2mem.checkshot_models import (
    AxisConditionedLookupTable,
    AxisTimeDepthTieRegisterEntry,
    CheckshotAvailabilityRecord,
    CheckshotFileContract,
    CheckshotHeaderInfo,
    CheckshotIngestionFailure,
    CheckshotIngestionIssue,
    CheckshotStationData,
    CheckshotSurveyDepthComparisonResult,
    CheckshotWellResult,
    ConditionedCheckshotData,
    DuplicateTieRegisterEntry,
    SonicCheckshotDriftResult,
    TimeDepthMappingSummary,
    VelocityDiagnosticsResult,
)
from p2mem.io.checkshot_inventory import (
    build_checkshot_depth_tie_qc_rows,
    build_checkshot_file_inventory_rows,
    build_checkshot_issues_rows,
    build_checkshot_time_axis_tie_register_rows,
    build_checkshot_time_depth_manifest,
    build_checkshot_velocity_summary_rows,
    build_duplicate_tie_register_rows,
    build_sonic_checkshot_drift_rows,
    build_time_depth_mapping_rows,
)


def _make_contract(project_well_key, identity_status, model_use) -> CheckshotFileContract:
    return CheckshotFileContract(
        source_filename=f"{project_well_key}-Checkshot.txt",
        expected_sha256="a" * 64,
        project_well_key=project_well_key,
        well_identity_evidence_status=identity_status,
        well_identity_evidence_notes="test",
        model_use_status=model_use,
        expected_survey_statement="stmt",
        expected_column_header_line="Depth\tTVDSS\tOWT(sec)",
        expected_column_order=("Depth", "TVDSS", "OWT(sec)"),
        expected_column_count=3,
        expected_time_type="OWT",
        expected_time_unit="s",
        expected_vertical_correction_fragment="vertically corrected",
        expected_srd_reference_fragment="SRD",
        expected_row_count=3,
        expected_depth_min_m=100.0,
        expected_depth_max_m=300.0,
        expected_tvdss_min_m=95.0,
        expected_tvdss_max_m=285.0,
        expected_owt_min_s=0.1,
        expected_owt_max_s=0.28,
        numeric_range_tolerance=0.0001,
        depth_basis_interpretation_status="candidate_md_evaluated_against_locked_survey",
        duplicate_tie_policy="test_policy",
        interpolation_policy="piecewise_linear_no_extrapolation",
        extrapolation_policy="disallowed_nan_outside_coverage",
        notes="test",
    )


def _make_result(well_key, identity_status="verified", model_use="primary_model") -> CheckshotWellResult:
    contract = _make_contract(well_key, identity_status, model_use)
    header = CheckshotHeaderInfo(
        source_filename=contract.source_filename,
        sha256="a" * 64,
        survey_statement="stmt",
        column_header_line="Depth\tTVDSS\tOWT(sec)",
        column_names=("Depth", "TVDSS", "OWT(sec)"),
        line_ending_convention="CRLF",
    )
    raw = CheckshotStationData(
        Depth_source_m=np.array([100.0, 200.0, 300.0]),
        TVDSS_source_m=np.array([95.0, 190.0, 285.0]),
        OWT_source_s=np.array([0.10, 0.19, 0.28]),
    )
    conditioned = ConditionedCheckshotData(
        Depth_conditioned_m=raw.Depth_source_m,
        TVDSS_conditioned_m=raw.TVDSS_source_m,
        OWT_conditioned_s=raw.OWT_source_s,
        n_raw_rows=3, n_conditioned_rows=3, n_tie_groups=0,
        conditioning_method="test",
    )
    velocity = VelocityDiagnosticsResult(
        well_key=well_key,
        Depth_conditioned_m=conditioned.Depth_conditioned_m,
        Vavg_m_s=np.array([950.0, 1000.0, 1017.857]),
        Vint_m_s=np.array([950.0, 1055.556]),
        vint_invalid_mask=np.array([False, False]),
        n_vint_intervals=2, n_vint_invalid=0,
        vint_invalid_reason_counts={"non_positive_delta_owt": 0, "non_positive_delta_tvdss": 0},
    )
    comparison = CheckshotSurveyDepthComparisonResult(
        well_key=well_key, n_compared=3, n_outside_survey_md_coverage=0,
        min_residual_m=-0.05, max_residual_m=0.05, max_abs_residual_m=0.05,
        mean_residual_m=0.0, median_residual_m=0.0, rmse_m=0.03,
        first_residual_m=0.05, last_residual_m=-0.05,
        residual_trend_description="near-constant residual across the compared depth interval",
        depth_basis_interpretation_status="candidate_md_evaluated_against_locked_survey",
        residual_sign_convention="residual_m = TVDSS_survey_interpolated_m - TVDSS_source_m",
    )
    issues = (
        CheckshotIngestionIssue("WARNING", "DEPTH_BASIS_NOT_EXPLICITLY_DECLARED", "msg", header.source_filename),
    )
    return CheckshotWellResult(
        header=header, contract=contract, raw=raw, duplicate_ties=(), conditioned=conditioned,
        velocity=velocity, depth_comparison=comparison, issues=issues, contract_status="PASSED",
    )


# ---------------------------------------------------------------------------
# File inventory rows
# ---------------------------------------------------------------------------
def test_file_inventory_rows_include_results_failures_and_availability_sorted():
    results = {"Boreas_1": _make_result("Boreas_1")}
    failures = {
        "Broken_Well": CheckshotIngestionFailure(
            well_key="Broken_Well", source_path="/abs/path/to/Broken_Well-Checkshot.txt",
            error_type="parsing_failure", message="/abs/path/to/Broken_Well-Checkshot.txt: bad row",
            exception=ValueError("bad row"),
        )
    }
    availability = {
        "Poseidon_North_1": CheckshotAvailabilityRecord(
            "Poseidon_North_1", "NOT_AVAILABLE", "No approved checkshot file exists."
        )
    }
    rows = build_checkshot_file_inventory_rows(results, failures, availability)
    well_keys = [r["well_key"] for r in rows]
    assert well_keys == sorted(well_keys)
    assert well_keys == ["Boreas_1", "Broken_Well", "Poseidon_North_1"]

    boreas_row = next(r for r in rows if r["well_key"] == "Boreas_1")
    assert boreas_row["contract_status"] == "PASSED"
    assert boreas_row["n_raw_rows"] == 3

    broken_row = next(r for r in rows if r["well_key"] == "Broken_Well")
    assert broken_row["contract_status"] == "FAILED"
    assert "/abs/path/to/" not in broken_row["error_message"]
    assert "Broken_Well-Checkshot.txt" in broken_row["error_message"]

    na_row = next(r for r in rows if r["well_key"] == "Poseidon_North_1")
    assert na_row["contract_status"] == "NOT_AVAILABLE"


def test_issues_rows_sanitize_absolute_paths():
    failures = {
        "Well_X": CheckshotIngestionFailure(
            well_key="Well_X", source_path="/home/user/secret_build_dir/Well_X-Checkshot.txt",
            error_type="file_not_found",
            message="/home/user/secret_build_dir/Well_X-Checkshot.txt not found",
            exception=FileNotFoundError(),
        )
    }
    rows = build_checkshot_issues_rows({}, failures)
    assert len(rows) == 1
    assert "/home/user/secret_build_dir/" not in rows[0]["message"]
    assert rows[0]["source_filename"] == "Well_X-Checkshot.txt"


def test_issues_rows_include_well_issues():
    results = {"Poseidon_2": _make_result("Poseidon_2")}
    rows = build_checkshot_issues_rows(results, {})
    assert len(rows) == 1
    assert rows[0]["code"] == "DEPTH_BASIS_NOT_EXPLICITLY_DECLARED"
    assert rows[0]["severity"] == "WARNING"


# ---------------------------------------------------------------------------
# Duplicate tie register rows
# ---------------------------------------------------------------------------
def test_duplicate_tie_register_rows_serialize_tuples_as_delimited_strings():
    result = _make_result("Poseidon_2")
    tie = DuplicateTieRegisterEntry(
        well_key="Poseidon_2", axis="Depth", tie_value_m=200.0,
        source_row_indices=(1, 2), original_tvdss_m=(190.0, 190.5), original_owt_s=(0.19, 0.191),
        duplicate_type="repeated_depth_distinct_tvdss_owt", group_size=2,
        tvdss_value_spread_m=0.5, owt_value_spread_s=0.001,
        selected_representative_tvdss_m=190.25, selected_representative_owt_s=0.1905,
        conditioning_rule="median of the tied group", affected_downstream_outputs=("a", "b"),
    )
    from dataclasses import replace
    result = replace(result, duplicate_ties=(tie,))
    rows = build_duplicate_tie_register_rows({"Poseidon_2": result})
    assert len(rows) == 1
    assert rows[0]["source_row_indices"] == "1;2"
    assert rows[0]["original_tvdss_m"] == "190.0;190.5"
    assert rows[0]["affected_downstream_outputs"] == "a;b"


# ---------------------------------------------------------------------------
# Axis-tie (TVDSS/OWT) register rows (Increment 4.1)
# ---------------------------------------------------------------------------
def _make_axis_tie_entry(well_key="Poseidon_2", direction="tvdss_to_owt"):
    axis, dependent = (
        ("TVDSS_conditioned_m", "OWT_conditioned_s")
        if direction == "tvdss_to_owt"
        else ("OWT_conditioned_s", "TVDSS_conditioned_m")
    )
    return AxisTimeDepthTieRegisterEntry(
        well_key=well_key,
        interpolation_direction=direction,
        axis=axis,
        dependent_axis=dependent,
        tie_axis_value=100.0,
        conditioned_row_indices=(1, 2),
        associated_depth_m=(200.0, 300.0),
        original_dependent_values=(0.20, 0.21),
        group_size=2,
        dependent_value_spread=0.01,
        selected_representative_dependent_value=0.205,
        conditioning_rule="median of the tied group's values, order-invariant",
        tie_kind="genuinely_non_unique",
        affected_downstream_outputs=(f"{direction}_lookup_table", "forward_inverse_time_depth_interpolation"),
    )


def test_axis_tie_register_rows_serialize_tuples_as_delimited_strings():
    from dataclasses import replace

    result = _make_result("Poseidon_2")
    entry = _make_axis_tie_entry("Poseidon_2")
    result = replace(result, axis_tie_entries=(entry,))
    rows = build_checkshot_time_axis_tie_register_rows({"Poseidon_2": result})
    assert len(rows) == 1
    row = rows[0]
    assert row["well_key"] == "Poseidon_2"
    assert row["interpolation_direction"] == "tvdss_to_owt"
    assert row["conditioned_row_indices"] == "1;2"
    assert row["associated_depth_m"] == "200.0;300.0"
    assert row["original_dependent_values"] == "0.2;0.21"
    assert row["group_size"] == 2
    assert row["tie_kind"] == "genuinely_non_unique"
    assert row["affected_downstream_outputs"] == "tvdss_to_owt_lookup_table;forward_inverse_time_depth_interpolation"


def test_axis_tie_register_rows_sorted_by_well_key_and_empty_when_no_ties():
    from dataclasses import replace

    result_a = replace(_make_result("Boreas_1"), axis_tie_entries=(_make_axis_tie_entry("Boreas_1"),))
    result_b = _make_result("Poseidon_2")  # default axis_tie_entries=() -- no ties
    rows = build_checkshot_time_axis_tie_register_rows({"Poseidon_2": result_b, "Boreas_1": result_a})
    assert len(rows) == 1
    assert rows[0]["well_key"] == "Boreas_1"


def test_axis_tie_register_rows_deterministic_and_json_serializable_no_absolute_paths():
    import json
    from dataclasses import replace

    result = replace(_make_result("Poseidon_2"), axis_tie_entries=(_make_axis_tie_entry("Poseidon_2"),))
    rows1 = build_checkshot_time_axis_tie_register_rows({"Poseidon_2": result})
    rows2 = build_checkshot_time_axis_tie_register_rows({"Poseidon_2": result})
    assert json.dumps(rows1, sort_keys=True) == json.dumps(rows2, sort_keys=True)
    serialized = json.dumps(rows1)
    assert "/home/" not in serialized and "/root/" not in serialized and "C:\\" not in serialized


# ---------------------------------------------------------------------------
# Per-well axis-tie-conditioning summary embedded in the manifest
# (Increment 4.1) - distinct from n_tie_groups (Depth-axis ties only)
# ---------------------------------------------------------------------------
def test_manifest_axis_tie_conditioning_summary_defaults_to_zero_counts_when_no_axis_tables():
    results = {"Poseidon_2": _make_result("Poseidon_2")}  # axis_tables defaults to {}
    manifest = build_checkshot_time_depth_manifest(results, {}, {}, {}, {})
    summary = manifest["wells"]["Poseidon_2"]["axis_tie_conditioning"]
    assert summary["n_tvdss_axis_tie_groups"] == 0
    assert summary["n_owt_axis_tie_groups"] == 0
    assert "order-invariant" in summary["axis_tie_conditioning_policy"]


def test_manifest_axis_tie_conditioning_summary_reflects_axis_tables():
    from dataclasses import replace

    table_to_owt = AxisConditionedLookupTable(
        well_key="Poseidon_2", interpolation_direction="tvdss_to_owt",
        independent_axis_name="TVDSS_conditioned_m", dependent_axis_name="OWT_conditioned_s",
        axis_values=np.array([90.0, 100.0]), dependent_values=np.array([0.10, 0.205]),
        n_input_points=3, n_output_points=2, n_axis_tie_groups=1,
        n_collapsed_points=1, n_identical_pairs=0, n_genuinely_nonunique_groups=1,
        conditioning_method="axis_tie_median_representative_order_invariant_v1",
    )
    table_to_tvdss = AxisConditionedLookupTable(
        well_key="Poseidon_2", interpolation_direction="owt_to_tvdss",
        independent_axis_name="OWT_conditioned_s", dependent_axis_name="TVDSS_conditioned_m",
        axis_values=np.array([0.10, 0.205]), dependent_values=np.array([90.0, 100.0]),
        n_input_points=3, n_output_points=2, n_axis_tie_groups=0,
        n_collapsed_points=0, n_identical_pairs=0, n_genuinely_nonunique_groups=0,
        conditioning_method="axis_tie_median_representative_order_invariant_v1",
    )
    result = replace(
        _make_result("Poseidon_2"),
        axis_tables={"tvdss_to_owt": table_to_owt, "owt_to_tvdss": table_to_tvdss},
    )
    manifest = build_checkshot_time_depth_manifest({"Poseidon_2": result}, {}, {}, {}, {})
    summary = manifest["wells"]["Poseidon_2"]["axis_tie_conditioning"]
    assert summary["n_tvdss_axis_tie_groups"] == 1
    assert summary["n_tvdss_axis_genuinely_nonunique_groups"] == 1
    assert summary["n_owt_axis_tie_groups"] == 0


# ---------------------------------------------------------------------------
# Depth-tie QC rows
# ---------------------------------------------------------------------------
def test_depth_tie_qc_rows_skip_none_comparison():
    from dataclasses import replace
    result = _make_result("Boreas_1")
    result_no_comparison = replace(result, depth_comparison=None)
    rows = build_checkshot_depth_tie_qc_rows({"Boreas_1": result_no_comparison})
    assert rows == []


def test_depth_tie_qc_rows_content():
    result = _make_result("Boreas_1")
    rows = build_checkshot_depth_tie_qc_rows({"Boreas_1": result})
    assert len(rows) == 1
    assert rows[0]["well_key"] == "Boreas_1"
    assert rows[0]["max_abs_residual_m"] == 0.05


# ---------------------------------------------------------------------------
# Velocity summary rows
# ---------------------------------------------------------------------------
def test_velocity_summary_rows_aggregate_statistics():
    result = _make_result("Proteus_1ST2")
    rows = build_checkshot_velocity_summary_rows({"Proteus_1ST2": result})
    assert len(rows) == 1
    row = rows[0]
    assert row["n_conditioned_rows"] == 3
    assert row["n_vint_intervals"] == 2
    assert row["n_vint_invalid"] == 0
    assert row["vavg_min_m_s"] == pytest.approx(950.0)


# ---------------------------------------------------------------------------
# Sonic drift / time-depth mapping rows
# ---------------------------------------------------------------------------
def test_sonic_drift_rows_join_limitations():
    drift = SonicCheckshotDriftResult(
        well_key="Poseidon_2", md_interval_start_m=2448.6, md_interval_end_m=4064.2,
        n_sonic_samples=10602, selection_criteria="longest run", integration_method="trapezoidal",
        sonic_transit_time_s=0.3913, checkshot_owt_increment_s=0.3746,
        sonic_minus_checkshot_ms=16.66, checkshot_minus_sonic_ms=-16.66,
        sonic_minus_checkshot_percent=4.45,
        limitations=("limitation one", "limitation two"),
    )
    rows = build_sonic_checkshot_drift_rows({"Poseidon_2": drift})
    assert rows[0]["limitations"] == "limitation one | limitation two"


def test_time_depth_mapping_rows_content():
    summary = TimeDepthMappingSummary(
        well_key="Poseidon_2", n_las_samples=31897, n_inside_coverage=22521,
        n_shallower_than_coverage=5104, n_deeper_than_coverage=4272,
        mapped_fraction=0.706, checkshot_depth_min_m=1267.8, checkshot_depth_max_m=4700.0,
        las_md_min_m=490.0, las_md_max_m=5350.9507,
        interpolation_method="piecewise_linear_no_extrapolation", n_extrapolated=0,
    )
    rows = build_time_depth_mapping_rows({"Poseidon_2": summary})
    assert rows[0]["n_extrapolated"] == 0
    assert rows[0]["mapped_fraction"] == pytest.approx(0.706)


# ---------------------------------------------------------------------------
# Manifest
# ---------------------------------------------------------------------------
def test_manifest_carries_tier_c_and_well_statuses():
    results = {
        "Poseidon_2": _make_result("Poseidon_2", "verified", "primary_model"),
        "Boreas_1": _make_result("Boreas_1", "verified", "qc_only"),
        "Proteus_1ST2": _make_result("Proteus_1ST2", "inferred_unverified", "qc_only"),
    }
    availability = {
        "Poseidon_North_1": CheckshotAvailabilityRecord(
            "Poseidon_North_1", "NOT_AVAILABLE", "No approved checkshot file exists."
        )
    }
    manifest = build_checkshot_time_depth_manifest(results, {}, availability, {}, {})
    assert manifest["tier_classification"].startswith("Tier C")
    assert manifest["wells"]["Poseidon_2"]["model_use_status"] == "primary_model"
    assert manifest["wells"]["Boreas_1"]["model_use_status"] == "qc_only"
    assert manifest["wells"]["Proteus_1ST2"]["well_identity_evidence_status"] == "inferred_unverified"
    assert manifest["wells"]["Poseidon_North_1"]["checkshot_availability"] == "NOT_AVAILABLE"
    assert manifest["n_wells_checkshot_available"] == 3
    assert manifest["n_wells_checkshot_not_available"] == 1


def test_manifest_is_json_serializable_and_deterministic():
    import json
    results = {"Poseidon_2": _make_result("Poseidon_2")}
    m1 = build_checkshot_time_depth_manifest(results, {}, {}, {}, {})
    m2 = build_checkshot_time_depth_manifest(results, {}, {}, {}, {})
    assert json.dumps(m1, sort_keys=True) == json.dumps(m2, sort_keys=True)


def test_manifest_sanitizes_failed_well_paths():
    failures = {
        "Well_X": CheckshotIngestionFailure(
            well_key="Well_X", source_path="/home/user/build/Well_X-Checkshot.txt",
            error_type="file_not_found", message="/home/user/build/Well_X-Checkshot.txt missing",
            exception=FileNotFoundError(),
        )
    }
    manifest = build_checkshot_time_depth_manifest({}, failures, {}, {}, {})
    assert "/home/user/build/" not in manifest["failed_wells"]["Well_X"]["message"]


#### Step 8 — Install the package in editable mode

**Technical objective:** (re)install `p2mem` from the just-written source tree so the notebook's Python kernel imports the exact code just written.

In [ ]:
%cd /content/drive/MyDrive/Poseidon_1D_MEM
!pip install -q -e .


#### Step 9 — Run the complete unit-test suite (Increments 1.1 through 4.1.1 together)

**Technical objective:** confirm every existing test still passes and every new Increment 4 / 4.1 / 4.1.1 test passes, in one combined run. The actual reported pass count is read from this cell's own output — never assumed.

**Increment 4.1.2 correction:** the prior version of this cell ran `!pytest -v` as a bare shell-escape line. Jupyter/Colab executes a `!`-prefixed shell command and unconditionally continues to the next cell regardless of that command's exit status — it does NOT stop the notebook, raise an error, or otherwise surface a non-zero return code on its own. A real Google Colab fresh-runtime run reported `1 failed, 384 passed` from this exact cell, and the notebook continued past it unblocked, because nothing downstream ever inspected pytest's actual result — this was a real, confirmed false-positive risk in the Completion Gate section below, independent of which test happened to fail. This cell now runs pytest via `subprocess.run` with the ACTIVE interpreter (`sys.executable` — never a bare `pytest` that could silently resolve to a different environment), explicitly checks `returncode`, sets a boolean `FULL_TEST_SUITE_PASSED` used by the completion gate, and raises `RuntimeError` (stopping the notebook) if the suite did not pass with zero failures.

**Failure behavior:** raises `RuntimeError` containing the actual subprocess return code if `returncode != 0`; `FULL_TEST_SUITE_PASSED` is left `False` unless this cell completes with `returncode == 0`.

In [ ]:
import subprocess
import sys

FULL_TEST_SUITE_PASSED = False  # Increment 4.1.2: default to NOT passed; only set True below on returncode == 0

result = subprocess.run([sys.executable, "-m", "pytest", "-v"])
if result.returncode == 0:
    FULL_TEST_SUITE_PASSED = True
    print(f"\npytest returncode = {result.returncode}: full test suite passed with zero failures.")
else:
    raise RuntimeError(
        f"pytest returncode = {result.returncode} (non-zero): the combined test suite did NOT "
        f"pass with zero failures. FULL_TEST_SUITE_PASSED remains False. Stopping here rather "
        f"than continuing past a failing test suite - see the pytest output above for the "
        f"specific failure(s)."
    )


### Real Checkshot Integration

#### Step 10 — Load the locked survey trajectories (Poseidon 2, Boreas 1, Proteus 1ST2)

**Technical objective:** load the LOCKED Increment 3/3.1.1 `petrel_source_trace` deviation-survey trajectories for exactly the three wells with an approved checkshot file — needed for the checkshot-vs-survey depth comparison. This reuses the locked `p2mem.io.deviation` loader unmodified; no survey is recomputed or replaced.

In [ ]:
from p2mem.io.deviation import load_deviation_contract_config, load_deviation_surveys

DEV_DIR = os.path.join(PROJECT_ROOT, "data", "raw", "deviation")
DEV_CK_FILES = {
    "Poseidon_2": "Poseidon 2_dev.txt",
    "Boreas_1": "Boreas 1_dev.txt",
    "Proteus_1ST2": "Proteus 1ST2_dev.txt",
}
dev_contracts = load_deviation_contract_config(os.path.join(PROJECT_ROOT, "config", "deviation_survey_contracts.yml"))
dev_paths = {k: os.path.join(DEV_DIR, fn) for k, fn in DEV_CK_FILES.items()}
dev_results, dev_failures = load_deviation_surveys(dev_paths, dev_contracts)
print(f"Locked survey trajectories loaded: {sorted(dev_results)}  failed: {sorted(dev_failures)}")

survey_trajectories = {
    wk: (r.raw.MD_source_m, r.raw.TVD_source_m, r.header.datum_elevation_m)
    for wk, r in dev_results.items()
}


#### Step 11 — Load the three approved checkshot files (full ingestion + conditioning + depth comparison)

**Technical objective:** ingest all three approved checkshot files against `config/checkshot_contracts.yml`, attaching the checkshot-vs-locked-survey depth comparison for each. Poseidon North 1 is recorded separately as `NOT_AVAILABLE` — never substituted.

In [ ]:
from p2mem.checkshot_models import CheckshotAvailabilityRecord
from p2mem.io.checkshot import load_checkshot_contract_config, load_checkshot_surveys

ck_contracts = load_checkshot_contract_config(os.path.join(PROJECT_ROOT, "config", "checkshot_contracts.yml"))
ck_paths = {k: os.path.join(CK_DIR, fn) for k, fn in CK_FILES.items()}
ck_results, ck_failures = load_checkshot_surveys(ck_paths, ck_contracts, survey_trajectories=survey_trajectories)
print(f"Checkshot files loaded: {sorted(ck_results)}  failed: {sorted(ck_failures)}")

availability = {
    "Poseidon_North_1": CheckshotAvailabilityRecord(
        well_key="Poseidon_North_1",
        checkshot_availability="NOT_AVAILABLE",
        notes=(
            "No approved checkshot file exists for Poseidon North 1. This is a factual data "
            "gap, not an ingestion failure; no other well's checkshot file is substituted."
        ),
    )
}

for key in CK_FILES:
    r = ck_results[key]
    print(f"\n{key}: model_use={r.contract.model_use_status}  identity={r.contract.well_identity_evidence_status}")
    print(f"  n_raw={r.raw.Depth_source_m.size}  n_conditioned={r.conditioned.n_conditioned_rows}  n_tie_groups={r.conditioned.n_tie_groups}")
    for issue in r.issues:
        print(f"  [{issue.severity}] {issue.code}")
    if r.depth_comparison is not None:
        c = r.depth_comparison
        print(f"  depth comparison: n={c.n_compared}  max_abs_residual_m={c.max_abs_residual_m:.6f}  mean_residual_m={c.mean_residual_m:.6f}")
        print(f"    {c.residual_trend_description}")


> **QUALITY-CONTROL NOTE:** Poseidon 2's checkshot-vs-survey depth comparison recomputes a maximum absolute residual well within the sub-0.15 m expectation from the Rev 1 design review — supporting the interpretation that `Depth_source_m` behaves as measured depth for this well. Boreas 1 and Proteus 1ST2 both show a near-constant residual OFFSET (opposite sign to each other) — reported here as an OBSERVED datum-like offset pattern, not a proven datum error; neither the checkshot-supplied TVDSS nor the locked survey TVDSS is silently corrected to match the other. Proteus 1ST2's association with `Proteus1-Checkshot.txt` remains explicitly `inferred_unverified` regardless of how well its depth-tie plausibility check performs.

#### Step 11b — Order-invariant axis-tie conditioning and forward/inverse time-depth interpolation (Increment 4.1)

**Technical objective:** exercise the Increment 4.1 `AxisConditionedLookupTable`s (already built once per well inside `load_checkshot_file`, attached as `CheckshotWellResult.axis_tables`/`.axis_tie_entries`) against the real three-well data: print the actual, recomputed per-well/per-direction tie-group counts (never assumed), demonstrate a real forward/inverse round trip through `tvdss_to_owt`/`owt_to_tvdss`, and independently re-verify — against the code and data actually run in THIS notebook — the three specific real order-dependent examples the Increment 4.1 audit found in Increment 4, proving each now resolves to the tied group's MEDIAN rather than silently to only the first-parsed tied row.

**Failure behavior:** raises `AssertionError` if any of the three regression checks fails (i.e. if the order-dependent defect were still present).

In [ ]:
import numpy as np

from p2mem.time_depth import owt_to_tvdss as _owt_to_tvdss, tvdss_to_owt as _tvdss_to_owt
from p2mem.units import owt_to_twt as _owt_to_twt

print("=== Order-invariant axis-tie conditioning: real per-well/per-direction counts (Increment 4.1) ===")
axis_tie_summary = {}
for wk, r in sorted(ck_results.items()):
    table_to_owt = r.axis_tables["tvdss_to_owt"]
    table_to_tvdss = r.axis_tables["owt_to_tvdss"]
    axis_tie_summary[wk] = {
        "n_tvdss_axis_tie_groups": table_to_owt.n_axis_tie_groups,
        "n_tvdss_axis_collapsed_points": table_to_owt.n_collapsed_points,
        "n_tvdss_axis_identical_pairs": table_to_owt.n_identical_pairs,
        "n_tvdss_axis_genuinely_nonunique_groups": table_to_owt.n_genuinely_nonunique_groups,
        "n_owt_axis_tie_groups": table_to_tvdss.n_axis_tie_groups,
        "n_owt_axis_collapsed_points": table_to_tvdss.n_collapsed_points,
        "n_owt_axis_identical_pairs": table_to_tvdss.n_identical_pairs,
        "n_owt_axis_genuinely_nonunique_groups": table_to_tvdss.n_genuinely_nonunique_groups,
    }
    print(f"\n{wk}:")
    print(f"  tvdss_to_owt table: n_in={table_to_owt.n_input_points} n_out={table_to_owt.n_output_points} "
          f"n_tie_groups={table_to_owt.n_axis_tie_groups} "
          f"(identical_pair={table_to_owt.n_identical_pairs}, genuinely_non_unique={table_to_owt.n_genuinely_nonunique_groups})")
    print(f"  owt_to_tvdss table: n_in={table_to_tvdss.n_input_points} n_out={table_to_tvdss.n_output_points} "
          f"n_tie_groups={table_to_tvdss.n_axis_tie_groups} "
          f"(identical_pair={table_to_tvdss.n_identical_pairs}, genuinely_non_unique={table_to_tvdss.n_genuinely_nonunique_groups})")

    # Real forward/inverse round-trip demonstration: TVDSS -> OWT -> TWT -> TVDSS_back
    sample_tvdss = table_to_owt.axis_values[[0, table_to_owt.n_output_points // 2, -1]]
    owt_vals, owt_mask = _tvdss_to_owt(sample_tvdss, table_to_owt)
    twt_vals = _owt_to_twt(owt_vals)
    tvdss_back, tvdss_mask = _owt_to_tvdss(owt_vals, table_to_tvdss)
    print(f"  round-trip sample: TVDSS={np.round(sample_tvdss, 4).tolist()} -> OWT={np.round(owt_vals, 6).tolist()} "
          f"-> TWT={np.round(twt_vals, 6).tolist()} -> TVDSS_back={np.round(tvdss_back, 4).tolist()} "
          f"(all inside coverage: {bool(np.all(owt_mask))}/{bool(np.all(tvdss_mask))})")

print("\n=== Order-dependence regression checks (must be group MEDIAN, never the first-parsed row) ===")

def _lookup(table, axis_value):
    idx = np.where(np.isclose(table.axis_values, axis_value, atol=1e-9))[0]
    assert idx.size == 1, f"expected exactly one match for {axis_value}, got {idx.size}"
    return float(table.dependent_values[idx[0]])

p2_to_owt = ck_results["Poseidon_2"].axis_tables["tvdss_to_owt"]
p2_to_tvdss = ck_results["Poseidon_2"].axis_tables["owt_to_tvdss"]
boreas_to_owt = ck_results["Boreas_1"].axis_tables["tvdss_to_owt"]

tvdss_at_owt_1_0319 = _lookup(p2_to_tvdss, 1.0319)
owt_at_tvdss_4495_6 = _lookup(p2_to_owt, 4495.6)
owt_at_tvdss_3988_8 = _lookup(boreas_to_owt, 3988.8)

assert abs(tvdss_at_owt_1_0319 - 2591.35) < 1e-9, tvdss_at_owt_1_0319
assert tvdss_at_owt_1_0319 != 2591.3, "REGRESSION: silently resolved to first-parsed tied row only"
assert abs(owt_at_tvdss_4495_6 - 1.50175) < 1e-9, owt_at_tvdss_4495_6
assert owt_at_tvdss_4495_6 != 1.5015, "REGRESSION: silently resolved to first-parsed tied row only"
assert abs(owt_at_tvdss_3988_8 - 1.35385) < 1e-9, owt_at_tvdss_3988_8
assert owt_at_tvdss_3988_8 != 1.3531, "REGRESSION: silently resolved to first-parsed tied row only"

print(f"  Poseidon 2  OWT=1.0319 s   -> TVDSS = {tvdss_at_owt_1_0319} m  (median of 2591.3/2591.4; NOT 2591.3)")
print(f"  Poseidon 2  TVDSS=4495.6 m -> OWT   = {owt_at_tvdss_4495_6} s  (median of 1.5015/1.5020; NOT 1.5015)")
print(f"  Boreas 1    TVDSS=3988.8 m -> OWT   = {owt_at_tvdss_3988_8} s  (median of 1.3531/1.3546; NOT 1.3531)")
print("  All three PASS: order-dependent first-tied-row resolution no longer occurs.")

axis_tie_regressions_pass = True  # all three asserts above already raised on any failure


#### Step 11c — Increment 4.1.1 numerical-validation hardening: synthetic defect regression demonstrations

**Technical objective:** independently demonstrate, against the actual code in THIS notebook, that each of the four blocking defects and the unit-helper input-safety gap found by the Increment 4.1.1 audit are now corrected. These use small, portable SYNTHETIC arrays (never the real project data) - consistent with this project's convention that a structural/numerical failure mode is demonstrated with a minimal reproducing example, not by searching the real data for a case that happens to trigger it (zero genuine reversals, zero out-of-coverage-only checkshots, and zero non-finite MD samples exist anywhere in the three approved real checkshot files or the real Poseidon 2 LAS log - see Step 11b and `INCREMENT_04_1_MANIFEST.md`/`INCREMENT_04_1_1_MANIFEST.md`). The full, exhaustive regression-test suite for all five fixes runs in Step 9 above (`pytest -v`); this cell is a targeted, readable demonstration of the same fixes for the notebook reader.

**Failure behavior:** raises `AssertionError` if any demonstrated defect is still present (i.e. if the corresponding operation does NOT raise the typed error it must now raise, or if a legitimate adjacent tie were wrongly rejected).

In [ ]:
import numpy as np

from p2mem.time_depth import (
    TimeDepthError,
    build_axis_conditioned_lookup_table,
    compare_checkshot_to_survey,
    compute_sonic_checkshot_drift,
    milliseconds_to_seconds,
    seconds_to_milliseconds,
)
from p2mem.checkshot_models import ConditionedCheckshotData


def _cond(depth, tvdss, owt):
    depth = np.asarray(depth, dtype=np.float64)
    tvdss = np.asarray(tvdss, dtype=np.float64)
    owt = np.asarray(owt, dtype=np.float64)
    return ConditionedCheckshotData(
        Depth_conditioned_m=depth, TVDSS_conditioned_m=tvdss, OWT_conditioned_s=owt,
        n_raw_rows=int(depth.size), n_conditioned_rows=int(depth.size),
        n_tie_groups=0, conditioning_method="notebook_demo",
    )


print("=== Increment 4.1.1 Defect 1: hidden reversal via global exact-value grouping ===")
try:
    build_axis_conditioned_lookup_table(
        "Demo", "tvdss_to_owt", "TVDSS_conditioned_m", "OWT_conditioned_s",
        np.array([100.0, 200.0, 300.0]), np.array([100.0, 200.0, 100.0]), np.array([1.0, 2.0, 3.0]),
    )
    raise AssertionError("REGRESSION: hidden reversal [100, 200, 100] was NOT rejected")
except TimeDepthError as exc:
    print(f"  [PASS] [100.0, 200.0, 100.0] correctly raises TimeDepthError: {exc}")

# A legitimate adjacent tie must NOT be rejected by the same fix.
_, table = build_axis_conditioned_lookup_table(
    "Demo", "tvdss_to_owt", "TVDSS_conditioned_m", "OWT_conditioned_s",
    np.array([100.0, 200.0, 300.0]), np.array([100.0, 100.0, 200.0]), np.array([1.0, 2.0, 3.0]),
)
assert np.allclose(table.axis_values, [100.0, 200.0])
print("  [PASS] [100.0, 100.0, 200.0] (a genuine adjacent tie) remains valid, median-conditioned")
defect_1_pass = True

print("\n=== Increment 4.1.1 Defect 2: full-canonical-MD validation (not just the selected run) ===")
try:
    # md_m = [0.0, 1.0, 0.0]: decreasing at the last station, but its
    # VP_m_s is NaN so it would be EXCLUDED from the selected
    # finite-positive-VP run under the pre-4.1.1 implementation.
    compute_sonic_checkshot_drift(
        "Demo", np.array([0.0, 1.0, 0.0]), np.array([2000.0, 2000.0, np.nan]), _cond([0.0, 1.0], [0.0, 1.0], [0.0, 0.0005]),
    )
    raise AssertionError("REGRESSION: decreasing MD outside the selected run was NOT rejected")
except TimeDepthError as exc:
    print(f"  [PASS] decreasing full-MD [0.0, 1.0, 0.0] correctly raises TimeDepthError: {exc}")
defect_2_pass = True

print("\n=== Increment 4.1.1 Defect 3: zero in-coverage checkshot rows (typed error, not a bare NumPy crash) ===")
try:
    compare_checkshot_to_survey(
        "Demo", np.array([500.0, 600.0]), np.array([499.0, 599.0]),
        np.array([0.0, 100.0, 200.0]), np.array([0.0, 95.0, 190.0]), datum_elevation_m=0.0,
        depth_basis_interpretation_status="candidate_md_evaluated_against_locked_survey",
    )
    raise AssertionError("REGRESSION: zero in-coverage rows did NOT raise a typed error")
except TimeDepthError as exc:
    print(f"  [PASS] zero in-coverage checkshot rows correctly raise TimeDepthError (not a bare ValueError): {exc}")
defect_3_pass = True

print("\n=== Increment 4.1.1 Defect 4: batch isolation for numerical-conditioning failures ===")
print("  Fully exercised in tests/test_checkshot.py::test_batch_isolates_numerical_conditioning_failure_from_successful_wells")
print("  (see Step 9's pytest output above) using the synthetic tests/fixtures/checkshot_hidden_reversal.txt fixture -")
print("  not re-implemented here to avoid duplicating file-based fixture setup in the notebook.")
defect_4_pass = True

print("\n=== Increment 4.1.1 unit-helper audit: seconds_to_milliseconds / milliseconds_to_seconds reject ambiguous dtypes ===")
for bad_value, label in [(True, "boolean"), ("3.5", "numeric string"), (1.0 + 2.0j, "complex")]:
    try:
        seconds_to_milliseconds(bad_value)
        raise AssertionError(f"REGRESSION: seconds_to_milliseconds accepted {label} input")
    except TypeError:
        print(f"  [PASS] seconds_to_milliseconds correctly rejects {label} input with TypeError")
assert seconds_to_milliseconds(1.5) == 1500.0
assert milliseconds_to_seconds(1500.0) == 1.5
print("  [PASS] valid Python numeric scalars and NumPy numeric arrays are still accepted unchanged")
unit_helper_pass = True

increment_4_1_1_regressions_pass = all([defect_1_pass, defect_2_pass, defect_3_pass, defect_4_pass, unit_helper_pass])


#### Step 12 — Poseidon 2 sonic-checkshot drift diagnostic

**Technical objective:** load the LOCKED canonical Poseidon 2 `VP_m_s`/`MD_m` LAS arrays, algorithmically select the longest contiguous finite-VP interval, and compare its trapezoidally-integrated one-way transit time against the checkshot-interpolated OWT increment over the identical MD/Depth interval. Diagnostic only — no correction is applied to any input.

In [ ]:
from p2mem.io.las import load_file_contract_config as load_las_contracts, load_wells
from p2mem.time_depth import compute_sonic_checkshot_drift

LOGS_DIR = os.path.join(PROJECT_ROOT, "data", "raw", "logs")
las_contracts = load_las_contracts(os.path.join(PROJECT_ROOT, "config", "las_curve_contracts.yml"))
las_results, las_failures = load_wells({"Poseidon_2": os.path.join(LOGS_DIR, "Poseidon_2_logs.las")}, las_contracts)
print(f"LAS loaded: {sorted(las_results)}  failed: {sorted(las_failures)}")

r2_las = las_results["Poseidon_2"]
drift = compute_sonic_checkshot_drift(
    "Poseidon_2", r2_las.canonical_data["MD_m"], r2_las.canonical_data["VP_m_s"],
    ck_results["Poseidon_2"].conditioned,
)
drift_results = {"Poseidon_2": drift}
print(f"Sonic interval MD: [{drift.md_interval_start_m:.4f}, {drift.md_interval_end_m:.4f}]  n_samples={drift.n_sonic_samples}")
print(f"Sonic transit time: {drift.sonic_transit_time_s * 1000:.4f} ms   Checkshot OWT increment: {drift.checkshot_owt_increment_s * 1000:.4f} ms")
print(f"sonic_minus_checkshot_ms = {drift.sonic_minus_checkshot_ms:.4f}   sonic_minus_checkshot_percent = {drift.sonic_minus_checkshot_percent:.4f}%")
for lim in drift.limitations:
    print("  LIMITATION:", lim)


#### Step 13 — Poseidon 2 LAS MD → checkshot-derived time mapping (partial coverage)

**Technical objective:** map the canonical Poseidon 2 LAS `MD_m` array onto checkshot-derived OWT/TWT, strictly within validated checkshot Depth coverage. Poseidon 2's checkshot coverage (1267.8–4700.0 m) is a genuine partial subset of its LAS MD interval — samples outside it are `NaN`, never extrapolated (`n_extrapolated` must be 0).

In [ ]:
from p2mem.time_depth import map_las_md_to_checkshot_time

owt_mapped, twt_mapped, mapping_summary = map_las_md_to_checkshot_time(
    "Poseidon_2", r2_las.canonical_data["MD_m"], ck_results["Poseidon_2"].conditioned
)
mappings = {"Poseidon_2": mapping_summary}
print(f"LAS samples: {mapping_summary.n_las_samples}  inside coverage: {mapping_summary.n_inside_coverage}  "
      f"shallower: {mapping_summary.n_shallower_than_coverage}  deeper: {mapping_summary.n_deeper_than_coverage}")
print(f"Mapped fraction: {mapping_summary.mapped_fraction:.6f}   n_extrapolated: {mapping_summary.n_extrapolated} (must be 0)")


### Quality-Control Results

#### Step 14 — Generate the deterministic inventory outputs

**Technical objective:** write the nine deterministic, metadata-only Increment 4 / 4.1 output files under `outputs/04_checkshot_time_depth/` — never raw per-sample checkshot or LAS arrays. New in Increment 4.1: `checkshot_time_axis_tie_register.csv` (the order-invariant TVDSS/OWT-axis tie audit register — separate from, and never confused with, `checkshot_duplicate_tie_register.csv`'s Depth-axis register).

In [ ]:
import csv
import json

from p2mem.io.checkshot_inventory import (
    build_checkshot_depth_tie_qc_rows,
    build_checkshot_file_inventory_rows,
    build_checkshot_issues_rows,
    build_checkshot_time_axis_tie_register_rows,
    build_checkshot_time_depth_manifest,
    build_checkshot_velocity_summary_rows,
    build_duplicate_tie_register_rows,
    build_sonic_checkshot_drift_rows,
    build_time_depth_mapping_rows,
)

OUT_DIR = os.path.join(PROJECT_ROOT, "outputs", "04_checkshot_time_depth")

def write_csv(path, rows):
    fieldnames = list(rows[0].keys()) if rows else []
    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

write_csv(os.path.join(OUT_DIR, "checkshot_file_inventory.csv"), build_checkshot_file_inventory_rows(ck_results, ck_failures, availability))
write_csv(os.path.join(OUT_DIR, "checkshot_ingestion_issues.csv"), build_checkshot_issues_rows(ck_results, ck_failures))
write_csv(os.path.join(OUT_DIR, "checkshot_duplicate_tie_register.csv"), build_duplicate_tie_register_rows(ck_results))
write_csv(os.path.join(OUT_DIR, "checkshot_time_axis_tie_register.csv"), build_checkshot_time_axis_tie_register_rows(ck_results))
write_csv(os.path.join(OUT_DIR, "checkshot_depth_tie_qc.csv"), build_checkshot_depth_tie_qc_rows(ck_results))
write_csv(os.path.join(OUT_DIR, "checkshot_velocity_summary.csv"), build_checkshot_velocity_summary_rows(ck_results))
write_csv(os.path.join(OUT_DIR, "sonic_checkshot_drift_summary.csv"), build_sonic_checkshot_drift_rows(drift_results))
write_csv(os.path.join(OUT_DIR, "time_depth_mapping_summary.csv"), build_time_depth_mapping_rows(mappings))

manifest = build_checkshot_time_depth_manifest(ck_results, ck_failures, availability, drift_results, mappings)
with open(os.path.join(OUT_DIR, "checkshot_time_depth_manifest.json"), "w") as f:
    json.dump(manifest, f, indent=2, sort_keys=True)
    f.write("\n")

print("Outputs written to:", OUT_DIR)
for fn in sorted(os.listdir(OUT_DIR)):
    fp = os.path.join(OUT_DIR, fn)
    if os.path.isfile(fp):
        print(" -", fn, os.path.getsize(fp), "bytes")


#### Step 15 — Display concise summary tables

**Technical objective:** display the checkshot inventory and depth-tie QC summaries directly in the notebook.

In [ ]:
import pandas as pd

df_inventory = pd.read_csv(os.path.join(OUT_DIR, "checkshot_file_inventory.csv"))
display(df_inventory[["well_key", "source_filename", "well_identity_evidence_status", "model_use_status",
                       "n_raw_rows", "n_conditioned_rows", "n_tie_groups", "contract_status"]])

df_depth_qc = pd.read_csv(os.path.join(OUT_DIR, "checkshot_depth_tie_qc.csv"))
display(df_depth_qc[["well_key", "n_compared", "max_abs_residual_m", "mean_residual_m", "residual_trend_description"]])

df_drift = pd.read_csv(os.path.join(OUT_DIR, "sonic_checkshot_drift_summary.csv"))
display(df_drift[["well_key", "md_interval_start_m", "md_interval_end_m", "sonic_minus_checkshot_ms", "sonic_minus_checkshot_percent"]])

df_axis_tie = pd.read_csv(os.path.join(OUT_DIR, "checkshot_time_axis_tie_register.csv"))
print(f"\ncheckshot_time_axis_tie_register.csv: {len(df_axis_tie)} row(s) (Increment 4.1)")
if len(df_axis_tie):
    display(df_axis_tie[["well_key", "interpolation_direction", "axis", "tie_axis_value", "group_size",
                          "selected_representative_dependent_value", "tie_kind"]])


#### Step 16 — Generate professional QC figures

**Technical objective:** produce five portfolio-quality QC figures (checkshot time-depth, velocity diagnostics, checkshot-vs-survey residuals, Poseidon 2 sonic-checkshot drift, Poseidon 2 time-mapping coverage), each with units, legends, well names, evidence status, and a Tier C footer — no calibration claims, no hidden invalid intervals, no extrapolated curves.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path as _Path

FIG_DIR = _Path(OUT_DIR) / "figures"
ASSURANCE_TIER = "Tier C - Screening-Level / Uncalibrated Educational"

colors = {"Poseidon_2": "tab:blue", "Boreas_1": "tab:orange", "Proteus_1ST2": "tab:green"}

# fig01: checkshot time-depth (Depth vs OWT/TWT), all three wells, conditioned points
fig, ax = plt.subplots(figsize=(7, 8))
for wk, r in sorted(ck_results.items()):
    label = f"{wk} ({r.contract.model_use_status})"
    ax.plot(r.conditioned.OWT_conditioned_s, r.conditioned.Depth_conditioned_m, "o-", ms=3,
            color=colors.get(wk), label=label)
ax.invert_yaxis()
ax.set_xlabel("OWT_conditioned_s (one-way time, s — Depth-tie conditioned)")
ax.set_ylabel("Depth_conditioned_m (candidate MD, m — Depth-tie conditioned)")
ax.set_title("Fig 1. Checkshot time-depth (Depth-tie conditioned) — Poseidon 2 / Boreas 1 / Proteus 1ST2\n"
             "Proteus 1ST2 well-identity: inferred/unverified", fontsize=10)
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
fig.text(0.5, 0.01, f"{ASSURANCE_TIER} — diagnostic/QC only, no calibration implied", ha="center", fontsize=7)
fig.tight_layout(rect=[0, 0.03, 1, 1])
fig.savefig(FIG_DIR / "fig01_checkshot_time_depth.png", dpi=150)
plt.close(fig)

# fig02: velocity diagnostics (Vavg and Vint vs Depth) for all three wells
fig, axes = plt.subplots(1, 2, figsize=(11, 7), sharey=True)
for wk, r in sorted(ck_results.items()):
    v = r.velocity
    axes[0].plot(v.Vavg_m_s, v.Depth_conditioned_m, "-", color=colors.get(wk), label=wk)
    mid_depth = (v.Depth_conditioned_m[:-1] + v.Depth_conditioned_m[1:]) / 2.0
    valid = ~v.vint_invalid_mask
    axes[1].plot(v.Vint_m_s[valid], mid_depth[valid], "o", ms=2, color=colors.get(wk), label=wk)
axes[0].invert_yaxis()
axes[0].set_xlabel("Vavg_m_s (TVDSS / OWT)")
axes[0].set_ylabel("Depth_source_m (m)")
axes[0].set_title("Average velocity")
axes[0].grid(alpha=0.3)
axes[0].legend(fontsize=8)
axes[1].set_xlabel("Vint_m_s (valid intervals only)")
axes[1].set_title("Interval velocity (NaN-invalid intervals excluded)")
axes[1].grid(alpha=0.3)
fig.suptitle("Fig 2. Checkshot velocity diagnostics", fontsize=11)
fig.text(0.5, 0.01, f"{ASSURANCE_TIER} — diagnostic/QC only", ha="center", fontsize=7)
fig.tight_layout(rect=[0, 0.03, 1, 0.96])
fig.savefig(FIG_DIR / "fig02_velocity_diagnostics.png", dpi=150)
plt.close(fig)

# fig03: checkshot-vs-survey depth residuals
fig, ax = plt.subplots(figsize=(8, 6))
for wk, r in sorted(ck_results.items()):
    c = r.depth_comparison
    survey_md, survey_tvd, datum = survey_trajectories[wk]
    survey_tvdss = survey_tvd - datum
    depth = r.raw.Depth_source_m
    inside = (depth >= survey_md.min()) & (depth <= survey_md.max())
    interp_tvdss = np.interp(depth[inside], survey_md, survey_tvdss)
    residual = interp_tvdss - r.raw.TVDSS_source_m[inside]
    ax.plot(residual, depth[inside], "o-", ms=3, color=colors.get(wk),
            label=f"{wk} (max|resid|={c.max_abs_residual_m:.4f} m)")
ax.axvline(0.0, color="k", lw=0.8)
ax.invert_yaxis()
ax.set_xlabel("residual_m = TVDSS_survey_interpolated_m - TVDSS_source_m")
ax.set_ylabel("Depth_source_m (m)")
ax.set_title("Fig 3. Checkshot Depth vs. locked survey TVDSS — residuals\n"
             "(near-constant offsets are an observed pattern, not a proven datum error)", fontsize=9)
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
fig.text(0.5, 0.01, f"{ASSURANCE_TIER}", ha="center", fontsize=7)
fig.tight_layout(rect=[0, 0.03, 1, 1])
fig.savefig(FIG_DIR / "fig03_checkshot_survey_depth_residuals.png", dpi=150)
plt.close(fig)

# fig04: Poseidon 2 sonic-checkshot drift
md = r2_las.canonical_data["MD_m"]
vp = r2_las.canonical_data["VP_m_s"]
finite = np.isfinite(vp)
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(vp[finite], md[finite], ".", ms=1, color="gray", alpha=0.4, label="VP_m_s (all finite)")
in_run = (md >= drift.md_interval_start_m) & (md <= drift.md_interval_end_m) & finite
ax.plot(vp[in_run], md[in_run], ".", ms=1.5, color="tab:red",
        label=f"selected sonic run (n={drift.n_sonic_samples})")
ax.invert_yaxis()
ax.set_xlabel("VP_m_s")
ax.set_ylabel("MD_m")
ax.set_title(
    f"Fig 4. Poseidon 2 sonic-checkshot drift diagnostic\n"
    f"sonic - checkshot = {drift.sonic_minus_checkshot_ms:.2f} ms "
    f"({drift.sonic_minus_checkshot_percent:.2f}%) over MD "
    f"[{drift.md_interval_start_m:.1f}, {drift.md_interval_end_m:.1f}] m — diagnostic only, no correction applied",
    fontsize=9,
)
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
fig.text(0.5, 0.01, f"{ASSURANCE_TIER}", ha="center", fontsize=7)
fig.tight_layout(rect=[0, 0.03, 1, 1])
fig.savefig(FIG_DIR / "fig04_poseidon2_sonic_checkshot_drift.png", dpi=150)
plt.close(fig)

# fig05: Poseidon 2 time mapping coverage
fig, ax = plt.subplots(figsize=(8, 6))
inside_mask = np.isfinite(owt_mapped)
ax.plot(md[inside_mask], owt_mapped[inside_mask], "-", color="tab:blue", lw=1.5,
        label=f"mapped (n={mapping_summary.n_inside_coverage})")
ax.axhspan(0, 0, alpha=0)  # placeholder to keep autoscale stable
shallower = md < mapping_summary.checkshot_depth_min_m
deeper = md > mapping_summary.checkshot_depth_max_m
ymin, ymax = np.nanmin(owt_mapped[inside_mask]), np.nanmax(owt_mapped[inside_mask])
ax.fill_betweenx([ymin, ymax], mapping_summary.las_md_min_m, mapping_summary.checkshot_depth_min_m,
                  color="tab:red", alpha=0.15,
                  label=f"shallower than coverage (n={mapping_summary.n_shallower_than_coverage})")
ax.fill_betweenx([ymin, ymax], mapping_summary.checkshot_depth_max_m, mapping_summary.las_md_max_m,
                  color="tab:purple", alpha=0.15,
                  label=f"deeper than coverage (n={mapping_summary.n_deeper_than_coverage})")
ax.set_xlabel("LAS MD_m (treated as checkshot Depth axis)")
ax.set_ylabel("Mapped OWT_s (checkshot-derived; NaN outside coverage)")
ax.set_title(
    f"Fig 5. Poseidon 2 LAS MD -> checkshot time mapping coverage\n"
    f"mapped fraction = {mapping_summary.mapped_fraction:.3f}; extrapolated = "
    f"{mapping_summary.n_extrapolated} (never > 0)", fontsize=9,
)
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
fig.text(0.5, 0.01, f"{ASSURANCE_TIER}", ha="center", fontsize=7)
fig.tight_layout(rect=[0, 0.03, 1, 1])
fig.savefig(FIG_DIR / "fig05_poseidon2_time_mapping_coverage.png", dpi=150)
plt.close(fig)


#### Step 17 — Integration regression checks

**Technical objective:** independently re-verify, against the code and data actually run in this notebook, every regression figure this increment was designed against — never assumed, always recomputed here.

In [ ]:
print("=== INTEGRATION REGRESSION CHECKS (Increment 4) ===")

EXPECTED_ROW_COUNTS = {"Poseidon_2": 220, "Boreas_1": 212, "Proteus_1ST2": 232}
for key, expected in EXPECTED_ROW_COUNTS.items():
    actual = ck_results[key].raw.Depth_source_m.size
    print(f"{key}: {actual} raw rows (expected {expected}) -> {'MATCH' if actual == expected else 'MISMATCH'}")

print(f"Contracts passed: {sum(1 for k in CK_FILES if ck_results[k].contract_status == 'PASSED')}/3")
print(f"Poseidon 2 max_abs_residual_m < 0.15 m: {ck_results['Poseidon_2'].depth_comparison.max_abs_residual_m:.6f} -> "
      f"{'PASS' if ck_results['Poseidon_2'].depth_comparison.max_abs_residual_m < 0.15 else 'FAIL'}")
print(f"Poseidon North 1 checkshot_availability: {availability['Poseidon_North_1'].checkshot_availability}")
print(f"Time-depth mapping n_extrapolated (must be 0): {mappings['Poseidon_2'].n_extrapolated}")
print(f"Sonic-checkshot drift sign/magnitude: {drift.sonic_minus_checkshot_ms:+.4f} ms "
      f"({drift.sonic_minus_checkshot_percent:+.4f}%)")

_expected_outputs = [
    "checkshot_file_inventory.csv", "checkshot_ingestion_issues.csv", "checkshot_duplicate_tie_register.csv",
    "checkshot_time_axis_tie_register.csv", "checkshot_depth_tie_qc.csv", "checkshot_velocity_summary.csv",
    "sonic_checkshot_drift_summary.csv", "time_depth_mapping_summary.csv", "checkshot_time_depth_manifest.json",
]
print("All 9 deterministic outputs present:", all(os.path.exists(os.path.join(OUT_DIR, fn)) for fn in _expected_outputs))
print(f"Axis-tie-conditioning per-well counts (Increment 4.1, actual, not assumed): {axis_tie_summary}")
print(f"Order-invariant axis-tie regression checks passed: {axis_tie_regressions_pass}")

_expected_figures = [
    "fig01_checkshot_time_depth.png", "fig02_velocity_diagnostics.png",
    "fig03_checkshot_survey_depth_residuals.png", "fig04_poseidon2_sonic_checkshot_drift.png",
    "fig05_poseidon2_time_mapping_coverage.png",
]
print("All 5 QC figures present:", all(os.path.exists(os.path.join(FIG_DIR, fn)) for fn in _expected_figures))


### Interpretation

Poseidon 2's checkshot Depth column behaves consistently with measured depth: the recomputed maximum absolute residual against the locked survey MD→TVDSS relationship is small and comfortably within the sub-0.15 m expectation carried from the Rev 1 design review, supporting (not proving beyond all doubt) that interpretation for this well. Boreas 1 and Proteus 1ST2 each show a near-constant residual offset of opposite sign — an observed, disclosed pattern consistent with a datum-like difference between the checkshot survey and the Petrel deviation-survey reference for those two wells, but not adjudicated as a proven datum error here; both TVDSS references are preserved and reported side by side. The Poseidon 2 sonic-checkshot drift diagnostic finds the integrated sonic one-way transit time modestly (a few percent) longer than the checkshot-interpolated OWT increment over the same interval — consistent with the well-known qualitative difference between along-borehole sonic transit time and a vertically corrected checkshot time, and reported strictly as a diagnostic, uncorrected observation.

### Limitations

- This increment's checkshot ingestion and time-depth layer are, like the rest of this project, screening-level and uncalibrated — no independently surveyed VSP or additional checkshot run exists to adjudicate the Boreas 1 / Proteus 1ST2 offset patterns or the Poseidon 2 sonic-checkshot drift.
- `Proteus1-Checkshot.txt`'s association with Proteus 1ST2 remains `inferred_unverified` — a plausible depth-tie comparison is not independent proof of well identity.
- Boreas 1 and Proteus 1ST2 checkshot data are supporting QC only (`model_use_status: qc_only`) and are never used as, or blended into, Poseidon 2's time-depth relationship.
- Poseidon 2's checkshot Depth coverage (1267.8–4700.0 m) is a genuine partial subset of its LAS MD interval; roughly 30% of LAS samples fall outside checkshot coverage and are reported, never extrapolated, as `NaN` in the time-depth mapping.
- The sonic-checkshot drift diagnostic compares an along-borehole (MD) sonic transit time against a vertically-corrected checkshot time with no acquisition/environmental corrections or run-merging metadata supplied for either — it is a diagnostic, not a calibration, and no correction is applied to any input as a result.
- **(Increment 4.1)** The axis-tie-conditioning MEDIAN representative used to resolve a repeated TVDSS or OWT value for inversion is a disclosed, order-invariant, screening-level choice — it is NOT evidence that the original TVDSS↔OWT relationship was single-valued at that tied axis value; both original rows remain fully preserved in `checkshot_time_axis_tie_register.csv`.
- **(Increment 4.1.1)** The five numerical-validation corrections in this patch harden this layer's failure modes; they do not add, remove, or change any real Poseidon 2/Boreas 1/Proteus 1ST2 result, statistic, figure, or output file. No genuine reversal of either kind (grouped or hidden), no full-MD monotonicity violation, and no zero-coverage checkshot case exists in any of the three approved real checkshot files or the real Poseidon 2 LAS log — see Step 11c for synthetic demonstrations of each fix.
- **(Increment 4.1.2)** This is a packaging/notebook-execution correction only: the CRLF-fixture reconstruction defect and the non-blocking test-suite check were both notebook/runtime-environment issues, not defects in `p2mem`'s numerical methods, contracts, or scientific outputs — none of which changed. This environment cannot execute inside a real Google Colab runtime, so the fix's verification here (Step 7a's cell executed and byte-compared directly, the full test suite run against notebook-generated files, a JSON round-trip of the rebuilt notebook) is the strongest verification available in a Linux/Python environment, not a literal re-run inside Colab itself — disclosed explicitly, not assumed away.
- Downstream phases (formation-top correction, petrophysics, pore pressure, elastic properties, rock strength, stresses, wellbore stability) all remain explicitly out of scope and unimplemented.

### Completion Gate

> **PHASE COMPLETION GATE:** Increment 4 + Increment 4.1 + Increment 4.1.1 + Increment 4.1.2 corrective patches (Checkshot QC & Time-Depth Framework) is complete when: (0) **(Increment 4.1.2)** the complete combined test suite (Step 9) passed with ZERO failures, checked by actual subprocess return code, not assumed or left uninspected; (1) all locked Increment 1–3.1.1 tests and all Increment 4 / 4.1 / 4.1.1 tests pass in the combined suite; (2) all three approved checkshot files load with zero contract-resolution ERRORs and their contracts behave as specified (Poseidon 2 primary_model, Boreas 1 / Proteus 1ST2 qc_only, Proteus 1ST2 identity inferred_unverified); (3) Poseidon North 1 is recorded as `NOT_AVAILABLE`, never substituted; (4) every raw checkshot row is preserved and every duplicate tie is registered, with zero infinite/negative interval velocities; (5) the Poseidon 2 depth-vs-survey residual and sonic-checkshot drift are independently reproduced (not hardcoded) and reported; (6) the LAS-to-checkshot-time mapping never extrapolates (`n_extrapolated == 0`); (7) all 9 deterministic outputs and 5 QC figures are generated; (8) **(Increment 4.1)** the order-invariant axis-tie-conditioning regression checks pass — no first/last order-dependent tie selection remains for TVDSS↔OWT/TWT inversion; (9) **(Increment 4.1.1)** no hidden-reversal-via-global-grouping defect, no incomplete-full-MD-validation defect, no untyped zero-coverage crash, no unisolated batch failure, and no unit-helper dtype-coercion gap remains — all five demonstrated corrected in Step 11c. All conditions are verified programmatically below, not asserted.
>
> **Increment 4.1.2 correction:** a prior version of this gate never checked condition (0) at all — Step 9 ran `!pytest -v` as a bare shell-escape line with no exit-code check, so a real Google Colab run that reported `1 failed, 384 passed` would still have reached this cell and printed every one of conditions (1)–(9) as `[PASS]`, declaring the increment complete despite a failing test. Condition (0) below is checked FIRST and reads `FULL_TEST_SUITE_PASSED`, a variable Step 9 sets to `True` only when its pytest subprocess's `returncode == 0` (Step 9 also raises `RuntimeError` itself on a non-zero return code, stopping the notebook before this cell is ever reached — condition (0) is a second, independent line of defense: it fails closed, as `False`, rather than assuming success, if `FULL_TEST_SUITE_PASSED` was ever left unset for any reason).

In [ ]:
print("=" * 78)
print("INCREMENT 4 / 4.1 / 4.1.1 / 4.1.2 COMPLETION GATE")
print("=" * 78)

gate_checks = {
    # Increment 4.1.2: read via globals().get(..., False), NOT a bare
    # `FULL_TEST_SUITE_PASSED` name reference, so that if Step 9 were ever
    # skipped entirely (the variable never defined at all) this check
    # fails CLEANLY as False - printed as an ordinary [FAIL] line below -
    # rather than crashing the whole gate cell with an uncaught
    # NameError. Either way, "the variable was never set" cannot result
    # in this gate printing overall completion: a clean [FAIL] here does
    # not (`is True` only matches the literal boolean True, set only by
    # Step 9 on a confirmed returncode == 0).
    "(4.1.2) Complete combined test suite passed with zero failures": (
        globals().get("FULL_TEST_SUITE_PASSED", False) is True
    ),
    "All 3 checkshot files loaded (0 failures)": len(ck_failures) == 0,
    "All 3 contracts PASSED": all(ck_results[k].contract_status == "PASSED" for k in CK_FILES),
    "Poseidon 2 = primary_model, Boreas 1 / Proteus 1ST2 = qc_only": (
        ck_results["Poseidon_2"].contract.model_use_status == "primary_model"
        and ck_results["Boreas_1"].contract.model_use_status == "qc_only"
        and ck_results["Proteus_1ST2"].contract.model_use_status == "qc_only"
    ),
    "Proteus 1ST2 identity is inferred_unverified (never verified)": (
        ck_results["Proteus_1ST2"].contract.well_identity_evidence_status == "inferred_unverified"
    ),
    "Poseidon North 1 recorded as NOT_AVAILABLE": (
        availability["Poseidon_North_1"].checkshot_availability == "NOT_AVAILABLE"
    ),
    "Time-depth mapping never extrapolates (n_extrapolated == 0)": (
        mappings["Poseidon_2"].n_extrapolated == 0
    ),
    "9 deterministic output files present": all(
        os.path.exists(os.path.join(OUT_DIR, fn)) for fn in _expected_outputs
    ),
    "5 QC figures present": all(
        os.path.exists(os.path.join(FIG_DIR, fn)) for fn in _expected_figures
    ),
    "(4.1) Order-invariant axis-tie regression checks passed (no order-dependent tie selection)": (
        axis_tie_regressions_pass
    ),
    "(4.1.1) All five numerical-validation corrections demonstrated (hidden reversal, full-MD, "
    "zero-coverage, batch isolation, unit-helper dtype rejection)": (
        increment_4_1_1_regressions_pass
    ),
}
for label, passed in gate_checks.items():
    print(f"  [{'PASS' if passed else 'FAIL'}] {label}")

if all(gate_checks.values()):
    print("\nIncrement 4 + Increment 4.1 + Increment 4.1.1 + Increment 4.1.2 corrective patches are complete. Stopping here per the approved scope.")
    print("Not implemented (explicitly out of scope): formation-top correction, lithology")
    print("interpretation, density modelling, pore-pressure prediction, elastic properties,")
    print("rock strength, stress modelling, wellbore-stability analysis. Increment 5 has NOT")
    print("been started.")
else:
    raise RuntimeError("Increment 4 / 4.1 / 4.1.1 / 4.1.2 completion gate FAILED - see failed check(s) above.")
